In [2]:
#!/usr/bin/env python3
"""
ANATOMICALLY-CORRECT INFARCT ASSIGNMENT
Based on AHA 17-Segment Model and Coronary Artery Territories

This script assigns infarct regions that follow realistic coronary artery 
distributions.

CORONARY ARTERY TERRITORIES:

LAD (Left Anterior Descending):
  - Anterior wall
  - Anterior septum  
  - Apex
  - ~40-50% of LV mass
  - Most common infarct territory

RCA (Right Coronary Artery):
  - Inferior wall
  - Inferior septum (proximal)
  - ~30-40% of LV mass

LCx (Left Circumflex):
  - Lateral wall
  - Posterior wall
  - ~15-25% of LV mass

AHA 17-SEGMENT MODEL:

Basal (segments 1-6):
  1. Basal anterior
  2. Basal anteroseptal
  3. Basal inferoseptal
  4. Basal inferior
  5. Basal inferolateral
  6. Basal anterolateral

Mid (segments 7-12):
  7. Mid anterior
  8. Mid anteroseptal
  9. Mid inferoseptal
  10. Mid inferior
  11. Mid inferolateral
  12. Mid anterolateral

Apical (segments 13-16):
  13. Apical anterior
  14. Apical septal
  15. Apical inferior
  16. Apical lateral

Apex (segment 17):
  17. Apex

INFARCT PATTERNS:
- LAD: Segments 1, 2, 7, 8, 13, 14, 17 (anterior + septal + apex)
- RCA: Segments 3, 4, 9, 10, 15 (inferior + inferoseptal)
- LCx: Segments 5, 6, 11, 12, 16 (lateral)

"""

import numpy as np
from scipy.spatial import cKDTree
from scipy.ndimage import gaussian_filter1d
from collections import defaultdict
import os
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')


PATIENT_IDS = [
    "SCD0000101", "SCD0000201", "SCD0000301", "SCD0000401",
    "SCD0000601", "SCD0000701", "SCD0000801", "SCD0001001",
    "SCD0001101", "SCD0001201"
]

BASE_DIR = "/home/shadeform/SCD_MODELS"
OUTPUT_DIR = "/home/shadeform/SCD_MODELS/infarct_coronary_territory"

# Tissue tags
TAG_HEALTHY = 1
TAG_BORDER = 2
TAG_INFARCT = 3

# AHA 17-SEGMENT MODEL DEFINITION

class AHA17Segments:
    """
    AHA 17-segment model for left ventricle.
    
    Circumferential angles (viewed from apex, RV on left):
    - 0° = Anterior (12 o'clock)
    - 60° = Anteroseptal
    - 120° = Inferoseptal
    - 180° = Inferior (6 o'clock)
    - 240° = Inferolateral
    - 300° = Anterolateral
    
    Longitudinal levels:
    - Basal: 0-33% from base
    - Mid: 33-67% from base
    - Apical: 67-95% from base
    - Apex: 95-100% (tip)
    """
    
    # Segment names
    NAMES = {
        1: "Basal anterior",
        2: "Basal anteroseptal",
        3: "Basal inferoseptal",
        4: "Basal inferior",
        5: "Basal inferolateral",
        6: "Basal anterolateral",
        7: "Mid anterior",
        8: "Mid anteroseptal",
        9: "Mid inferoseptal",
        10: "Mid inferior",
        11: "Mid inferolateral",
        12: "Mid anterolateral",
        13: "Apical anterior",
        14: "Apical septal",
        15: "Apical inferior",
        16: "Apical lateral",
        17: "Apex"
    }
    
    # Coronary artery territories (typical distribution)
    # Note: There's anatomical variation, but this is the standard mapping
    LAD_SEGMENTS = [1, 2, 7, 8, 13, 14, 17]  # Anterior + anteroseptal + apex
    RCA_SEGMENTS = [3, 4, 9, 10, 15]          # Inferior + inferoseptal
    LCX_SEGMENTS = [5, 6, 11, 12, 16]         # Lateral
    
    # Extended LAD (when LAD wraps around apex) - common variant
    LAD_EXTENDED = [1, 2, 7, 8, 13, 14, 15, 17]
    
    # Circumferential angle boundaries (degrees, from anterior = 0°)
    # For basal and mid levels (6 segments each)
    CIRC_BOUNDARIES_6 = [0, 60, 120, 180, 240, 300, 360]
    
    # For apical level (4 segments)
    CIRC_BOUNDARIES_4 = [0, 90, 180, 270, 360]
    
    # Longitudinal boundaries (fraction from base)
    LONG_BOUNDARIES = {
        'basal': (0.0, 0.33),
        'mid': (0.33, 0.67),
        'apical': (0.67, 0.95),
        'apex': (0.95, 1.0)
    }


# INFARCT PATTERNS (Realistic distributions)

class InfarctPatterns:
    """
    Predefined infarct patterns based on clinical presentations.
    Each pattern specifies which AHA segments are affected.
    """
    
    PATTERNS = {
        # LAD territory infarcts (most common, ~40-50% of MIs)
        'LAD_anterior': {
            'name': 'LAD - Anterior MI',
            'core_segments': [1, 7, 13],  # Anterior wall
            'border_segments': [2, 6, 8, 12, 14, 16],
            'description': 'Proximal LAD occlusion - anterior wall'
        },
        'LAD_anteroseptal': {
            'name': 'LAD - Anteroseptal MI',
            'core_segments': [2, 8, 14],  # Anteroseptal
            'border_segments': [1, 3, 7, 9, 13, 15, 17],
            'description': 'LAD with septal branches - anteroseptal'
        },
        'LAD_extensive': {
            'name': 'LAD - Extensive Anterior MI',
            'core_segments': [1, 2, 7, 8, 13, 14, 17],  # Full LAD territory
            'border_segments': [3, 6, 9, 12, 15, 16],
            'description': 'Proximal LAD - extensive anterior + apex'
        },
        'LAD_apical': {
            'name': 'LAD - Apical MI',
            'core_segments': [13, 14, 17],  # Apical
            'border_segments': [7, 8, 15, 16],
            'description': 'Distal LAD - apical'
        },
        
        # RCA territory infarcts (~30-40% of MIs)
        'RCA_inferior': {
            'name': 'RCA - Inferior MI',
            'core_segments': [4, 10, 15],  # Inferior wall
            'border_segments': [3, 5, 9, 11, 14, 16],
            'description': 'RCA occlusion - inferior wall'
        },
        'RCA_inferoseptal': {
            'name': 'RCA - Inferoseptal MI',
            'core_segments': [3, 4, 9, 10],  # Inferior + inferoseptal
            'border_segments': [2, 5, 8, 11, 14, 15],
            'description': 'Proximal RCA - inferoseptal'
        },
        
        # LCx territory infarcts (~15-20% of MIs)
        'LCX_lateral': {
            'name': 'LCx - Lateral MI',
            'core_segments': [5, 11, 16],  # Lateral wall
            'border_segments': [4, 6, 10, 12, 15],
            'description': 'LCx occlusion - lateral wall'
        },
        'LCX_posterolateral': {
            'name': 'LCx - Posterolateral MI',
            'core_segments': [5, 6, 11, 12, 16],  # Lateral + anterolateral
            'border_segments': [1, 4, 7, 10, 13, 15],
            'description': 'Proximal LCx - posterolateral'
        },
    }
    
    # Probability weights for random selection (based on clinical frequency)
    PATTERN_WEIGHTS = {
        'LAD_anterior': 0.15,
        'LAD_anteroseptal': 0.15,
        'LAD_extensive': 0.10,
        'LAD_apical': 0.10,
        'RCA_inferior': 0.20,
        'RCA_inferoseptal': 0.10,
        'LCX_lateral': 0.12,
        'LCX_posterolateral': 0.08,
    }


# MESH LOADING

def load_mesh(patient_id, base_dir):
    """Load tetrahedral mesh"""
    pts_file = f"{base_dir}/simulation_ready/{patient_id}/{patient_id}_tet.pts"
    elem_file = f"{base_dir}/simulation_ready/{patient_id}/{patient_id}_tet.elem"
    
    with open(pts_file, 'r') as f:
        n_nodes = int(f.readline().strip())
        coords = np.zeros((n_nodes, 3), dtype=np.float64)
        for i in range(n_nodes):
            coords[i] = [float(x) for x in f.readline().split()[:3]]
    
    with open(elem_file, 'r') as f:
        n_elems = int(f.readline().strip())
        elements = np.zeros((n_elems, 4), dtype=np.int32)
        for i in range(n_elems):
            parts = f.readline().split()
            elements[i] = [int(x) for x in parts[1:5]]
    
    return coords, elements


def load_fibers(patient_id, base_dir):
    """Load fiber orientations"""
    lon_file = f"{base_dir}/fibers/{patient_id}/{patient_id}.lon"
    
    with open(lon_file, 'r') as f:
        _ = f.readline()
        lines = f.readlines()
        fibers = np.zeros((len(lines), 3), dtype=np.float64)
        for i, line in enumerate(lines):
            fibers[i] = [float(x) for x in line.split()[:3]]
    
    norms = np.linalg.norm(fibers, axis=1, keepdims=True)
    norms[norms == 0] = 1
    return fibers / norms


# ANATOMICAL COORDINATE SYSTEM

def compute_anatomical_coordinates(coords, elements):
    """
    Compute anatomical coordinate system for the LV.
    
    Returns:
    - long_axis: Unit vector from apex to base
    - apex_point: Coordinates of apex
    - base_center: Center of base
    - centroids: Element centroids
    - longitudinal: Fractional position along long axis (0=base, 1=apex)
    - circumferential: Angle around long axis (0=anterior, 180=inferior)
    """
    n_elems = len(elements)
    centroids = np.mean(coords[elements], axis=1)
    
    # Find long axis using PCA
    center = np.mean(centroids, axis=0)
    centered = centroids - center
    cov = np.cov(centered.T)
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    
    # Long axis = direction of maximum variance
    long_axis = eigenvectors[:, np.argmax(eigenvalues)]
    
    # Project all points onto long axis
    projections = centered @ long_axis
    
    # Apex = most negative projection, Base = most positive
    # (Convention: apex is at bottom/negative end)
    if np.mean(projections[projections < np.median(projections)]) > 0:
        long_axis = -long_axis
        projections = -projections
    
    # Normalize longitudinal coordinate: 0 = base, 1 = apex
    proj_min, proj_max = projections.min(), projections.max()
    longitudinal = (projections - proj_min) / (proj_max - proj_min)
    
    # Find apex and base points
    apex_idx = np.argmax(longitudinal)
    base_indices = np.where(longitudinal < 0.1)[0]
    
    apex_point = centroids[apex_idx]
    base_center = np.mean(centroids[base_indices], axis=0)
    
    # Compute circumferential angle
    # First, establish a reference direction (perpendicular to long axis)
    # We'll use the direction to the RV (typically +x in standard orientation)
    
    # Find a perpendicular reference direction
    arbitrary = np.array([1, 0, 0])
    if abs(np.dot(long_axis, arbitrary)) > 0.9:
        arbitrary = np.array([0, 1, 0])
    
    ref_dir = arbitrary - np.dot(arbitrary, long_axis) * long_axis
    ref_dir = ref_dir / np.linalg.norm(ref_dir)
    
    # Second perpendicular direction
    perp_dir = np.cross(long_axis, ref_dir)
    
    # Compute circumferential angle for each element
    circumferential = np.zeros(n_elems)
    
    for i in range(n_elems):
        # Vector from axis to centroid (perpendicular to long axis)
        to_centroid = centered[i] - projections[i] * long_axis
        
        if np.linalg.norm(to_centroid) < 1e-10:
            circumferential[i] = 0
            continue
        
        to_centroid = to_centroid / np.linalg.norm(to_centroid)
        
        # Angle from reference direction
        cos_angle = np.dot(to_centroid, ref_dir)
        sin_angle = np.dot(to_centroid, perp_dir)
        
        angle = np.arctan2(sin_angle, cos_angle)
        circumferential[i] = np.degrees(angle)
        
        # Normalize to [0, 360)
        if circumferential[i] < 0:
            circumferential[i] += 360
    
    return {
        'long_axis': long_axis,
        'apex_point': apex_point,
        'base_center': base_center,
        'centroids': centroids,
        'longitudinal': longitudinal,  # 0=base, 1=apex
        'circumferential': circumferential,  # 0-360 degrees
        'ref_dir': ref_dir,
        'perp_dir': perp_dir
    }


def assign_aha_segments(anat_coords, n_elems):
    """
    Assign each element to an AHA segment based on anatomical coordinates.
    
    Circumferential mapping (viewed from apex):
    - Anterior: 330-30° (segment x.1)
    - Anteroseptal: 30-90° (segment x.2)
    - Inferoseptal: 90-150° (segment x.3)
    - Inferior: 150-210° (segment x.4)
    - Inferolateral: 210-270° (segment x.5)
    - Anterolateral: 270-330° (segment x.6)
    
    For apical level (4 segments):
    - Anterior: 315-45° (segment 13)
    - Septal: 45-135° (segment 14)
    - Inferior: 135-225° (segment 15)
    - Lateral: 225-315° (segment 16)
    """
    longitudinal = anat_coords['longitudinal']
    circumferential = anat_coords['circumferential']
    
    segments = np.zeros(n_elems, dtype=np.int32)
    
    for i in range(n_elems):
        long_pos = longitudinal[i]  # 0=base, 1=apex
        circ_pos = circumferential[i]  # 0-360 degrees
        
        # Determine longitudinal level
        if long_pos < 0.33:
            level = 'basal'
            base_segment = 0
        elif long_pos < 0.67:
            level = 'mid'
            base_segment = 6
        elif long_pos < 0.95:
            level = 'apical'
            base_segment = 12
        else:
            # Apex (segment 17)
            segments[i] = 17
            continue
        
        # Determine circumferential position
        if level in ['basal', 'mid']:
            # 6 segments per level
            # Shift angle so anterior (segment 1) is centered at 0°
            shifted_angle = (circ_pos + 30) % 360
            
            if shifted_angle < 60:
                circ_segment = 1  # Anterior
            elif shifted_angle < 120:
                circ_segment = 2  # Anteroseptal
            elif shifted_angle < 180:
                circ_segment = 3  # Inferoseptal
            elif shifted_angle < 240:
                circ_segment = 4  # Inferior
            elif shifted_angle < 300:
                circ_segment = 5  # Inferolateral
            else:
                circ_segment = 6  # Anterolateral
                
        else:  # apical level - 4 segments
            shifted_angle = (circ_pos + 45) % 360
            
            if shifted_angle < 90:
                circ_segment = 1  # Anterior (13)
            elif shifted_angle < 180:
                circ_segment = 2  # Septal (14)
            elif shifted_angle < 270:
                circ_segment = 3  # Inferior (15)
            else:
                circ_segment = 4  # Lateral (16)
        
        segments[i] = base_segment + circ_segment
    
    return segments


# INFARCT ASSIGNMENT

def select_infarct_pattern(patient_id, seed=None):
    """
    Select an infarct pattern for a patient.
    Uses patient ID as seed for reproducibility but varied results.
    """
    if seed is None:
        # Use patient ID to generate consistent but varied patterns
        seed = int(''.join(filter(str.isdigit, patient_id))) % 1000
    
    np.random.seed(seed)
    
    patterns = list(InfarctPatterns.PATTERNS.keys())
    weights = [InfarctPatterns.PATTERN_WEIGHTS[p] for p in patterns]
    weights = np.array(weights) / sum(weights)
    
    selected = np.random.choice(patterns, p=weights)
    
    return selected, InfarctPatterns.PATTERNS[selected]


def compute_segment_distances(segments, anat_coords, target_segments):
    """
    Compute distance from each element to the nearest target segment.
    Used for creating smooth transitions at borders.
    """
    n_elems = len(segments)
    centroids = anat_coords['centroids']
    
    # Find elements in target segments
    target_mask = np.isin(segments, target_segments)
    target_centroids = centroids[target_mask]
    
    if len(target_centroids) == 0:
        return np.ones(n_elems) * 1000  # No targets
    
    # Build KD-tree for fast nearest neighbor
    tree = cKDTree(target_centroids)
    
    # Query distance for all elements
    distances, _ = tree.query(centroids)
    
    return distances


def assign_infarct_territory(segments, anat_coords, pattern_info, elements,
                             infarct_fraction=0.08, border_fraction=0.12):
    """
    Assign infarct and border zones based on coronary territory pattern.
    
    Parameters:
    - segments: AHA segment assignment for each element
    - anat_coords: Anatomical coordinate system
    - pattern_info: Dictionary with core_segments and border_segments
    - elements: Mesh element connectivity (for adjacency)
    - infarct_fraction: Target fraction of LV for dense scar (default 8%)
    - border_fraction: Target fraction for border zone (default 12%)
    
    Returns:
    - classification: Array with 1=healthy, 2=border, 3=infarct
    """
    n_elems = len(segments)
    longitudinal = anat_coords['longitudinal']
    centroids = anat_coords['centroids']
    
    core_segments = pattern_info['core_segments']
    border_segments = pattern_info['border_segments']
    
    # Initialize all as healthy
    classification = np.ones(n_elems, dtype=np.int32)
    
    # Step 1: Mark core infarct segments
    core_mask = np.isin(segments, core_segments)
    
    # Step 2: Add transmural gradient (infarct more likely subendocardially)
    # We don't have exact endo/epi info, so use a probability gradient
    # Elements closer to apex in core segments are more likely to be infarct
    
    # Compute distance to infarct center for probability weighting
    core_centroids = centroids[core_mask]
    if len(core_centroids) > 0:
        infarct_center = np.mean(core_centroids, axis=0)
        distances_to_center = np.linalg.norm(centroids - infarct_center, axis=1)
        max_dist = np.percentile(distances_to_center[core_mask], 95)
        
        # Probability decreases with distance from center
        prob_infarct = np.clip(1 - distances_to_center / (max_dist * 1.5), 0, 1)
    else:
        prob_infarct = np.zeros(n_elems)
    
    # Step 3: Assign infarct based on segment + probability
    # Elements in core segments with high probability become infarct
    
    # Calculate how many elements we need for target fraction
    target_infarct_count = int(n_elems * infarct_fraction)
    target_border_count = int(n_elems * border_fraction)
    
    # Score for infarct: high if in core segment and high probability
    infarct_score = np.zeros(n_elems)
    infarct_score[core_mask] = prob_infarct[core_mask] + 1.0  # Boost for core segments
    
    # Add slight randomness for natural appearance
    np.random.seed(42)
    infarct_score += np.random.uniform(0, 0.3, n_elems)
    
    # Select top elements as infarct
    infarct_threshold = np.percentile(infarct_score, 100 * (1 - infarct_fraction))
    infarct_mask = infarct_score >= infarct_threshold
    
    # Ensure infarcts are primarily in core segments
    infarct_mask = infarct_mask & (core_mask | np.isin(segments, border_segments))
    
    classification[infarct_mask] = TAG_INFARCT
    
    # Step 4: Create border zone around infarct
    # Border = adjacent to infarct OR in border segments near infarct
    
    # Build adjacency
    adjacency = build_adjacency(elements)
    
    # Find elements adjacent to infarct
    adjacent_to_infarct = set()
    for i in np.where(infarct_mask)[0]:
        for j in adjacency.get(i, []):
            if not infarct_mask[j]:
                adjacent_to_infarct.add(j)
    
    # Also include border segments that are near the infarct
    border_segment_mask = np.isin(segments, border_segments)
    
    # Distance from each element to nearest infarct element
    infarct_centroids = centroids[infarct_mask]
    if len(infarct_centroids) > 0:
        tree = cKDTree(infarct_centroids)
        dist_to_infarct, _ = tree.query(centroids)
    else:
        dist_to_infarct = np.ones(n_elems) * 1000
    
    # Border zone: adjacent to infarct OR (in border segment AND close to infarct)
    median_dist = np.median(dist_to_infarct[list(adjacent_to_infarct)]) if adjacent_to_infarct else 10
    close_threshold = median_dist * 3
    
    border_mask = np.zeros(n_elems, dtype=bool)
    border_mask[list(adjacent_to_infarct)] = True
    border_mask |= (border_segment_mask & (dist_to_infarct < close_threshold))
    border_mask &= ~infarct_mask  # Don't overwrite infarct
    
    # Expand border slightly using adjacency
    for _ in range(2):  # 2 layers
        new_border = set()
        for i in np.where(border_mask)[0]:
            for j in adjacency.get(i, []):
                if not infarct_mask[j] and not border_mask[j]:
                    if dist_to_infarct[j] < close_threshold * 1.5:
                        new_border.add(j)
        border_mask[list(new_border)] = True
    
    classification[border_mask] = TAG_BORDER
    
    return classification


def build_adjacency(elements):
    """Build element adjacency via shared faces"""
    face_to_elem = defaultdict(list)
    for i, nodes in enumerate(elements):
        for face in [
            tuple(sorted([nodes[0], nodes[1], nodes[2]])),
            tuple(sorted([nodes[0], nodes[1], nodes[3]])),
            tuple(sorted([nodes[0], nodes[2], nodes[3]])),
            tuple(sorted([nodes[1], nodes[2], nodes[3]]))
        ]:
            face_to_elem[face].append(i)
    
    adjacency = defaultdict(list)
    for face, elems in face_to_elem.items():
        if len(elems) == 2:
            adjacency[elems[0]].append(elems[1])
            adjacency[elems[1]].append(elems[0])
    return adjacency


# SMOOTHING AND REFINEMENT

def smooth_classification(classification, adjacency, iterations=3):
    """
    Smooth classification boundaries to remove isolated elements.
    Uses majority voting in local neighborhood.
    """
    n_elems = len(classification)
    
    for _ in range(iterations):
        new_classification = classification.copy()
        
        for i in range(n_elems):
            neighbors = adjacency.get(i, [])
            if len(neighbors) < 2:
                continue
            
            # Count neighbor types
            neighbor_types = [classification[j] for j in neighbors]
            neighbor_types.append(classification[i])  # Include self
            
            # Majority vote
            counts = {TAG_HEALTHY: 0, TAG_BORDER: 0, TAG_INFARCT: 0}
            for t in neighbor_types:
                counts[t] += 1
            
            # Only change if strong majority
            max_count = max(counts.values())
            if max_count >= len(neighbor_types) * 0.6:
                majority_type = max(counts, key=counts.get)
                
                # Don't change infarct to healthy directly
                if classification[i] == TAG_INFARCT and majority_type == TAG_HEALTHY:
                    new_classification[i] = TAG_BORDER
                # Don't change healthy to infarct directly
                elif classification[i] == TAG_HEALTHY and majority_type == TAG_INFARCT:
                    new_classification[i] = TAG_BORDER
                else:
                    new_classification[i] = majority_type
        
        classification = new_classification
    
    return classification


def ensure_border_continuity(classification, adjacency):
    """
    Ensure border zone forms a continuous layer around infarct.
    Any healthy element adjacent to infarct becomes border.
    """
    n_elems = len(classification)
    
    for i in range(n_elems):
        if classification[i] == TAG_HEALTHY:
            # Check if adjacent to infarct
            for j in adjacency.get(i, []):
                if classification[j] == TAG_INFARCT:
                    classification[i] = TAG_BORDER
                    break
    
    return classification


# OUTPUT FUNCTIONS

def write_vtk(filepath, coords, elements, classification, segments, anat_coords):
    """Save VTK with classification and anatomical data"""
    n_nodes, n_elems = len(coords), len(elements)
    
    with open(filepath, 'w') as f:
        f.write("# vtk DataFile Version 3.0\n")
        f.write("Coronary Territory Infarct Classification\n")
        f.write("ASCII\n")
        f.write("DATASET UNSTRUCTURED_GRID\n")
        
        f.write(f"POINTS {n_nodes} float\n")
        for c in coords:
            f.write(f"{c[0]:.6f} {c[1]:.6f} {c[2]:.6f}\n")
        
        f.write(f"\nCELLS {n_elems} {n_elems * 5}\n")
        for e in elements:
            f.write(f"4 {e[0]} {e[1]} {e[2]} {e[3]}\n")
        
        f.write(f"\nCELL_TYPES {n_elems}\n")
        f.write("10\n" * n_elems)
        
        # Cell data
        f.write(f"\nCELL_DATA {n_elems}\n")
        
        f.write("SCALARS TissueType int 1\nLOOKUP_TABLE default\n")
        for c in classification:
            f.write(f"{c}\n")
        
        f.write("\nSCALARS AHA_Segment int 1\nLOOKUP_TABLE default\n")
        for s in segments:
            f.write(f"{s}\n")
        
        f.write("\nSCALARS Longitudinal float 1\nLOOKUP_TABLE default\n")
        for l in anat_coords['longitudinal']:
            f.write(f"{l:.6f}\n")
        
        f.write("\nSCALARS Circumferential float 1\nLOOKUP_TABLE default\n")
        for c in anat_coords['circumferential']:
            f.write(f"{c:.6f}\n")


def write_region_vtk(filepath, coords, elements, classification, region_code):
    """Save VTK for single region"""
    mask = classification == region_code
    region_elems = np.where(mask)[0]
    
    if len(region_elems) == 0:
        return 0
    
    region_elements = elements[region_elems]
    unique_nodes = np.unique(region_elements.flatten())
    node_map = {old: new for new, old in enumerate(unique_nodes)}
    remapped = np.array([[node_map[n] for n in elem] for elem in region_elements])
    
    with open(filepath, 'w') as f:
        f.write("# vtk DataFile Version 3.0\nRegion\nASCII\n")
        f.write("DATASET UNSTRUCTURED_GRID\n")
        
        f.write(f"POINTS {len(unique_nodes)} float\n")
        for n in unique_nodes:
            c = coords[n]
            f.write(f"{c[0]:.6f} {c[1]:.6f} {c[2]:.6f}\n")
        
        f.write(f"\nCELLS {len(region_elems)} {len(region_elems) * 5}\n")
        for e in remapped:
            f.write(f"4 {e[0]} {e[1]} {e[2]} {e[3]}\n")
        
        f.write(f"\nCELL_TYPES {len(region_elems)}\n")
        f.write("10\n" * len(region_elems))
    
    return len(region_elems)


def write_tagged_elem(filepath, elements, classification):
    """Save OpenCARP format"""
    with open(filepath, 'w') as f:
        f.write(f"{len(elements)}\n")
        for i, e in enumerate(elements):
            f.write(f"Tt {e[0]} {e[1]} {e[2]} {e[3]} {classification[i]}\n")


# MAIN PIPELINE

def process_patient(patient_id, pattern_override=None):
    """
    Process a single patient with coronary territory-based infarct assignment.
    """
    print(f"PROCESSING: {patient_id}")
    
    # Load mesh
    print("\n  Loading mesh")
    coords, elements = load_mesh(patient_id, BASE_DIR)
    n_elems = len(elements)
    print(f"      {len(coords):,} nodes, {n_elems:,} elements")
    
    # Make elements globally available for adjacency building
    global elements_global
    elements_global = elements
    
    # Compute anatomical coordinates
    print("\n  Computing anatomical coordinate system")
    anat_coords = compute_anatomical_coordinates(coords, elements)
    print(f"      Long axis: [{anat_coords['long_axis'][0]:.3f}, "
          f"{anat_coords['long_axis'][1]:.3f}, {anat_coords['long_axis'][2]:.3f}]")
    
    # Assign AHA segments
    print("\n  Assigning AHA 17-segment model")
    segments = assign_aha_segments(anat_coords, n_elems)
    
    segment_counts = {}
    for s in range(1, 18):
        count = np.sum(segments == s)
        if count > 0:
            segment_counts[s] = count
    print(f"      Segments assigned: {len(segment_counts)} of 17")
    
    # Select infarct pattern
    print("\n  Selecting infarct pattern")
    if pattern_override:
        pattern_name = pattern_override
        pattern_info = InfarctPatterns.PATTERNS[pattern_name]
    else:
        pattern_name, pattern_info = select_infarct_pattern(patient_id)
    
    print(f"      Pattern: {pattern_info['name']}")
    print(f"      {pattern_info['description']}")
    print(f"      Core segments: {pattern_info['core_segments']}")
    print(f"      Border segments: {pattern_info['border_segments']}")
    
    # Assign infarct territory
    print("\n  Assigning infarct territory")
    
    # Build adjacency for this mesh
    adjacency = build_adjacency(elements)
    
    # Vary infarct size slightly per patient (6-12%)
    np.random.seed(int(''.join(filter(str.isdigit, patient_id))) % 1000 + 1)
    infarct_fraction = np.random.uniform(0.06, 0.12)
    border_fraction = np.random.uniform(0.10, 0.18)
    
    classification = assign_infarct_territory(
        segments, anat_coords, pattern_info, elements,
        infarct_fraction=infarct_fraction,
        border_fraction=border_fraction
    )
    
    # Smooth and refine
    print("\n  Smoothing boundaries")
    classification = smooth_classification(classification, adjacency)
    classification = ensure_border_continuity(classification, adjacency)
    
    # Statistics
    n_healthy = np.sum(classification == TAG_HEALTHY)
    n_border = np.sum(classification == TAG_BORDER)
    n_infarct = np.sum(classification == TAG_INFARCT)
    
    print(f"\n  FINAL CLASSIFICATION:")
    print(f"      Healthy: {n_healthy:,} ({100*n_healthy/n_elems:.1f}%)")
    print(f"      Border:  {n_border:,} ({100*n_border/n_elems:.1f}%)")
    print(f"      Infarct: {n_infarct:,} ({100*n_infarct/n_elems:.1f}%)")
    
    # Save outputs
    print("\n  Saving outputs")
    patient_output = os.path.join(OUTPUT_DIR, patient_id)
    os.makedirs(patient_output, exist_ok=True)
    
    write_vtk(
        os.path.join(patient_output, f"{patient_id}_classified.vtk"),
        coords, elements, classification, segments, anat_coords
    )
    
    write_region_vtk(
        os.path.join(patient_output, f"{patient_id}_INFARCT.vtk"),
        coords, elements, classification, TAG_INFARCT
    )
    
    write_region_vtk(
        os.path.join(patient_output, f"{patient_id}_BORDER.vtk"),
        coords, elements, classification, TAG_BORDER
    )
    
    write_tagged_elem(
        os.path.join(patient_output, f"{patient_id}_tagged.elem"),
        elements, classification
    )
    
    # Summary JSON
    summary = {
        'patient_id': patient_id,
        'timestamp': datetime.now().isoformat(),
        'method': 'Coronary Territory-Based Infarct Assignment',
        'pattern': {
            'name': pattern_info['name'],
            'description': pattern_info['description'],
            'core_segments': pattern_info['core_segments'],
            'border_segments': pattern_info['border_segments']
        },
        'statistics': {
            'n_elements': int(n_elems),
            'n_healthy': int(n_healthy),
            'n_border': int(n_border),
            'n_infarct': int(n_infarct),
            'pct_healthy': round(100*n_healthy/n_elems, 2),
            'pct_border': round(100*n_border/n_elems, 2),
            'pct_infarct': round(100*n_infarct/n_elems, 2),
        },
        'aha_segments': {str(k): int(v) for k, v in segment_counts.items()}
    }
    
    with open(os.path.join(patient_output, f"{patient_id}_summary.json"), 'w') as f:
        json.dump(summary, f, indent=2)
    
    print(f"\n  Outputs saved to: {patient_output}")
    
    return summary


def main():
    """Main entry point"""
    print("CORONARY TERRITORY-BASED INFARCT ASSIGNMENT")
    
    print("\nMETHOD:")
    print("  - AHA 17-segment model for anatomical localization")
    print("  - Coronary artery territory mapping (LAD, RCA, LCx)")
    print("  - Realistic infarct patterns based on clinical presentations")
    print("  - Smooth transitions with proper border zones")
    
    print("\nPATTERN DISTRIBUTION:")
    for name, prob in InfarctPatterns.PATTERN_WEIGHTS.items():
        pattern = InfarctPatterns.PATTERNS[name]
        print(f"  {prob*100:4.0f}% - {pattern['name']}")
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    all_results = []
    
    # Process each patient
    for patient_id in PATIENT_IDS:
        try:
            result = process_patient(patient_id)
            all_results.append(result)
        except Exception as e:
            print(f"\n  ERROR: {e}")
            import traceback
            traceback.print_exc()
            all_results.append({
                'patient_id': patient_id,
                'status': 'FAILED',
                'error': str(e)
            })
    
    # Summary table
    print("SUMMARY")
    
    print(f"\n{'Patient':<15} {'Pattern':<30} {'Healthy%':>10} {'Border%':>10} {'Infarct%':>10}")
    
    for result in all_results:
        if 'status' not in result:
            stats = result['statistics']
            pattern = result['pattern']['name'][:28]
            print(f"{result['patient_id']:<15} {pattern:<30} "
                  f"{stats['pct_healthy']:>10.1f} {stats['pct_border']:>10.1f} "
                  f"{stats['pct_infarct']:>10.1f}")
    
    # Combined summary
    successful = [r for r in all_results if 'status' not in r]
    if successful:
        avg_healthy = np.mean([r['statistics']['pct_healthy'] for r in successful])
        avg_border = np.mean([r['statistics']['pct_border'] for r in successful])
        avg_infarct = np.mean([r['statistics']['pct_infarct'] for r in successful])
        std_infarct = np.std([r['statistics']['pct_infarct'] for r in successful])
        
        print(f"{'AVERAGE':<15} {'':<30} {avg_healthy:>10.1f} {avg_border:>10.1f} {avg_infarct:>10.1f}")
        print(f"{'STD DEV':<15} {'':<30} {'':<10} {'':<10} {std_infarct:>10.1f}")
    
    # Save combined summary
    with open(os.path.join(OUTPUT_DIR, "all_patients_summary.json"), 'w') as f:
        json.dump({
            'timestamp': datetime.now().isoformat(),
            'method': 'Coronary Territory-Based Infarct Assignment',
            'patients': all_results
        }, f, indent=2)
    
    print(f"\nResults saved to: {OUTPUT_DIR}")
    
    return all_results


if __name__ == "__main__":
    results = main()

CORONARY TERRITORY-BASED INFARCT ASSIGNMENT

METHOD:
  - AHA 17-segment model for anatomical localization
  - Coronary artery territory mapping (LAD, RCA, LCx)
  - Realistic infarct patterns based on clinical presentations
  - Smooth transitions with proper border zones

PATTERN DISTRIBUTION:
    15% - LAD - Anterior MI
    15% - LAD - Anteroseptal MI
    10% - LAD - Extensive Anterior MI
    10% - LAD - Apical MI
    20% - RCA - Inferior MI
    10% - RCA - Inferoseptal MI
    12% - LCx - Lateral MI
     8% - LCx - Posterolateral MI
PROCESSING: SCD0000101

  Loading mesh...
      70,518 nodes, 383,439 elements

  Computing anatomical coordinate system...
      Long axis: [-0.117, -0.001, 0.993]

  Assigning AHA 17-segment model...
      Segments assigned: 17 of 17

  Selecting infarct pattern...
      Pattern: RCA - Inferior MI
      RCA occlusion - inferior wall
      Core segments: [4, 10, 15]
      Border segments: [3, 5, 9, 11, 14, 16]

  Assigning infarct territory...

  Smoothing

In [5]:
#!/usr/bin/env python3
"""
COMPLETE LAPLACE-DIRICHLET CARDIAC ANALYSIS FRAMEWORK


COMPONENT 1: LAPLACE-DIRICHLET TRANSMURAL COORDINATE
    - Solve ∇²φ = 0 with Dirichlet BCs
    - φ ∈ [0,1] defines transmural position
    
COMPONENT 2: HELICAL FIBER RECONSTRUCTION VIA φ
    - α(φ) = α_endo + (α_epi - α_endo) × φ
    - Fiber: f = cos(α)·e_c + sin(α)·e_l
    - Modified in infarct regions
    
COMPONENT 3: WALL STRESS FROM MODIFIED LAPLACE LAW WITH CURVATURE TENSOR
    - Classical: σ = Pr/(2h)
    - Modified: σ = Pr/(2h) × f(κ₁, κ₂)
    - κ₁, κ₂ = principal curvatures from shape operator
    
COMPONENT 4: LAPLACIAN-GEODESIC INJECTION SITE OPTIMIZATION
    - Geodesic distance via Heat Method (Crane et al. 2013)
    - Optimization: min(geodesic_to_border) + max(stress_reduction)
    - Constraints: avoid core scar, prefer mid-wall

"""

import numpy as np
from scipy.sparse import lil_matrix, csr_matrix, diags
from scipy.sparse.linalg import spsolve, eigsh
from scipy.spatial import cKDTree
from collections import defaultdict
import os
import json
from datetime import datetime
from multiprocessing import Pool, cpu_count
from functools import partial
import warnings
warnings.filterwarnings('ignore')

# Try to import joblib for parallel loops (more notebook-friendly)
try:
    from joblib import Parallel, delayed
    HAS_JOBLIB = True
except ImportError:
    HAS_JOBLIB = False
    print("Note: joblib not available, using sequential processing for loops")

# PARALLEL PROCESSING CONFIGURATION
N_CPUS = 44  # Number of CPUs to use
PARALLEL_PATIENTS = True  # Process patients in parallel
PARALLEL_ELEMENTS = True  # Parallelize element loops within each patient


PATIENT_IDS = [
    "SCD0000101", "SCD0000201", "SCD0000301", "SCD0000401",
    "SCD0000601", "SCD0000701", "SCD0000801", "SCD0001001",
    "SCD0001101", "SCD0001201"
]

BASE_DIR = "/home/shadeform/SCD_MODELS"
OUTPUT_DIR = "/home/shadeform/SCD_MODELS/laplace_complete"

# Fiber parameters (Streeter et al. 1969)
FIBER_ANGLE_ENDO_DEG = 60.0    # Subendocardial: right-handed helix
FIBER_ANGLE_EPI_DEG = -60.0    # Subepicardial: left-handed helix

# Pressure for wall stress
LV_PRESSURE_KPA = 16.0         # Peak systolic

# Injection optimization
N_INJECTION_SITES = 5
MIN_SITE_SEPARATION_MM = 15.0

# Tissue tags
TAG_HEALTHY = 1
TAG_BORDER = 2
TAG_INFARCT = 3


# MESH LOADING

def load_mesh(patient_id, base_dir):
    """Load tetrahedral mesh"""
    pts_file = f"{base_dir}/simulation_ready/{patient_id}/{patient_id}_tet.pts"
    elem_file = f"{base_dir}/simulation_ready/{patient_id}/{patient_id}_tet.elem"
    
    # Load coordinates
    with open(pts_file, 'r') as f:
        n_nodes = int(f.readline().strip())
        coords = np.zeros((n_nodes, 3), dtype=np.float64)
        for i in range(n_nodes):
            coords[i] = [float(x) for x in f.readline().split()[:3]]
    
    # Load elements
    with open(elem_file, 'r') as f:
        n_elems = int(f.readline().strip())
        elements = np.zeros((n_elems, 4), dtype=np.int32)
        for i in range(n_elems):
            parts = f.readline().split()
            elements[i] = [int(x) for x in parts[1:5]]
    
    return coords, elements


def load_classification(patient_id, base_dir):
    """Load tissue classification from coronary territory-based detection"""
    # Try new coronary territory results first, then fall back to comprehensive
    class_dir = f"{base_dir}/infarct_coronary_territory/{patient_id}"
    elem_file = f"{class_dir}/{patient_id}_tagged.elem"
    
    if not os.path.exists(elem_file):
        # Fallback to old location
        class_dir = f"{base_dir}/infarct_results_comprehensive/{patient_id}"
        elem_file = f"{class_dir}/{patient_id}_tagged.elem"
    
    if not os.path.exists(elem_file):
        return None
    
    with open(elem_file, 'r') as f:
        n_elems = int(f.readline().strip())
        tags = np.zeros(n_elems, dtype=np.int32)
        for i in range(n_elems):
            parts = f.readline().split()
            tags[i] = int(parts[5])
    
    return tags


# COMPONENT 1: LAPLACE-DIRICHLET TRANSMURAL COORDINATE

def extract_surfaces(coords, elements):
    """Extract endocardial and epicardial surfaces"""
    # Find boundary faces
    face_count = defaultdict(list)
    for elem_idx, elem in enumerate(elements):
        faces = [
            tuple(sorted([elem[0], elem[1], elem[2]])),
            tuple(sorted([elem[0], elem[1], elem[3]])),
            tuple(sorted([elem[0], elem[2], elem[3]])),
            tuple(sorted([elem[1], elem[2], elem[3]])),
        ]
        for face in faces:
            face_count[face].append(elem_idx)
    
    boundary_faces = [(f, elems[0]) for f, elems in face_count.items() if len(elems) == 1]
    
    # Get boundary nodes
    boundary_nodes = set()
    for face, _ in boundary_faces:
        boundary_nodes.update(face)
    boundary_nodes = np.array(list(boundary_nodes))
    
    # Classify by radial position
    z_vals = coords[:, 2]
    z_min, z_max = z_vals.min(), z_vals.max()
    z_range = z_max - z_min
    
    endo_nodes = set()
    epi_nodes = set()
    endo_faces = []
    epi_faces = []
    
    for i in range(20):
        z_lo = z_min + i * z_range / 20
        z_hi = z_min + (i + 1) * z_range / 20
        
        slice_mask = (coords[boundary_nodes, 2] >= z_lo) & (coords[boundary_nodes, 2] < z_hi)
        slice_nodes = boundary_nodes[slice_mask]
        
        if len(slice_nodes) < 20:
            continue
        
        slice_coords = coords[slice_nodes, :2]
        center = np.median(slice_coords, axis=0)
        radii = np.linalg.norm(slice_coords - center, axis=1)
        
        r_30 = np.percentile(radii, 30)
        r_70 = np.percentile(radii, 70)
        
        for j, node in enumerate(slice_nodes):
            if radii[j] < r_30:
                endo_nodes.add(node)
            elif radii[j] > r_70:
                epi_nodes.add(node)
    
    # Classify faces
    for face, elem_idx in boundary_faces:
        face_nodes = list(face)
        if all(n in endo_nodes for n in face_nodes):
            endo_faces.append(face_nodes)
        elif all(n in epi_nodes for n in face_nodes):
            epi_faces.append(face_nodes)
    
    return {
        'endo_nodes': np.array(list(endo_nodes)),
        'epi_nodes': np.array(list(epi_nodes)),
        'endo_faces': np.array(endo_faces) if endo_faces else np.array([]).reshape(0, 3),
        'epi_faces': np.array(epi_faces) if epi_faces else np.array([]).reshape(0, 3),
    }


def build_fem_laplacian(coords, elements):
    """Build FEM Laplacian stiffness matrix"""
    n_nodes = len(coords)
    K = lil_matrix((n_nodes, n_nodes))
    
    for elem in elements:
        X = coords[elem]
        J = np.array([X[1] - X[0], X[2] - X[0], X[3] - X[0]]).T
        
        detJ = np.linalg.det(J)
        if abs(detJ) < 1e-15:
            continue
        
        vol = abs(detJ) / 6.0
        Jinv = np.linalg.inv(J)
        
        # Shape function gradients
        dN = np.zeros((4, 3))
        dN[0] = -Jinv.sum(axis=1)
        dN[1] = Jinv[:, 0]
        dN[2] = Jinv[:, 1]
        dN[3] = Jinv[:, 2]
        
        # Element stiffness
        Ke = vol * (dN @ dN.T)
        
        for i in range(4):
            for j in range(4):
                K[elem[i], elem[j]] += Ke[i, j]
    
    return K.tocsr()


def solve_laplace_dirichlet(coords, elements, surfaces):
    """
    COMPONENT 1: Solve Laplace equation for transmural coordinate.
    
    Mathematical Formulation:
    ∇²φ = 0           in Ω (myocardium)
    φ = 0             on Γ_endo
    φ = 1             on Γ_epi
    
    Returns: φ(x) ∈ [0, 1] - smooth transmural scalar field
    """
    n_nodes = len(coords)
    
    print("      Building FEM Laplacian")
    K = build_fem_laplacian(coords, elements)
    
    endo_set = set(surfaces['endo_nodes'])
    epi_set = set(surfaces['epi_nodes'])
    
    print(f"      Dirichlet BCs: {len(endo_set)} endo, {len(epi_set)} epi nodes")
    
    # Apply Dirichlet BCs via penalty method
    K_mod = K.tolil()
    rhs = np.zeros(n_nodes)
    penalty = 1e12
    
    for node in endo_set:
        K_mod[node, :] = 0
        K_mod[node, node] = penalty
        rhs[node] = 0.0 * penalty
    
    for node in epi_set:
        K_mod[node, :] = 0
        K_mod[node, node] = penalty
        rhs[node] = 1.0 * penalty
    
    print("      Solving Laplace equation")
    phi = spsolve(K_mod.tocsr(), rhs)
    phi = np.clip(phi, 0, 1)
    
    return phi


def compute_phi_gradient(coords, elements, phi):
    """Compute gradient of transmural coordinate at each element"""
    n_elems = len(elements)
    grad_phi = np.zeros((n_elems, 3))
    
    for i, elem in enumerate(elements):
        X = coords[elem]
        phi_elem = phi[elem]
        
        J = np.array([X[1] - X[0], X[2] - X[0], X[3] - X[0]]).T
        detJ = np.linalg.det(J)
        
        if abs(detJ) < 1e-15:
            continue
        
        Jinv = np.linalg.inv(J)
        dphi_dxi = np.array([phi_elem[1] - phi_elem[0], 
                            phi_elem[2] - phi_elem[0], 
                            phi_elem[3] - phi_elem[0]])
        grad_phi[i] = Jinv @ dphi_dxi
    
    return grad_phi


# COMPONENT 2: HELICAL FIBER RECONSTRUCTION VIA TRANSMURAL COORDINATE

def compute_local_coordinate_system(coords, elements, phi, grad_phi):
    """
    Compute local orthonormal coordinate system at each element.
    
    e_t: Transmural direction = ∇φ / |∇φ|
    e_l: Longitudinal direction (apex → base), orthogonalized
    e_c: Circumferential direction = e_l × e_t
    """
    n_elems = len(elements)
    
    # Transmural direction
    grad_norm = np.linalg.norm(grad_phi, axis=1, keepdims=True)
    grad_norm = np.maximum(grad_norm, 1e-10)
    e_t = grad_phi / grad_norm
    
    # Long axis direction (roughly +z, pointing base-ward)
    centroids = np.mean(coords[elements], axis=1)
    
    # Find apex and base
    z_vals = centroids[:, 2]
    apex_z = z_vals.min()
    base_z = z_vals.max()
    
    # Initial longitudinal direction (towards base)
    e_l_init = np.zeros((n_elems, 3))
    e_l_init[:, 2] = 1.0  # +z direction
    
    # Orthogonalize e_l to e_t
    e_l = np.zeros((n_elems, 3))
    for i in range(n_elems):
        # Gram-Schmidt: e_l = e_l_init - (e_l_init · e_t) e_t
        proj = np.dot(e_l_init[i], e_t[i])
        e_l[i] = e_l_init[i] - proj * e_t[i]
        norm = np.linalg.norm(e_l[i])
        if norm > 1e-10:
            e_l[i] /= norm
        else:
            # Fallback: use perpendicular direction
            e_l[i] = np.cross(e_t[i], [1, 0, 0])
            norm = np.linalg.norm(e_l[i])
            if norm > 1e-10:
                e_l[i] /= norm
    
    # Circumferential direction: e_c = e_l × e_t
    e_c = np.cross(e_l, e_t)
    e_c /= np.linalg.norm(e_c, axis=1, keepdims=True) + 1e-10
    
    return e_t, e_l, e_c


def reconstruct_helical_fibers(coords, elements, phi, grad_phi, tags=None,
                               alpha_endo=FIBER_ANGLE_ENDO_DEG,
                               alpha_epi=FIBER_ANGLE_EPI_DEG):
    """
    COMPONENT 2: Reconstruct myofibers via helical angle rotation.
    
    Mathematical Formulation:
    α(φ) = α_endo + (α_epi - α_endo) × φ
         = 60° - 120° × φ
    
    Fiber direction:
    f = cos(α) · e_c + sin(α) · e_l
    
    Sheet direction:
    s = f × e_t (perpendicular to fiber, in wall plane)
    
    In infarct regions:
    - Core: α = 0° (circumferential, no helical rotation)
    - Border: α reduced to ±30° (partial preservation)
    
    Reference: Streeter et al. "Fiber orientation in the canine left ventricle 
               during diastole and systole." Circ Res 1969.
    """
    n_elems = len(elements)
    
    print("      Computing local coordinate system")
    e_t, e_l, e_c = compute_local_coordinate_system(coords, elements, phi, grad_phi)
    
    # Transmural position per element
    phi_elem = np.array([np.mean(phi[elem]) for elem in elements])
    
    # Fiber angle: linear interpolation through wall
    alpha_endo_rad = np.radians(alpha_endo)
    alpha_epi_rad = np.radians(alpha_epi)
    
    alpha = alpha_endo_rad + (alpha_epi_rad - alpha_endo_rad) * phi_elem
    
    # Modify in infarct regions
    if tags is not None:
        for i in range(n_elems):
            if tags[i] == TAG_INFARCT:
                # Core scar: no helical rotation (fibrotic, circumferential only)
                alpha[i] = 0.0
            elif tags[i] == TAG_BORDER:
                # Border zone: reduced rotation (50% of normal)
                alpha[i] = alpha[i] * 0.5
    
    # Compute fiber and sheet directions
    print("      Computing fiber orientations via helical rotation")
    fibers = np.zeros((n_elems, 3))
    sheets = np.zeros((n_elems, 3))
    
    for i in range(n_elems):
        # Fiber: f = cos(α) e_c + sin(α) e_l
        fibers[i] = np.cos(alpha[i]) * e_c[i] + np.sin(alpha[i]) * e_l[i]
        fibers[i] /= np.linalg.norm(fibers[i]) + 1e-10
        
        # Sheet: s = f × e_t
        sheets[i] = np.cross(fibers[i], e_t[i])
        sheets[i] /= np.linalg.norm(sheets[i]) + 1e-10
    
    fiber_angles_deg = np.degrees(alpha)
    
    print(f"      Fiber angle range: {fiber_angles_deg.min():.1f}° to {fiber_angles_deg.max():.1f}°")
    
    return fibers, sheets, fiber_angles_deg, (e_t, e_l, e_c)


# COMPONENT 3: WALL STRESS FROM MODIFIED LAPLACE LAW WITH CURVATURE TENSOR

def compute_surface_curvature_tensor(coords, faces, vertex_normals):
    """
    Compute principal curvatures from the shape operator (Weingarten map).
    
    The shape operator S relates the change in normal to surface position:
    S = -dN/dX
    
    Principal curvatures κ₁, κ₂ are the eigenvalues of S.
    Mean curvature: H = (κ₁ + κ₂) / 2
    Gaussian curvature: K = κ₁ × κ₂
    """
    n_nodes = len(coords)
    
    # Build vertex-to-face connectivity
    node_faces = defaultdict(list)
    for fi, face in enumerate(faces):
        for node in face:
            node_faces[node].append(fi)
    
    kappa_1 = np.zeros(n_nodes)  # Max principal curvature
    kappa_2 = np.zeros(n_nodes)  # Min principal curvature
    mean_curvature = np.zeros(n_nodes)
    gaussian_curvature = np.zeros(n_nodes)
    
    for node in node_faces.keys():
        if len(node_faces[node]) < 3:
            continue
        
        p = coords[node]
        n = vertex_normals[node]
        
        if np.linalg.norm(n) < 0.1:
            continue
        
        # Find neighbors
        neighbors = set()
        for fi in node_faces[node]:
            neighbors.update(faces[fi])
        neighbors.discard(node)
        
        if len(neighbors) < 3:
            continue
        
        neighbor_list = list(neighbors)
        neighbor_coords = coords[neighbor_list]
        neighbor_normals = vertex_normals[neighbor_list]
        
        # Project to tangent plane
        diffs = neighbor_coords - p
        
        # Build tangent basis
        t1 = diffs[0] - np.dot(diffs[0], n) * n
        if np.linalg.norm(t1) < 1e-10:
            continue
        t1 /= np.linalg.norm(t1)
        t2 = np.cross(n, t1)
        
        # Fit shape operator using least squares
        # dn = S @ dx in tangent coordinates
        A = []
        b = []
        for i, neighbor in enumerate(neighbor_list):
            dx = diffs[i]
            dn = neighbor_normals[i] - n
            
            # Project to tangent plane
            dx_t = np.array([np.dot(dx, t1), np.dot(dx, t2)])
            dn_t = np.array([np.dot(dn, t1), np.dot(dn, t2)])
            
            if np.linalg.norm(dx_t) > 1e-10:
                # Shape operator equation: dn = -S @ dx
                A.append([dx_t[0], dx_t[1], 0, 0])
                A.append([0, 0, dx_t[0], dx_t[1]])
                b.append(-dn_t[0])
                b.append(-dn_t[1])
        
        if len(A) < 4:
            continue
        
        A = np.array(A)
        b = np.array(b)
        
        # Solve for shape operator components [S11, S12, S21, S22]
        try:
            S_flat, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
            S = np.array([[S_flat[0], S_flat[1]], 
                         [S_flat[2], S_flat[3]]])
            
            # Symmetrize (shape operator should be symmetric)
            S = (S + S.T) / 2
            
            # Eigenvalues = principal curvatures
            eigenvalues = np.linalg.eigvalsh(S)
            kappa_1[node] = np.max(eigenvalues)
            kappa_2[node] = np.min(eigenvalues)
            mean_curvature[node] = (kappa_1[node] + kappa_2[node]) / 2
            gaussian_curvature[node] = kappa_1[node] * kappa_2[node]
        except:
            pass
    
    return {
        'kappa_1': kappa_1,
        'kappa_2': kappa_2,
        'mean_curvature': mean_curvature,
        'gaussian_curvature': gaussian_curvature
    }


def compute_vertex_normals(coords, faces):
    """Compute area-weighted vertex normals from surface faces"""
    n_nodes = len(coords)
    vertex_normals = np.zeros((n_nodes, 3))
    
    for face in faces:
        v0, v1, v2 = coords[face]
        e1 = v1 - v0
        e2 = v2 - v0
        n = np.cross(e1, e2)
        area = np.linalg.norm(n) / 2
        if area > 1e-10:
            n = n / (2 * area)  # Unit normal
            for node in face:
                vertex_normals[node] += n * area
    
    # Normalize
    norms = np.linalg.norm(vertex_normals, axis=1, keepdims=True)
    norms = np.maximum(norms, 1e-10)
    vertex_normals /= norms
    
    return vertex_normals


def compute_wall_stress_with_curvature(coords, elements, phi, grad_phi, surfaces,
                                        pressure=LV_PRESSURE_KPA, tags=None):
    """
    COMPONENT 3: Wall stress from modified Law of Laplace with curvature tensor.
    
    Mathematical Formulation:
    
    Classical Laplace Law (thin-walled sphere):
    σ = P × r / (2 × h)
    
    Modified with local curvature tensor:
    σ(x) = P × r_local(x) / (2 × h(x)) × f(κ₁, κ₂)
    
    Where:
    - h(x) = 1/|∇φ| = local wall thickness
    - r_local = 1/H = local radius of curvature (H = mean curvature)
    - κ₁, κ₂ = principal curvatures from shape operator
    - f(κ₁, κ₂) = curvature anisotropy factor = 1 + |κ₁/κ₂ - 1| × 0.5
    
    Reference: Zhong et al. "Finite element analysis of the stress distribution 
               in left ventricle aneurysm." Int J Cardiol 2008.
    """
    n_elems = len(elements)
    centroids = np.mean(coords[elements], axis=1)
    
    # Wall thickness from Laplace gradient: h = 1/|∇φ|
    grad_norm = np.linalg.norm(grad_phi, axis=1)
    grad_norm = np.maximum(grad_norm, 1e-8)
    wall_thickness = 1.0 / grad_norm
    
    # Clip to physiological bounds
    wall_thickness = np.clip(wall_thickness, 0.5, 25.0)
    
    print("      Computing surface curvature tensor")
    
    # Compute curvature on endocardial surface
    if len(surfaces['endo_faces']) > 0:
        vertex_normals = compute_vertex_normals(coords, surfaces['endo_faces'])
        curvature = compute_surface_curvature_tensor(coords, surfaces['endo_faces'], vertex_normals)
    else:
        # Fallback: estimate from radial position
        curvature = {'kappa_1': np.zeros(len(coords)), 
                     'kappa_2': np.zeros(len(coords)),
                     'mean_curvature': np.zeros(len(coords))}
    
    # Interpolate curvature to elements
    kappa_1_elem = np.zeros(n_elems)
    kappa_2_elem = np.zeros(n_elems)
    mean_curv_elem = np.zeros(n_elems)
    
    for i, elem in enumerate(elements):
        kappa_1_elem[i] = np.mean(curvature['kappa_1'][elem])
        kappa_2_elem[i] = np.mean(curvature['kappa_2'][elem])
        mean_curv_elem[i] = np.mean(curvature['mean_curvature'][elem])
    
    # Local radius of curvature: r = 1/|H|
    mean_curv_elem = np.maximum(np.abs(mean_curv_elem), 1e-6)
    local_radius = 1.0 / mean_curv_elem
    local_radius = np.clip(local_radius, 10, 200)  # mm bounds
    
    # Curvature anisotropy factor: f(κ₁, κ₂) = 1 + |κ₁/κ₂ - 1| × 0.5
    kappa_2_safe = np.where(np.abs(kappa_2_elem) > 1e-8, kappa_2_elem, 1e-8)
    anisotropy = np.abs(kappa_1_elem / kappa_2_safe - 1.0)
    curvature_factor = 1.0 + anisotropy * 0.5
    curvature_factor = np.clip(curvature_factor, 1.0, 2.5)
    
    # For elements far from surface, use geometric estimate
    z_vals = centroids[:, 2]
    for i in range(n_elems):
        if local_radius[i] > 150 or local_radius[i] < 15:
            # Estimate from radial position at this z-level
            z = z_vals[i]
            z_mask = np.abs(z_vals - z) < 5.0
            if np.sum(z_mask) > 10:
                level_centroids = centroids[z_mask, :2]
                center = np.mean(level_centroids, axis=0)
                radii = np.linalg.norm(level_centroids - center, axis=1)
                local_radius[i] = np.mean(radii)
    
    # Modified Law of Laplace: σ = P × r / (2h) × f(κ)
    wall_stress = (pressure * local_radius) / (2 * wall_thickness) * curvature_factor
    
    # Modify for tissue type
    if tags is not None:
        for i in range(n_elems):
            if tags[i] == TAG_INFARCT:
                # Scar is stiffer, but load is transferred to border
                wall_stress[i] *= 0.7
            elif tags[i] == TAG_BORDER:
                # Stress concentration at border!
                wall_stress[i] *= 1.5
    
    print(f"      Wall thickness: {wall_thickness.min():.2f} - {wall_thickness.max():.2f} mm")
    print(f"      Local radius: {local_radius.min():.2f} - {local_radius.max():.2f} mm")
    print(f"      Wall stress: {wall_stress.min():.2f} - {wall_stress.max():.2f} kPa")
    
    return {
        'wall_stress': wall_stress,
        'wall_thickness': wall_thickness,
        'local_radius': local_radius,
        'curvature_factor': curvature_factor,
        'kappa_1': kappa_1_elem,
        'kappa_2': kappa_2_elem,
        'mean_curvature': mean_curv_elem
    }


# COMPONENT 4: LAPLACIAN-GEODESIC INJECTION SITE OPTIMIZATION

def build_mass_matrix(coords, elements):
    """Build lumped mass matrix for heat equation"""
    n_nodes = len(coords)
    M = np.zeros(n_nodes)
    
    for elem in elements:
        X = coords[elem]
        J = np.array([X[1] - X[0], X[2] - X[0], X[3] - X[0]]).T
        vol = abs(np.linalg.det(J)) / 6.0
        
        for node in elem:
            M[node] += vol / 4.0
    
    return diags(M)


def compute_geodesic_heat_method(coords, elements, source_nodes, L, t_factor=1.0):
    """
    Compute geodesic distance via the Heat Method (Crane et al. 2013).
    
    Algorithm:
    1. Solve heat equation: (M - tL)u = δ_source
    2. Compute normalized gradient: X = -∇u / |∇u|
    3. Solve Poisson equation: Lφ = ∇·X
    
    The solution φ gives approximate geodesic distances.
    
    Reference: Crane, Weischedel, Wardetzky. "Geodesics in Heat: A New Approach 
               to Computing Distance Based on Heat Flow." ACM TOG 2013.
    """
    n_nodes = len(coords)
    n_elems = len(elements)
    
    # Build mass matrix
    M = build_mass_matrix(coords, elements)
    
    # Estimate time step from mean edge length
    edge_lengths = []
    for elem in elements[:min(1000, n_elems)]:
        for i in range(4):
            for j in range(i+1, 4):
                edge_lengths.append(np.linalg.norm(coords[elem[i]] - coords[elem[j]]))
    h = np.mean(edge_lengths)
    t = t_factor * h * h
    
    # Step 1: Solve heat equation (M - tL)u = b
    A = M - t * L
    
    b = np.zeros(n_nodes)
    for node in source_nodes:
        b[node] = 1.0
    
    u = spsolve(A.tocsr(), b)
    
    # Step 2: Compute normalized gradient field
    X = np.zeros((n_elems, 3))
    
    for i, elem in enumerate(elements):
        verts = coords[elem]
        u_elem = u[elem]
        
        J = np.array([verts[1] - verts[0], verts[2] - verts[0], verts[3] - verts[0]]).T
        detJ = np.linalg.det(J)
        
        if abs(detJ) < 1e-15:
            continue
        
        Jinv = np.linalg.inv(J)
        du_dxi = np.array([u_elem[1] - u_elem[0], u_elem[2] - u_elem[0], u_elem[3] - u_elem[0]])
        grad_u = Jinv @ du_dxi
        
        norm = np.linalg.norm(grad_u)
        if norm > 1e-10:
            X[i] = -grad_u / norm  # Normalize and flip
    
    # Step 3: Compute divergence and solve Poisson
    div_X = np.zeros(n_nodes)
    
    for i, elem in enumerate(elements):
        verts = coords[elem]
        J = np.array([verts[1] - verts[0], verts[2] - verts[0], verts[3] - verts[0]]).T
        detJ = np.linalg.det(J)
        
        if abs(detJ) < 1e-15:
            continue
        
        vol = abs(detJ) / 6.0
        Jinv = np.linalg.inv(J)
        
        # Shape function gradients
        dN = np.zeros((4, 3))
        dN[0] = -Jinv.sum(axis=1)
        dN[1] = Jinv[:, 0]
        dN[2] = Jinv[:, 1]
        dN[3] = Jinv[:, 2]
        
        for j in range(4):
            div_X[elem[j]] += vol * np.dot(dN[j], X[i])
    
    # Fix one node for Poisson equation
    L_mod = L.tolil()
    ref_node = source_nodes[0]
    L_mod[ref_node, :] = 0
    L_mod[ref_node, ref_node] = 1
    div_X[ref_node] = 0
    
    phi = spsolve(L_mod.tocsr(), div_X)
    phi = phi - phi[source_nodes].min()
    
    return phi


def optimize_injection_sites(coords, elements, tags, wall_stress, phi_trans,
                             geodesic_to_border, centroids, n_sites=5):
    """
    COMPONENT 4: Laplacian-geodesic optimization for injection sites.
    
    Optimization Objective:
    Maximize: score(x) = w₁ × proximity_to_border + w₂ × stress_reduction + 
                         w₃ × mid_wall_preference + w₄ × border_bonus
    
    Subject to:
    - Avoid core scar (no perfusion → injection won't diffuse)
    - Maintain spatial separation between sites (min_separation)
    - Prefer border zone (therapeutic target)
    - Prefer mid-wall (φ ≈ 0.5) for better distribution
    
    The geodesic distance ensures optimal path lengths on the manifold,
    while the stress component targets high-risk regions for remodeling.
    """
    n_elems = len(elements)
    
    # Transmural position per element
    phi_elem = np.array([np.mean(phi_trans[elem]) for elem in elements])
    
    # Geodesic distance to elements (average of node distances)
    geodesic_elem = np.array([np.mean(geodesic_to_border[elem]) for elem in elements])
    
    # Normalize stress
    stress_norm = wall_stress['wall_stress'] / np.median(wall_stress['wall_stress'])
    
    # Initialize scores
    scores = np.zeros(n_elems)
    
    for i in range(n_elems):
        # EXCLUDE: Core scar (no perfusion)
        if tags[i] == TAG_INFARCT:
            scores[i] = -np.inf
            continue
        
        # Component 1: Proximity to border zone (minimize geodesic distance)
        # Higher score when CLOSER to border
        dist_score = 1.0 / (geodesic_elem[i] + 1.0)
        
        # Component 2: Wall stress reduction potential (target high stress)
        stress_score = stress_norm[i]
        
        # Component 3: Mid-wall preference (φ ≈ 0.5)
        mid_wall_score = 1.0 - 2.0 * abs(phi_elem[i] - 0.5)
        
        # Component 4: Border zone bonus
        border_bonus = 2.0 if tags[i] == TAG_BORDER else 1.0
        
        # Combined score with weights
        scores[i] = (0.3 * dist_score + 
                     0.3 * stress_score + 
                     0.2 * mid_wall_score + 
                     0.2 * border_bonus)
    
    # Select top sites with spatial diversity
    selected_sites = []
    sorted_indices = np.argsort(scores)[::-1]
    
    for idx in sorted_indices:
        if len(selected_sites) >= n_sites:
            break
        
        if scores[idx] == -np.inf:
            continue
        
        # Check spatial separation
        centroid = centroids[idx]
        too_close = False
        
        for site_idx in selected_sites:
            dist = np.linalg.norm(centroid - centroids[site_idx])
            if dist < MIN_SITE_SEPARATION_MM:
                too_close = True
                break
        
        if not too_close:
            selected_sites.append(idx)
    
    # Compile results
    injection_sites = []
    for idx in selected_sites:
        injection_sites.append({
            'element_id': int(idx),
            'coordinates': centroids[idx].tolist(),
            'score': float(scores[idx]),
            'transmural_position': float(phi_elem[idx]),
            'wall_stress_kPa': float(wall_stress['wall_stress'][idx]),
            'geodesic_distance': float(geodesic_elem[idx]),
            'tissue_type': 'border' if tags[idx] == TAG_BORDER else 'healthy'
        })
    
    return injection_sites


# OUTPUT FUNCTIONS

def write_vtk_complete(filepath, coords, elements, scalars, vectors):
    """Write VTK with all computed fields"""
    n_nodes, n_elems = len(coords), len(elements)
    
    with open(filepath, 'w') as f:
        f.write("# vtk DataFile Version 3.0\n")
        f.write("Complete Laplace-Dirichlet Analysis\n")
        f.write("ASCII\n")
        f.write("DATASET UNSTRUCTURED_GRID\n")
        
        f.write(f"POINTS {n_nodes} float\n")
        for c in coords:
            f.write(f"{c[0]:.6f} {c[1]:.6f} {c[2]:.6f}\n")
        
        f.write(f"\nCELLS {n_elems} {n_elems * 5}\n")
        for e in elements:
            f.write(f"4 {e[0]} {e[1]} {e[2]} {e[3]}\n")
        
        f.write(f"\nCELL_TYPES {n_elems}\n")
        f.write("10\n" * n_elems)
        
        # Point data
        point_scalars = {k: v for k, v in scalars.items() if len(v) == n_nodes}
        if point_scalars:
            f.write(f"\nPOINT_DATA {n_nodes}\n")
            for name, data in point_scalars.items():
                f.write(f"SCALARS {name} float 1\nLOOKUP_TABLE default\n")
                for val in data:
                    f.write(f"{float(val):.6f}\n")
        
        # Cell data
        cell_scalars = {k: v for k, v in scalars.items() if len(v) == n_elems}
        if cell_scalars or vectors:
            f.write(f"\nCELL_DATA {n_elems}\n")
            
            for name, data in cell_scalars.items():
                f.write(f"SCALARS {name} float 1\nLOOKUP_TABLE default\n")
                for val in data:
                    f.write(f"{float(val):.6f}\n")
            
            for name, data in vectors.items():
                if len(data) == n_elems:
                    f.write(f"\nVECTORS {name} float\n")
                    for v in data:
                        f.write(f"{v[0]:.6f} {v[1]:.6f} {v[2]:.6f}\n")


def write_lon_file(filepath, fibers, sheets):
    """Write CARP-format fiber file"""
    with open(filepath, 'w') as f:
        f.write("2\n")  # 2 directions
        for fiber, sheet in zip(fibers, sheets):
            f.write(f"{fiber[0]:.6f} {fiber[1]:.6f} {fiber[2]:.6f} "
                   f"{sheet[0]:.6f} {sheet[1]:.6f} {sheet[2]:.6f}\n")


def write_injection_coords(filepath, sites):
    """Write injection site coordinates"""
    with open(filepath, 'w') as f:
        f.write("# Optimal injection sites from Laplacian-geodesic optimization\n")
        f.write("# X Y Z score stress_kPa geodesic_dist tissue_type\n")
        for site in sites:
            c = site['coordinates']
            f.write(f"{c[0]:.4f} {c[1]:.4f} {c[2]:.4f} "
                   f"{site['score']:.4f} {site['wall_stress_kPa']:.2f} "
                   f"{site['geodesic_distance']:.2f} {site['tissue_type']}\n")


# MAIN PIPELINE

def process_patient_complete(patient_id):
    """
    Complete Laplace-Dirichlet analysis implementing ALL components from the abstract.
    """
    print(f"PROCESSING: {patient_id}")
    
    results = {'patient_id': patient_id}
    
    # Load mesh
    print("\n  Loading mesh")
    coords, elements = load_mesh(patient_id, BASE_DIR)
    n_nodes, n_elems = len(coords), len(elements)
    print(f"      {n_nodes:,} nodes, {n_elems:,} elements")
    
    centroids = np.mean(coords[elements], axis=1)
    
    # Load tissue classification
    tags = load_classification(patient_id, BASE_DIR)
    if tags is None:
        print("      No classification found - using all healthy")
        tags = np.ones(n_elems, dtype=np.int32)
    else:
        print(f"      Classification loaded: {np.sum(tags==TAG_INFARCT)} infarct, "
              f"{np.sum(tags==TAG_BORDER)} border")
    
    # Extract surfaces
    print("\n  [1] LAPLACE-DIRICHLET TRANSMURAL COORDINATE")
    print("      Extracting surfaces")
    surfaces = extract_surfaces(coords, elements)
    print(f"      Endo: {len(surfaces['endo_nodes'])} nodes, "
          f"Epi: {len(surfaces['epi_nodes'])} nodes")
    
    # Solve Laplace-Dirichlet
    phi = solve_laplace_dirichlet(coords, elements, surfaces)
    print(f"      φ range: [{phi.min():.4f}, {phi.max():.4f}]")
    
    # Compute gradient
    grad_phi = compute_phi_gradient(coords, elements, phi)
    
    # COMPONENT 2: Helical Fiber Reconstruction
    print("\n  [2] HELICAL FIBER RECONSTRUCTION")
    fibers, sheets, fiber_angles, coord_system = reconstruct_helical_fibers(
        coords, elements, phi, grad_phi, tags,
        FIBER_ANGLE_ENDO_DEG, FIBER_ANGLE_EPI_DEG
    )
    
    # COMPONENT 3: Wall Stress with Curvature Tensor
    print("\n  [3] WALL STRESS (MODIFIED LAPLACE + CURVATURE TENSOR)")
    stress_data = compute_wall_stress_with_curvature(
        coords, elements, phi, grad_phi, surfaces, LV_PRESSURE_KPA, tags
    )
    
    # COMPONENT 4: Laplacian-Geodesic Injection Optimization
    print("\n  [4] LAPLACIAN-GEODESIC INJECTION OPTIMIZATION")
    
    # Compute geodesic distance to border zone
    border_elements = np.where(tags == TAG_BORDER)[0]
    if len(border_elements) > 0:
        border_nodes = list(set(elements[border_elements[:100]].flatten()))
        
        print("      Computing geodesic distances (Heat Method)")
        L = build_fem_laplacian(coords, elements)
        geodesic_to_border = compute_geodesic_heat_method(coords, elements, border_nodes, L)
        print(f"      Geodesic range: [{geodesic_to_border.min():.2f}, "
              f"{geodesic_to_border.max():.2f}]")
    else:
        print("      No border zone - using endo distance")
        geodesic_to_border = np.zeros(n_nodes)
        endo_tree = cKDTree(coords[surfaces['endo_nodes']])
        for i, c in enumerate(coords):
            d, _ = endo_tree.query(c)
            geodesic_to_border[i] = d
    
    # Optimize injection sites
    print("      Optimizing injection sites")
    injection_sites = optimize_injection_sites(
        coords, elements, tags, stress_data, phi,
        geodesic_to_border, centroids, N_INJECTION_SITES
    )
    
    print(f"      Selected {len(injection_sites)} optimal sites:")
    for site in injection_sites:
        print(f"        Element {site['element_id']}: "
              f"score={site['score']:.3f}, stress={site['wall_stress_kPa']:.1f}kPa, "
              f"type={site['tissue_type']}")
    
    # Save outputs
    print("\n  [5] SAVING OUTPUTS")
    
    out_dir = os.path.join(OUTPUT_DIR, patient_id)
    os.makedirs(out_dir, exist_ok=True)
    
    # Transmural position per element
    phi_elem = np.array([np.mean(phi[elem]) for elem in elements])
    
    # VTK with all fields
    write_vtk_complete(
        os.path.join(out_dir, f"{patient_id}_complete_analysis.vtk"),
        coords, elements,
        scalars={
            'TransmuralPhi': phi,  # Node
            'TransmuralPhi_elem': phi_elem,  # Cell
            'TissueType': tags.astype(float),
            'FiberAngle_deg': fiber_angles,
            'WallThickness_mm': stress_data['wall_thickness'],
            'WallStress_kPa': stress_data['wall_stress'],
            'LocalRadius_mm': stress_data['local_radius'],
            'CurvatureFactor': stress_data['curvature_factor'],
            'Kappa1': stress_data['kappa_1'],
            'Kappa2': stress_data['kappa_2'],
        },
        vectors={
            'FiberDirection': fibers,
            'SheetDirection': sheets,
            'TransmuralDirection': coord_system[0],
        }
    )
    
    # Fiber file (CARP format)
    write_lon_file(os.path.join(out_dir, f"{patient_id}_reconstructed.lon"), fibers, sheets)
    
    # Injection sites
    with open(os.path.join(out_dir, f"{patient_id}_injection_sites.json"), 'w') as f:
        json.dump(injection_sites, f, indent=2)
    
    write_injection_coords(
        os.path.join(out_dir, f"{patient_id}_injection_coords.txt"),
        injection_sites
    )
    
    # Summary
    summary = {
        'patient_id': patient_id,
        'timestamp': datetime.now().isoformat(),
        'methodology': {
            'component_1': 'Laplace-Dirichlet transmural coordinate (∇²φ=0)',
            'component_2': 'Helical fiber reconstruction (α(φ) = 60° - 120°×φ)',
            'component_3': 'Wall stress with curvature tensor (σ = Pr/(2h)×f(κ₁,κ₂))',
            'component_4': 'Laplacian-geodesic injection optimization (Heat Method)'
        },
        'mesh': {
            'n_nodes': int(n_nodes),
            'n_elements': int(n_elems),
        },
        'transmural': {
            'phi_range': [float(phi.min()), float(phi.max())],
        },
        'fibers': {
            'angle_range_deg': [float(fiber_angles.min()), float(fiber_angles.max())],
            'endo_angle': FIBER_ANGLE_ENDO_DEG,
            'epi_angle': FIBER_ANGLE_EPI_DEG,
        },
        'wall_stress': {
            'range_kPa': [float(stress_data['wall_stress'].min()), 
                         float(stress_data['wall_stress'].max())],
            'mean_kPa': float(np.mean(stress_data['wall_stress'])),
            'pressure_kPa': LV_PRESSURE_KPA,
        },
        'wall_thickness': {
            'range_mm': [float(stress_data['wall_thickness'].min()),
                        float(stress_data['wall_thickness'].max())],
            'mean_mm': float(np.mean(stress_data['wall_thickness'])),
        },
        'injection_sites': injection_sites,
    }
    
    with open(os.path.join(out_dir, f"{patient_id}_summary.json"), 'w') as f:
        json.dump(summary, f, indent=2, default=str)
    
    print(f"\n  Outputs saved to: {out_dir}")
    
    return summary


def process_patient_wrapper(patient_id):
    """Wrapper for parallel processing"""
    try:
        result = process_patient_complete(patient_id)
        return result
    except Exception as e:
        import traceback
        traceback.print_exc()
        return {'patient_id': patient_id, 'status': 'FAILED', 'error': str(e)}


def main():
    """Main entry point with parallel processing"""
    print("COMPLETE LAPLACE-DIRICHLET CARDIAC ANALYSIS FRAMEWORK")
    
    print(f"\n  Using {N_CPUS} CPUs for parallel processing")
    
    print("\nIMPLEMENTED COMPONENTS (from abstract):")
    print("  [1] Laplace-Dirichlet transmural coordinate: ∇²φ = 0")
    print("  [2] Helical fiber reconstruction: α(φ) = α_endo + (α_epi - α_endo)×φ")
    print("  [3] Wall stress with curvature tensor: σ = Pr/(2h) × f(κ₁,κ₂)")
    print("  [4] Laplacian-geodesic injection optimization: Heat Method + stress")
    
    print("\n  Loading tissue tags from: infarct_coronary_territory/")
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # Set environment variables for multi-threaded linear algebra
    import os as os_env
    threads_per_patient = max(1, N_CPUS // len(PATIENT_IDS))
    os_env.environ['OMP_NUM_THREADS'] = str(threads_per_patient)
    os_env.environ['MKL_NUM_THREADS'] = str(threads_per_patient)
    os_env.environ['OPENBLAS_NUM_THREADS'] = str(threads_per_patient)
    os_env.environ['NUMEXPR_NUM_THREADS'] = str(threads_per_patient)
    
    print(f"  Processing {len(PATIENT_IDS)} patients in parallel")
    print(f"  Threads per patient: {threads_per_patient}")
    
    # Process patients in parallel
    n_workers = min(N_CPUS, len(PATIENT_IDS))
    
    with Pool(processes=n_workers) as pool:
        all_results = pool.map(process_patient_wrapper, PATIENT_IDS)
    
    # Summary table
    print("SUMMARY")
    
    print(f"\n{'Patient':<15} {'φ range':<15} {'Fiber°':<15} {'Stress kPa':<15} {'Sites':<10}")
    
    for r in all_results:
        if 'status' not in r:
            phi_r = r.get('transmural', {}).get('phi_range', [0, 1])
            fib_r = r.get('fibers', {}).get('angle_range_deg', [-60, 60])
            stress_r = r.get('wall_stress', {}).get('range_kPa', [0, 0])
            sites = len(r.get('injection_sites', []))
            print(f"{r['patient_id']:<15} "
                  f"[{phi_r[0]:.2f}, {phi_r[1]:.2f}]     "
                  f"[{fib_r[0]:.0f}°, {fib_r[1]:.0f}°]    "
                  f"[{stress_r[0]:.1f}, {stress_r[1]:.1f}]    "
                  f"{sites}")
        else:
            print(f"{r['patient_id']:<15} FAILED: {r.get('error', 'Unknown')[:40]}")
    
    # Save combined
    with open(os.path.join(OUTPUT_DIR, "complete_analysis_summary.json"), 'w') as f:
        json.dump(all_results, f, indent=2, default=str)
    
    print(f"\nResults saved to: {OUTPUT_DIR}")
    
    return all_results


if __name__ == "__main__":
    results = main()


def main_notebook():
    """
    Alternative main function optimized for Jupyter notebooks.
    Uses joblib instead of multiprocessing Pool (more compatible with notebooks).
    """
    print("COMPLETE LAPLACE-DIRICHLET CARDIAC ANALYSIS FRAMEWORK")
    
    print(f"\n  Using {N_CPUS} CPUs for parallel processing (notebook mode)")
    
    print("\nIMPLEMENTED COMPONENTS (from abstract):")
    print("  [1] Laplace-Dirichlet transmural coordinate: ∇²φ = 0")
    print("  [2] Helical fiber reconstruction: α(φ) = α_endo + (α_epi - α_endo)×φ")
    print("  [3] Wall stress with curvature tensor: σ = Pr/(2h) × f(κ₁,κ₂)")
    print("  [4] Laplacian-geodesic injection optimization: Heat Method + stress")
    
    print("\n  Loading tissue tags from: infarct_coronary_territory/")
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # Set environment variables for multi-threaded linear algebra
    os.environ['OMP_NUM_THREADS'] = str(N_CPUS)
    os.environ['MKL_NUM_THREADS'] = str(N_CPUS)
    os.environ['OPENBLAS_NUM_THREADS'] = str(N_CPUS)
    os.environ['NUMEXPR_NUM_THREADS'] = str(N_CPUS)
    
    print(f"  Processing {len(PATIENT_IDS)} patients")
    
    if HAS_JOBLIB:
        # Use joblib for parallel processing (notebook-friendly)
        all_results = Parallel(n_jobs=min(N_CPUS, len(PATIENT_IDS)), verbose=10)(
            delayed(process_patient_wrapper)(pid) for pid in PATIENT_IDS
        )
    else:
        # Sequential fallback
        all_results = []
        for pid in PATIENT_IDS:
            result = process_patient_wrapper(pid)
            all_results.append(result)
    
    # Summary table
    print("SUMMARY")
    
    print(f"\n{'Patient':<15} {'φ range':<15} {'Fiber°':<15} {'Stress kPa':<15} {'Sites':<10}")
    
    for r in all_results:
        if 'status' not in r:
            phi_r = r.get('transmural', {}).get('phi_range', [0, 1])
            fib_r = r.get('fibers', {}).get('angle_range_deg', [-60, 60])
            stress_r = r.get('wall_stress', {}).get('range_kPa', [0, 0])
            sites = len(r.get('injection_sites', []))
            print(f"{r['patient_id']:<15} "
                  f"[{phi_r[0]:.2f}, {phi_r[1]:.2f}]     "
                  f"[{fib_r[0]:.0f}°, {fib_r[1]:.0f}°]    "
                  f"[{stress_r[0]:.1f}, {stress_r[1]:.1f}]    "
                  f"{sites}")
        else:
            print(f"{r['patient_id']:<15} FAILED: {r.get('error', 'Unknown')[:40]}")
    
    # Save combined
    with open(os.path.join(OUTPUT_DIR, "complete_analysis_summary.json"), 'w') as f:
        json.dump(all_results, f, indent=2, default=str)
    
    print(f"\nResults saved to: {OUTPUT_DIR}")
    
    return all_results


# For notebook usage:
# from laplace_dirichlet_complete import main_notebook
# results = main_notebook()

Note: joblib not available, using sequential processing for loops
COMPLETE LAPLACE-DIRICHLET CARDIAC ANALYSIS FRAMEWORK

  Using 44 CPUs for parallel processing

IMPLEMENTED COMPONENTS (from abstract):
  [1] Laplace-Dirichlet transmural coordinate: ∇²φ = 0
  [2] Helical fiber reconstruction: α(φ) = α_endo + (α_epi - α_endo)×φ
  [3] Wall stress with curvature tensor: σ = Pr/(2h) × f(κ₁,κ₂)
  [4] Laplacian-geodesic injection optimization: Heat Method + stress

  Loading tissue tags from: infarct_coronary_territory/
  Processing 10 patients in parallel...
  Threads per patient: 4
PROCESSING: SCD0000301PROCESSING: SCD0000101PROCESSING: SCD0000401PROCESSING: SCD0000601PROCESSING: SCD0000701PROCESSING: SCD0000801PROCESSING: SCD0001201PROCESSING: SCD0001101PROCESSING: SCD0001001
PROCESSING: SCD0000201









  Loading mesh...
  Loading mesh...
  Loading mesh...
  Loading mesh...
  Loading mesh...
  Loading mesh...
  Loading mesh...
  Loading mesh...
  Loading mesh...
  Loading mesh...







In [6]:
#!/usr/bin/env python3
"""
 INFARCT DETECTION FRAMEWORK

MULTI-METRIC INFARCT CLASSIFICATION

This algorithm integrates FIVE complementary methodologies to accurately identify
infarct, border zone, and healthy myocardial tissue without LGE-MRI:

METHODOLOGY OVERVIEW:

1. LAPLACE-DIRICHLET TRANSMURAL COORDINATE (φ)
   - Solve ∇²φ = 0 with φ=0 (endo), φ=1 (epi)
   - Provides smooth transmural position field
   - Reference: Bishop et al. Am J Physiol 2010

2. ROBUST WALL THICKNESS (h) FROM GRADIENT
   - h(x) = 1/|∇φ(x)| with robust outlier filtering
   - Literature: Infarcted wall = 2.86±1.11mm vs healthy = 8.73±1.01mm
   - Reference: Assessing regional LV thickening (J Magn Reson Imaging 2018)

3. WALL STRESS (σ) VIA MODIFIED LAW OF LAPLACE
   - σ = P×r/(2h) × κ(x) where κ is curvature correction
   - Stress concentrations at infarct borders
   - Reference: Zhong et al. Int J Cardiol 2008

4. LOCAL FIBER COHERENCE (LFC)
   - LFC = |mean(fiber vectors)| in local neighborhood
   - Low coherence indicates tissue disorganization
   - Reference: Mekkaoui et al. J Cardiovasc Magn Reson 2012

5. TRANSMURAL EXTENT & SUBENDOCARDIAL PREFERENCE
   - Ischemic infarcts originate subendocardially
   - Transmurality affects classification confidence
   - Reference: LGE patterns (MDPI J Cardiovasc Dev Dis 2024)

CLASSIFICATION CRITERIA:

INFARCT (Dense Scar):
- Wall thickness < 4.0mm (absolute) OR < μ - 2σ (relative)
- Combined score > 85th percentile
- Anatomically localized (angular spread < 150°)
- Preferentially subendocardial

BORDER ZONE (Peri-infarct):
- Wall thickness 4.0-6.0mm OR μ-2σ to μ-1σ
- Adjacent to infarct core
- Intermediate fiber coherence
- Stress concentration region

HEALTHY:
- Wall thickness > 6.0mm AND > μ - 1σ
- Normal fiber architecture (LFC > 0.85)
- No stress concentration

LITERATURE BASIS:
- Wall thickness ≤5mm = transmural scar: Penicka et al. (92% sens, 96% spec)
- Infarcted WT = 2.86mm, Healthy WT = 8.73mm: J Magn Reson Imaging 2018
- WT < 3mm shows EP changes: JACC Clin Electrophysiol 2023
- 5SD threshold robust for LGE: Circ Cardiovasc Imaging 2015
- Expected HF-I scar: 8-15%: Puntmann et al. JACC 2016

"""

import numpy as np
from scipy.sparse import lil_matrix, csr_matrix
from scipy.sparse.linalg import spsolve
from scipy.spatial import cKDTree
from scipy.stats import zscore
from scipy.ndimage import gaussian_filter1d
from collections import defaultdict
import os
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')


PATIENT_IDS = [
    "SCD0000101", "SCD0000201", "SCD0000301", "SCD0000401",
    "SCD0000601", "SCD0000701", "SCD0000801", "SCD0001001",
    "SCD0001101", "SCD0001201"
]

BASE_DIR = "/home/shadeform/SCD_MODELS"
OUTPUT_DIR = "/home/shadeform/SCD_MODELS/infarct_results_comprehensive"

class Config:
    """
    Literature-validated parameters for infarct detection.
    
    References:
    [1] Penicka et al. - WT≤5mm: 92% sens, 96% spec for transmural scar
    [2] J Magn Reson Imaging 2018 - Infarcted WT=2.86mm, Healthy WT=8.73mm
    [3] JACC Clin EP 2023 - WT<3mm shows significant EP changes
    [4] Puntmann et al. JACC 2016 - HF-I scar burden 8-15%
    [5] Circ Cardiovasc Imaging - 5SD threshold robust for LGE
    """
    
    # WALL THICKNESS THRESHOLDS (mm) - from literature
    # Dense scar: <4mm (between 2.86mm infarcted and clearly abnormal)
    WT_DENSE_SCAR_MM = 4.0
    
    # Scar threshold: ≤5.5mm (Penicka: ≤5mm, with margin)
    WT_SCAR_THRESHOLD_MM = 5.5
    
    # Border zone: 5.5-7.0mm (transition region)
    WT_BORDER_THRESHOLD_MM = 7.0
    
    # Normal minimum: >7mm (healthy = 8.73±1.01mm)
    WT_HEALTHY_MIN_MM = 7.0
    
    # RELATIVE THRESHOLDS (z-scores)
    # Based on 5SD being robust for LGE (but we use negative z for thin walls)
    WT_INFARCT_ZSCORE = 2.0   # WT < μ - 2σ
    WT_BORDER_ZSCORE = 1.0    # WT < μ - 1σ
    
    # FIBER COHERENCE THRESHOLDS
    LFC_DENSE_SCAR = 0.65     # Very disorganized
    LFC_SCAR_THRESHOLD = 0.75 # Disorganized
    LFC_BORDER_THRESHOLD = 0.85 # Mildly disorganized
    
    # WALL STRESS (for identifying high-risk regions)
    LV_PRESSURE_KPA = 16.0    # Peak systolic pressure
    STRESS_CONCENTRATION_FACTOR = 1.5  # Border zone stress multiplier
    
    # COMBINED SCORE WEIGHTS
    # Primary: Wall thickness (from Laplace, most validated)
    WEIGHT_WT = 0.45
    
    # Secondary: Fiber coherence (from LDRB)
    WEIGHT_LFC = 0.20
    
    # Tertiary: Wall stress (from modified Laplace law)
    WEIGHT_STRESS = 0.15
    
    # Quaternary: Subendocardial preference (ischemic pattern)
    WEIGHT_SUBENDO = 0.10
    
    # Quinary: Transmural thinning ratio
    WEIGHT_THINNING = 0.10
    
    # ANATOMICAL CONSTRAINTS
    MAX_ANGULAR_SPREAD_DEG = 150  # Single coronary territory
    MIN_CONTIGUITY_SIZE = 100     # Minimum connected component
    
    # EXPECTED SCAR BURDEN (for validation)
    MIN_INFARCT_PCT = 3.0    # Below this → likely healthy or detection issue
    MAX_INFARCT_PCT = 20.0   # Above this → likely algorithm issue
    TARGET_INFARCT_PCT = 10.0  # Expected mean for HF-I
    
    # BORDER ZONE
    BORDER_LAYERS = 4
    
    # ROBUST STATISTICS
    WT_PERCENTILE_LOW = 2     # Clip below this percentile
    WT_PERCENTILE_HIGH = 98   # Clip above this percentile

# MESH LOADING

def load_pts(filepath):
    """Load node coordinates"""
    with open(filepath, 'r') as f:
        n = int(f.readline().strip())
        coords = np.zeros((n, 3), dtype=np.float64)
        for i in range(n):
            coords[i] = [float(x) for x in f.readline().split()[:3]]
    return coords

def load_elem(filepath):
    """Load tetrahedral elements"""
    with open(filepath, 'r') as f:
        n = int(f.readline().strip())
        elements = np.zeros((n, 4), dtype=np.int32)
        for i in range(n):
            line = f.readline().split()
            elements[i] = [int(x) for x in line[1:5]]
    return elements

def load_lon(filepath):
    """Load fiber orientations"""
    with open(filepath, 'r') as f:
        _ = f.readline()
        lines = f.readlines()
        fibers = np.zeros((len(lines), 3), dtype=np.float64)
        for i, line in enumerate(lines):
            fibers[i] = [float(x) for x in line.split()[:3]]
    norms = np.linalg.norm(fibers, axis=1, keepdims=True)
    norms[norms == 0] = 1
    return fibers / norms

def find_mesh_files(patient_id, base_dir):
    """Find mesh files for a patient"""
    pts = f"{base_dir}/simulation_ready/{patient_id}/{patient_id}_tet.pts"
    elem = f"{base_dir}/simulation_ready/{patient_id}/{patient_id}_tet.elem"
    lon = f"{base_dir}/fibers/{patient_id}/{patient_id}.lon"
    
    if os.path.exists(pts) and os.path.exists(elem) and os.path.exists(lon):
        return pts, elem, lon
    raise FileNotFoundError(f"Could not find mesh files for {patient_id}")

# METHODOLOGY 1: LAPLACE-DIRICHLET TRANSMURAL COORDINATE

def extract_surfaces_robust(coords, elements):
    """
    Extract endocardial and epicardial surfaces using robust classification.
    
    Algorithm:
    1. Find boundary faces (faces belonging to single element)
    2. For each z-slice, classify by radial position relative to centroid
    3. Inner surface = endocardium, outer surface = epicardium
    """
    n_elems = len(elements)
    
    # Step 1: Find boundary faces
    face_count = defaultdict(list)
    for elem_idx, elem in enumerate(elements):
        faces = [
            tuple(sorted([elem[0], elem[1], elem[2]])),
            tuple(sorted([elem[0], elem[1], elem[3]])),
            tuple(sorted([elem[0], elem[2], elem[3]])),
            tuple(sorted([elem[1], elem[2], elem[3]])),
        ]
        for face in faces:
            face_count[face].append(elem_idx)
    
    # Boundary faces belong to only one element
    boundary_faces = [f for f, elems in face_count.items() if len(elems) == 1]
    
    # Get all boundary nodes
    boundary_nodes = set()
    for face in boundary_faces:
        boundary_nodes.update(face)
    boundary_nodes = np.array(list(boundary_nodes))
    
    # Step 2: Classify by radial position at each z-level
    z_vals = coords[:, 2]
    z_min, z_max = z_vals.min(), z_vals.max()
    z_range = z_max - z_min
    
    endo_nodes = set()
    epi_nodes = set()
    
    n_slices = 30  # More slices for better resolution
    for i in range(n_slices):
        z_lo = z_min + i * z_range / n_slices
        z_hi = z_min + (i + 1) * z_range / n_slices
        
        # Get boundary nodes in this slice
        slice_mask = (coords[boundary_nodes, 2] >= z_lo) & (coords[boundary_nodes, 2] < z_hi)
        slice_nodes = boundary_nodes[slice_mask]
        
        if len(slice_nodes) < 20:
            continue
        
        # Compute center and radii
        slice_coords = coords[slice_nodes, :2]  # XY only
        center = np.median(slice_coords, axis=0)  # Use median for robustness
        radii = np.linalg.norm(slice_coords - center, axis=1)
        
        # Use percentiles for robust classification
        r_25 = np.percentile(radii, 25)
        r_75 = np.percentile(radii, 75)
        
        for j, node in enumerate(slice_nodes):
            if radii[j] < r_25:
                endo_nodes.add(node)
            elif radii[j] > r_75:
                epi_nodes.add(node)
    
    return {
        'endo_nodes': np.array(list(endo_nodes)),
        'epi_nodes': np.array(list(epi_nodes)),
        'boundary_faces': boundary_faces
    }


def build_laplacian_matrix(coords, elements):
    """
    Build Laplacian stiffness matrix for FEM solution.
    
    For linear tetrahedral elements:
    K_ij = ∫_Ω ∇N_i · ∇N_j dV
    
    Where N_i are linear shape functions.
    """
    n_nodes = len(coords)
    K = lil_matrix((n_nodes, n_nodes))
    
    for elem in elements:
        X = coords[elem]
        
        # Jacobian: maps reference tet to physical tet
        J = np.array([
            X[1] - X[0],
            X[2] - X[0],
            X[3] - X[0]
        ]).T
        
        detJ = np.linalg.det(J)
        if abs(detJ) < 1e-15:
            continue
        
        vol = abs(detJ) / 6.0
        Jinv = np.linalg.inv(J)
        
        # Shape function gradients in physical space
        # For linear tet: N0 = 1-ξ-η-ζ, N1 = ξ, N2 = η, N3 = ζ
        dN = np.zeros((4, 3))
        dN[0] = -Jinv.sum(axis=1)  # ∇N0
        dN[1] = Jinv[:, 0]         # ∇N1
        dN[2] = Jinv[:, 1]         # ∇N2
        dN[3] = Jinv[:, 2]         # ∇N3
        
        # Element stiffness: K_e = V × (∇N ⊗ ∇N)
        Ke = vol * (dN @ dN.T)
        
        # Assemble into global matrix
        for i in range(4):
            for j in range(4):
                K[elem[i], elem[j]] += Ke[i, j]
    
    return K.tocsr()


def solve_laplace_dirichlet(coords, elements, surfaces):
    """
    Solve Laplace equation with Dirichlet boundary conditions.
    
    Mathematical formulation:
    ∇²φ = 0       in Ω (myocardial domain)
    φ = 0         on Γ_endo (endocardial surface)
    φ = 1         on Γ_epi (epicardial surface)
    
    Returns: φ(x) ∈ [0,1] transmural coordinate
    """
    n_nodes = len(coords)
    
    print("      Building Laplacian matrix")
    K = build_laplacian_matrix(coords, elements)
    
    endo_nodes = set(surfaces['endo_nodes'])
    epi_nodes = set(surfaces['epi_nodes'])
    
    print(f"      Dirichlet BCs: {len(endo_nodes)} endo, {len(epi_nodes)} epi nodes")
    
    # Penalty method for Dirichlet BCs
    K_mod = K.tolil()
    rhs = np.zeros(n_nodes)
    penalty = 1e12
    
    for node in endo_nodes:
        K_mod[node, :] = 0
        K_mod[node, node] = penalty
        rhs[node] = 0.0 * penalty  # φ = 0 on endocardium
    
    for node in epi_nodes:
        K_mod[node, :] = 0
        K_mod[node, node] = penalty
        rhs[node] = 1.0 * penalty  # φ = 1 on epicardium
    
    K_mod = K_mod.tocsr()
    
    print("      Solving Laplace equation (FEM)")
    phi = spsolve(K_mod, rhs)
    phi = np.clip(phi, 0, 1)
    
    print(f"      Transmural φ: [{phi.min():.4f}, {phi.max():.4f}]")
    
    return phi

# METHODOLOGY 2: ROBUST WALL THICKNESS FROM LAPLACE GRADIENT

def compute_wall_thickness_robust(coords, elements, phi):
    """
    Compute wall thickness from transmural coordinate gradient with robust filtering.
    
    Mathematical formulation:
    h(x) = 1 / |∇φ(x)|
    
    Physical interpretation:
    - Thin wall → φ changes rapidly → large |∇φ| → small h
    - Thick wall → φ changes slowly → small |∇φ| → large h
    
    Robust filtering:
    - Clip extreme gradients (singularities near surfaces)
    - Use percentile-based bounds
    - Smooth using local averaging
    """
    n_elems = len(elements)
    
    # Compute gradient at each element
    grad_phi = np.zeros((n_elems, 3))
    raw_wt = np.zeros(n_elems)
    
    for i, elem in enumerate(elements):
        X = coords[elem]
        phi_elem = phi[elem]
        
        # Jacobian
        J = np.array([
            X[1] - X[0],
            X[2] - X[0],
            X[3] - X[0]
        ]).T
        
        detJ = np.linalg.det(J)
        if abs(detJ) < 1e-15:
            raw_wt[i] = np.nan
            continue
        
        Jinv = np.linalg.inv(J)
        
        # Gradient in physical space: ∇φ = J^(-T) ∇_ξ φ
        dphi_dxi = np.array([
            phi_elem[1] - phi_elem[0],
            phi_elem[2] - phi_elem[0],
            phi_elem[3] - phi_elem[0]
        ])
        
        grad_phi[i] = Jinv @ dphi_dxi
        grad_norm = np.linalg.norm(grad_phi[i])
        
        # Wall thickness = 1/|∇φ|
        if grad_norm > 1e-8:
            raw_wt[i] = 1.0 / grad_norm
        else:
            raw_wt[i] = np.nan  # Will be filtered
    
    # ROBUST FILTERING
    
    # Step 1: Replace NaN with median
    valid_mask = ~np.isnan(raw_wt)
    median_wt = np.median(raw_wt[valid_mask])
    raw_wt[~valid_mask] = median_wt
    
    # Step 2: Compute percentile bounds
    p_low = np.percentile(raw_wt, Config.WT_PERCENTILE_LOW)
    p_high = np.percentile(raw_wt, Config.WT_PERCENTILE_HIGH)
    
    # Step 3: Clip to percentile bounds
    wt_clipped = np.clip(raw_wt, p_low, p_high)
    
    # Step 4: Calibrate to physiological scale
    # Target: median WT should be ~10mm (normal LV)
    wt_median = np.median(wt_clipped)
    scale_factor = 10.0 / wt_median
    
    wt_mm = wt_clipped * scale_factor
    
    # Step 5: Final clipping to physiological bounds
    wt_mm = np.clip(wt_mm, 0.5, 25.0)  # 0.5-25mm physiological range
    
    return wt_mm, grad_phi, scale_factor


def compute_transmural_metrics(coords, elements, phi, surfaces, scale_factor):
    """
    Compute transmural position and distance to endocardium.
    
    Returns:
    - phi_elem: Mean φ per element (transmural position)
    - endo_dist_mm: Distance to endocardial surface (mm)
    - transmural_depth: Relative depth (0=endo, 1=epi)
    """
    n_elems = len(elements)
    centroids = np.mean(coords[elements], axis=1) * scale_factor
    
    # Mean phi per element
    phi_elem = np.array([np.mean(phi[elem]) for elem in elements])
    
    # Distance to endocardium
    endo_coords_mm = coords[surfaces['endo_nodes']] * scale_factor
    endo_tree = cKDTree(endo_coords_mm)
    
    endo_dist_mm = np.zeros(n_elems)
    for i, centroid in enumerate(centroids):
        d, _ = endo_tree.query(centroid)
        endo_dist_mm[i] = d
    
    return phi_elem, endo_dist_mm, centroids

# METHODOLOGY 3: WALL STRESS (MODIFIED LAW OF LAPLACE)

def compute_wall_stress(coords_mm, elements, wt_mm, phi_elem, surfaces, scale_factor):
    """
    Compute wall stress using modified Law of Laplace.
    
    Mathematical formulation:
    σ = P × r_local / (2 × h) × κ(x)
    
    Where:
    - P = cavity pressure (kPa)
    - r_local = local radius of curvature (mm)
    - h = wall thickness (mm)
    - κ(x) = curvature correction factor
    
    Stress is elevated at:
    - Thin regions (low h)
    - High curvature regions
    - Infarct borders
    """
    n_elems = len(elements)
    centroids = np.mean(coords_mm[elements], axis=1)
    
    # Estimate local radius from radial position at each z-level
    z_vals = centroids[:, 2]
    local_radius = np.zeros(n_elems)
    
    # Group by z-level
    n_slices = 20
    z_min, z_max = z_vals.min(), z_vals.max()
    z_range = z_max - z_min
    
    for i in range(n_slices):
        z_lo = z_min + i * z_range / n_slices
        z_hi = z_min + (i + 1) * z_range / n_slices
        
        mask = (z_vals >= z_lo) & (z_vals < z_hi)
        if np.sum(mask) < 10:
            continue
        
        # Center at this level
        level_centroids = centroids[mask, :2]
        center = np.median(level_centroids, axis=0)
        
        # Radii
        radii = np.linalg.norm(level_centroids - center, axis=1)
        mean_radius = np.mean(radii)
        
        # Assign to elements
        local_radius[mask] = mean_radius
    
    # Fill any zeros with global mean
    local_radius[local_radius == 0] = np.mean(local_radius[local_radius > 0])
    
    # Curvature correction (simplified: higher for thin walls)
    curvature_factor = 1.0 + 0.5 * (Config.WT_HEALTHY_MIN_MM - np.clip(wt_mm, 2, 10)) / 5.0
    curvature_factor = np.clip(curvature_factor, 1.0, 2.0)
    
    # Wall stress: σ = P × r / (2h) × κ
    wall_stress_kpa = (Config.LV_PRESSURE_KPA * local_radius) / (2 * wt_mm) * curvature_factor
    
    # Normalize stress (relative to median)
    stress_normalized = wall_stress_kpa / np.median(wall_stress_kpa)
    
    return wall_stress_kpa, stress_normalized, local_radius, curvature_factor

# METHODOLOGY 4: FIBER COHERENCE

def build_adjacency(elements):
    """Build element adjacency graph via shared faces"""
    face_to_elem = defaultdict(list)
    for i, nodes in enumerate(elements):
        for face in [
            tuple(sorted([nodes[0], nodes[1], nodes[2]])),
            tuple(sorted([nodes[0], nodes[1], nodes[3]])),
            tuple(sorted([nodes[0], nodes[2], nodes[3]])),
            tuple(sorted([nodes[1], nodes[2], nodes[3]]))
        ]:
            face_to_elem[face].append(i)
    
    adjacency = defaultdict(list)
    for face, elems in face_to_elem.items():
        if len(elems) == 2:
            adjacency[elems[0]].append(elems[1])
            adjacency[elems[1]].append(elems[0])
    return adjacency


def compute_fiber_coherence(fibers, elements, adjacency):
    """
    Compute Local Fiber Coherence (LFC).
    
    Mathematical formulation:
    LFC(i) = |mean(f_j for j ∈ N(i))| / n
    
    Where:
    - f_j = fiber vector at element j
    - N(i) = neighborhood of element i (including i)
    - Vectors are aligned before averaging (sign correction)
    
    Range: [0, 1]
    - LFC ≈ 1: Highly aligned fibers (healthy)
    - LFC < 0.7: Disorganized (scar/border)
    """
    n_elems = len(elements)
    
    # Get element-level fibers
    if len(fibers) == n_elems:
        elem_fibers = fibers.copy()
    else:
        # Node-based → element average
        elem_fibers = np.zeros((n_elems, 3))
        for i, elem in enumerate(elements):
            node_fibers = fibers[elem].copy()
            # Align to first fiber
            ref = node_fibers[0]
            for j in range(1, 4):
                if np.dot(node_fibers[j], ref) < 0:
                    node_fibers[j] = -node_fibers[j]
            avg = np.mean(node_fibers, axis=0)
            norm = np.linalg.norm(avg)
            elem_fibers[i] = avg / norm if norm > 0 else ref
    
    # Compute LFC
    lfc = np.ones(n_elems)
    for i in range(n_elems):
        neighbors = adjacency.get(i, [])
        if not neighbors:
            continue
        
        # Collect fibers in neighborhood
        all_fibers = [elem_fibers[i]]
        for j in neighbors:
            fib = elem_fibers[j].copy()
            # Align to reference
            if np.dot(fib, all_fibers[0]) < 0:
                fib = -fib
            all_fibers.append(fib)
        
        # Mean resultant length
        mean_vec = np.mean(all_fibers, axis=0)
        lfc[i] = np.linalg.norm(mean_vec)
    
    return lfc, elem_fibers

# METHODOLOGY 5: COMPREHENSIVE SCORING

def compute_comprehensive_score(wt_mm, lfc, stress_norm, phi_elem, endo_dist_mm):
    """
    Compute comprehensive infarct likelihood score combining all metrics.
    
    Score = Σ w_i × S_i
    
    Where:
    - S_WT: Wall thickness score (lower WT = higher score)
    - S_LFC: Fiber coherence score (lower LFC = higher score)
    - S_stress: Wall stress score (higher stress = higher score)
    - S_subendo: Subendocardial preference (lower φ = higher score)
    - S_thinning: Thinning ratio score
    
    All scores normalized to [0, 1].
    """
    n_elems = len(wt_mm)
    
    # S_WT: Wall Thickness Score
    # Linear scaling: WT=2mm → 1.0, WT=8mm → 0.0
    wt_score = np.clip((Config.WT_HEALTHY_MIN_MM - wt_mm) / 
                       (Config.WT_HEALTHY_MIN_MM - 2.0), 0, 1)
    
    # Boost for very thin regions (WT < 4mm = dense scar threshold)
    very_thin = wt_mm < Config.WT_DENSE_SCAR_MM
    wt_score[very_thin] = np.clip(wt_score[very_thin] * 1.3, 0, 1)
    
    # S_LFC: Fiber Coherence Score
    # Linear scaling: LFC=0.5 → 1.0, LFC=1.0 → 0.0
    lfc_score = np.clip((1.0 - lfc) / 0.5, 0, 1)
    
    # Boost for very low coherence (LFC < 0.65 = dense scar)
    very_low = lfc < Config.LFC_DENSE_SCAR
    lfc_score[very_low] = np.clip(lfc_score[very_low] * 1.2, 0, 1)
    
    # S_stress: Wall Stress Score
    # Higher stress = higher score (identifies border zones)
    stress_score = np.clip((stress_norm - 1.0) / 1.0, 0, 1)
    
    # S_subendo: Subendocardial Preference
    # Ischemic infarcts are subendocardial: lower φ = higher score
    subendo_score = np.clip(1.0 - phi_elem * 1.5, 0, 1)
    
    # S_thinning: Thinning Ratio
    # Ratio of endo distance to expected (healthy WT ~10mm)
    # Thin subendo = high score
    expected_endo_dist = phi_elem * 5.0  # Expected distance if WT=10mm
    actual_endo_dist = np.clip(endo_dist_mm, 0.5, 10)
    thinning_ratio = np.clip((expected_endo_dist - actual_endo_dist + 2) / 4, 0, 1)
    
    # COMBINED SCORE
    combined_score = (
        Config.WEIGHT_WT * wt_score +
        Config.WEIGHT_LFC * lfc_score +
        Config.WEIGHT_STRESS * stress_score +
        Config.WEIGHT_SUBENDO * subendo_score +
        Config.WEIGHT_THINNING * thinning_ratio
    )
    
    return combined_score, {
        'wt_score': wt_score,
        'lfc_score': lfc_score,
        'stress_score': stress_score,
        'subendo_score': subendo_score,
        'thinning_score': thinning_ratio
    }

# INFARCT DETECTION WITH ANATOMICAL CONSTRAINTS

def compute_cylindrical_coords(centroids_mm):
    """Compute cylindrical coordinates for anatomical constraints"""
    center = np.mean(centroids_mm, axis=0)
    relative = centroids_mm - center
    
    # Principal component for long axis
    cov = np.cov(relative.T)
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    idx = np.argsort(eigenvalues)[::-1]
    long_axis = eigenvectors[:, idx[0]]
    
    # Z projection (along long axis)
    z = relative @ long_axis
    
    # Radial component
    radial = relative - np.outer(z, long_axis)
    r = np.linalg.norm(radial, axis=1)
    
    # Angular position
    ref_axis = eigenvectors[:, idx[1]]
    perp_axis = np.cross(long_axis, ref_axis)
    theta = np.arctan2(radial @ perp_axis, radial @ ref_axis)
    
    return z, theta, r


def detect_infarct_comprehensive(score, wt_mm, lfc, adjacency, centroids_mm, n_elems):
    """
    Detect infarct using comprehensive criteria with anatomical constraints.
    
    Algorithm:
    1. Identify candidates using BOTH absolute AND relative thresholds
    2. Cluster by angular position (coronary territory)
    3. Grow from best seed region with constraints
    4. Validate geometry
    """
    # Compute cylindrical coordinates
    z, theta, r = compute_cylindrical_coords(centroids_mm)
    
    # Apex/base exclusion (unreliable gradients)
    z_range = z.max() - z.min()
    apex_mask = z < z.min() + 0.08 * z_range
    base_mask = z > z.max() - 0.08 * z_range
    exclude_mask = apex_mask | base_mask
    
    # STEP 1: Identify candidates using multiple criteria
    
    # Criterion A: Absolute WT threshold (literature-based)
    wt_absolute = wt_mm < Config.WT_SCAR_THRESHOLD_MM
    
    # Criterion B: Relative WT threshold (patient-specific)
    valid_wt = wt_mm[(~exclude_mask) & (wt_mm > 2) & (wt_mm < 20)]
    wt_mean = np.mean(valid_wt)
    wt_std = np.std(valid_wt)
    wt_relative = wt_mm < (wt_mean - Config.WT_INFARCT_ZSCORE * wt_std)
    
    # Criterion C: High combined score
    score_threshold = np.percentile(score[~exclude_mask], 80)  # Top 20%
    high_score = score >= score_threshold
    
    # Criterion D: Low fiber coherence
    low_lfc = lfc < Config.LFC_SCAR_THRESHOLD
    
    # Combined: (Absolute OR Relative) AND (High Score OR Low LFC)
    primary_criterion = wt_absolute | wt_relative
    secondary_criterion = high_score | low_lfc
    
    candidates = primary_criterion & secondary_criterion & (~exclude_mask)
    candidate_idx = np.where(candidates)[0]
    
    print(f"      WT thresholds: absolute<{Config.WT_SCAR_THRESHOLD_MM}mm, relative<{wt_mean - Config.WT_INFARCT_ZSCORE * wt_std:.2f}mm")
    print(f"      Initial candidates: {len(candidate_idx)} ({100*len(candidate_idx)/n_elems:.1f}%)")
    
    if len(candidate_idx) < Config.MIN_CONTIGUITY_SIZE:
        print("      WARNING: Few candidates - relaxing criteria")
        # Relax to just absolute OR relative
        candidates = primary_criterion & (~exclude_mask)
        candidate_idx = np.where(candidates)[0]
    
    # STEP 2: Cluster by angular position
    if len(candidate_idx) < 50:
        print("      WARNING: Very few candidates - may indicate healthy tissue")
        # Still proceed with what we have
    
    candidate_theta = theta[candidate_idx]
    candidate_scores = score[candidate_idx]
    
    # Find angular region with highest score density
    n_bins = 12  # 30° bins
    bin_edges = np.linspace(-np.pi, np.pi, n_bins + 1)
    
    best_region_value = 0
    best_center_bin = 0
    
    for i in range(n_bins):
        # Consider this bin and two adjacent bins (90° total)
        in_region = np.zeros(len(candidate_idx), dtype=bool)
        for offset in [-1, 0, 1]:
            b = (i + offset) % n_bins
            in_bin = (candidate_theta >= bin_edges[b]) & (candidate_theta < bin_edges[b + 1])
            in_region |= in_bin
        
        # Handle wraparound
        if i == 0:
            in_region |= (candidate_theta >= bin_edges[-2])
        if i == n_bins - 1:
            in_region |= (candidate_theta < bin_edges[1])
        
        region_idx = candidate_idx[in_region]
        if len(region_idx) > 20:
            # Score = mean score × log(count)
            region_value = np.mean(score[region_idx]) * np.log1p(len(region_idx))
            if region_value > best_region_value:
                best_region_value = region_value
                best_center_bin = i
    
    # Select candidates in best angular region
    theta_center = (bin_edges[best_center_bin] + bin_edges[best_center_bin + 1]) / 2
    max_angular_spread = np.radians(Config.MAX_ANGULAR_SPREAD_DEG)
    
    angular_dist = np.abs(candidate_theta - theta_center)
    angular_dist = np.minimum(angular_dist, 2*np.pi - angular_dist)
    in_best_region = angular_dist <= max_angular_spread / 2
    
    seed_idx = candidate_idx[in_best_region]
    print(f"      Seed region: {len(seed_idx)} elements at θ={np.degrees(theta_center):.1f}°")
    
    # STEP 3: Initialize and grow infarct
    infarct_mask = np.zeros(n_elems, dtype=bool)
    infarct_mask[seed_idx] = True
    
    # Growth thresholds (relaxed from initial)
    growth_wt_threshold = wt_mean - Config.WT_BORDER_ZSCORE * wt_std
    growth_wt_threshold = min(growth_wt_threshold, Config.WT_BORDER_THRESHOLD_MM)
    growth_score_threshold = np.percentile(score[~exclude_mask], 60)
    
    # Iterative growth with constraints
    for iteration in range(100):
        # Find boundary elements
        boundary = set()
        for i in np.where(infarct_mask)[0]:
            for j in adjacency.get(i, []):
                if not infarct_mask[j] and not exclude_mask[j]:
                    boundary.add(j)
        
        if not boundary:
            break
        
        # Add elements meeting criteria
        added = 0
        for elem in boundary:
            # Check WT criterion
            if wt_mm[elem] >= growth_wt_threshold:
                continue
            
            # Check score criterion
            if score[elem] < growth_score_threshold:
                continue
            
            # Check angular constraint
            elem_theta = theta[elem]
            angular_dist = abs(elem_theta - theta_center)
            if angular_dist > np.pi:
                angular_dist = 2*np.pi - angular_dist
            if angular_dist > max_angular_spread / 2:
                continue
            
            infarct_mask[elem] = True
            added += 1
        
        if added == 0:
            break
        
        # Update theta center
        infarct_theta = theta[infarct_mask]
        theta_center = np.arctan2(np.mean(np.sin(infarct_theta)),
                                   np.mean(np.cos(infarct_theta)))
    
    # STEP 4: Ensure minimum size (HF-I should have scar)
    current_pct = 100 * np.sum(infarct_mask) / n_elems
    target_min = Config.MIN_INFARCT_PCT
    
    if current_pct < target_min:
        print(f"      Expanding from {current_pct:.1f}% toward {target_min}%")
        
        # Relax thresholds further
        relaxed_wt = wt_mean
        relaxed_score = np.percentile(score[~exclude_mask], 50)
        
        for iteration in range(50):
            if 100 * np.sum(infarct_mask) / n_elems >= target_min:
                break
            
            boundary = set()
            for i in np.where(infarct_mask)[0]:
                for j in adjacency.get(i, []):
                    if not infarct_mask[j] and not exclude_mask[j]:
                        boundary.add(j)
            
            if not boundary:
                break
            
            # Sort by score and add best candidates
            boundary_list = list(boundary)
            boundary_scores = score[boundary_list]
            order = np.argsort(boundary_scores)[::-1]
            
            for idx in order:
                elem = boundary_list[idx]
                
                # Relaxed criteria
                if wt_mm[elem] >= relaxed_wt:
                    continue
                
                # Angular constraint
                elem_theta = theta[elem]
                angular_dist = abs(elem_theta - theta_center)
                if angular_dist > np.pi:
                    angular_dist = 2*np.pi - angular_dist
                if angular_dist > max_angular_spread / 2:
                    continue
                
                infarct_mask[elem] = True
                
                if 100 * np.sum(infarct_mask) / n_elems >= target_min:
                    break
            
            # Update center
            infarct_theta = theta[infarct_mask]
            theta_center = np.arctan2(np.mean(np.sin(infarct_theta)),
                                       np.mean(np.cos(infarct_theta)))
    
    # STEP 5: Validate geometry
    n_infarct = np.sum(infarct_mask)
    if n_infarct > 0:
        infarct_theta = theta[infarct_mask]
        
        # Circular statistics
        theta_x = np.cos(infarct_theta)
        theta_y = np.sin(infarct_theta)
        R = np.sqrt(np.mean(theta_x)**2 + np.mean(theta_y)**2)
        
        # Angular span
        spread_x = np.std(theta_x)
        spread_y = np.std(theta_y)
        angular_span = np.degrees(2 * np.arcsin(np.clip(np.sqrt(spread_x**2 + spread_y**2), 0, 1)))
        
        validation = {
            'angular_span_deg': angular_span,
            'mean_resultant_length': R,
            'is_localized': R > 0.3,
            'not_circumferential': angular_span < Config.MAX_ANGULAR_SPREAD_DEG
        }
    else:
        validation = {
            'angular_span_deg': 0,
            'mean_resultant_length': 0,
            'is_localized': False,
            'not_circumferential': True
        }
    
    return infarct_mask, theta_center, validation, {
        'z': z,
        'theta': theta,
        'r': r,
        'exclude_mask': exclude_mask,
        'wt_mean': wt_mean,
        'wt_std': wt_std
    }


def create_border_zone(infarct_mask, adjacency, wt_mm, lfc, score, 
                       stats, n_layers=4):
    """
    Create border zone around infarct core.
    
    Border zone characteristics:
    - Surrounds infarct core
    - Has intermediate properties
    - Width based on pathological features
    """
    n_elems = len(infarct_mask)
    border_mask = np.zeros(n_elems, dtype=bool)
    exclude_mask = stats['exclude_mask']
    
    # Thresholds for border zone
    border_wt_threshold = stats['wt_mean'] - Config.WT_BORDER_ZSCORE * stats['wt_std']
    border_wt_threshold = min(border_wt_threshold, Config.WT_BORDER_THRESHOLD_MM)
    
    current_boundary = set(np.where(infarct_mask)[0])
    
    for layer in range(n_layers):
        next_boundary = set()
        
        for elem in current_boundary:
            for neighbor in adjacency.get(elem, []):
                if infarct_mask[neighbor] or border_mask[neighbor] or exclude_mask[neighbor]:
                    continue
                
                # Inner layers: always add
                if layer < 2:
                    next_boundary.add(neighbor)
                else:
                    # Outer layers: require some pathological feature
                    if (wt_mm[neighbor] < border_wt_threshold or 
                        lfc[neighbor] < Config.LFC_BORDER_THRESHOLD or
                        score[neighbor] > np.percentile(score[~exclude_mask], 50)):
                        next_boundary.add(neighbor)
        
        for elem in next_boundary:
            border_mask[elem] = True
        
        current_boundary = next_boundary
        if not current_boundary:
            break
    
    return border_mask

# MAIN PIPELINE

def classify_tissue_comprehensive(coords, elements, fibers, patient_id):
    """
    Main classification pipeline using all five methodologies.
    """
    n_elems = len(elements)
    print(f"\n  Mesh: {len(coords):,} nodes, {n_elems:,} elements")
    
    # METHODOLOGY 1: Laplace-Dirichlet
    print("\n  [1] LAPLACE-DIRICHLET TRANSMURAL COORDINATE")
    print("      Extracting surfaces")
    surfaces = extract_surfaces_robust(coords, elements)
    print(f"      Endo: {len(surfaces['endo_nodes'])} nodes, Epi: {len(surfaces['epi_nodes'])} nodes")
    
    if len(surfaces['endo_nodes']) < 50 or len(surfaces['epi_nodes']) < 50:
        raise ValueError("Could not identify surfaces properly")
    
    phi = solve_laplace_dirichlet(coords, elements, surfaces)
    
    # METHODOLOGY 2: Robust Wall Thickness
    print("\n  [2] ROBUST WALL THICKNESS FROM ∇φ")
    wt_mm, grad_phi, scale_factor = compute_wall_thickness_robust(coords, elements, phi)
    phi_elem, endo_dist_mm, centroids_mm = compute_transmural_metrics(
        coords, elements, phi, surfaces, scale_factor
    )
    coords_mm = coords * scale_factor
    
    print(f"      Scale factor: {scale_factor:.4f}")
    print(f"      WT: mean={np.mean(wt_mm):.2f}mm, median={np.median(wt_mm):.2f}mm")
    print(f"      WT: range=[{np.min(wt_mm):.2f}, {np.max(wt_mm):.2f}]mm")
    print(f"      WT < {Config.WT_SCAR_THRESHOLD_MM}mm: {np.sum(wt_mm < Config.WT_SCAR_THRESHOLD_MM)} ({100*np.mean(wt_mm < Config.WT_SCAR_THRESHOLD_MM):.1f}%)")
    
    # METHODOLOGY 3: Wall Stress
    print("\n  [3] WALL STRESS (MODIFIED LAPLACE LAW)")
    wall_stress, stress_norm, local_radius, curvature_factor = compute_wall_stress(
        coords_mm, elements, wt_mm, phi_elem, surfaces, scale_factor
    )
    print(f"      Stress: mean={np.mean(wall_stress):.2f}kPa, max={np.max(wall_stress):.2f}kPa")
    
    # METHODOLOGY 4: Fiber Coherence
    print("\n  [4] LOCAL FIBER COHERENCE")
    adjacency = build_adjacency(elements)
    lfc, elem_fibers = compute_fiber_coherence(fibers, elements, adjacency)
    print(f"      LFC: mean={np.mean(lfc):.3f}, range=[{np.min(lfc):.3f}, {np.max(lfc):.3f}]")
    print(f"      LFC < {Config.LFC_SCAR_THRESHOLD}: {np.sum(lfc < Config.LFC_SCAR_THRESHOLD)} ({100*np.mean(lfc < Config.LFC_SCAR_THRESHOLD):.1f}%)")
    
    # METHODOLOGY 5: Comprehensive Scoring
    print("\n  [5] COMPREHENSIVE INFARCT SCORE")
    combined_score, score_components = compute_comprehensive_score(
        wt_mm, lfc, stress_norm, phi_elem, endo_dist_mm
    )
    print(f"      Score: mean={np.mean(combined_score):.3f}, max={np.max(combined_score):.3f}")
    
    # INFARCT DETECTION
    print("\n  [6] INFARCT DETECTION (COMPREHENSIVE CRITERIA)")
    infarct_mask, theta_center, validation, stats = detect_infarct_comprehensive(
        combined_score, wt_mm, lfc, adjacency, centroids_mm, n_elems
    )
    
    n_infarct = np.sum(infarct_mask)
    infarct_pct = 100 * n_infarct / n_elems
    print(f"      Infarct: {n_infarct:,} elements ({infarct_pct:.1f}%)")
    print(f"      Angular span: {validation['angular_span_deg']:.1f}°")
    print(f"      Localized: {validation['is_localized']}")
    
    # BORDER ZONE
    print("\n  [7] BORDER ZONE CREATION")
    border_mask = create_border_zone(
        infarct_mask, adjacency, wt_mm, lfc, combined_score, stats, Config.BORDER_LAYERS
    )
    n_border = np.sum(border_mask)
    print(f"      Border: {n_border:,} elements ({100*n_border/n_elems:.1f}%)")
    
    # FINAL CLASSIFICATION
    classification = np.ones(n_elems, dtype=np.int32)  # 1 = healthy
    classification[border_mask] = 2   # 2 = border
    classification[infarct_mask] = 3  # 3 = infarct
    
    n_healthy = np.sum(classification == 1)
    
    print(f"\n  FINAL CLASSIFICATION:")
    print(f"      Healthy: {n_healthy:,} ({100*n_healthy/n_elems:.1f}%)")
    print(f"      Border:  {n_border:,} ({100*n_border/n_elems:.1f}%)")
    print(f"      Infarct: {n_infarct:,} ({infarct_pct:.1f}%)")
    
    # Validation
    if infarct_pct < Config.MIN_INFARCT_PCT:
        print(f"      ⚠ Below expected minimum ({Config.MIN_INFARCT_PCT}%)")
    elif infarct_pct > Config.MAX_INFARCT_PCT:
        print(f"      ⚠ Above expected maximum ({Config.MAX_INFARCT_PCT}%)")
    else:
        print(f"      ✓ Within expected range ({Config.MIN_INFARCT_PCT}-{Config.MAX_INFARCT_PCT}%)")
    
    # Compile results
    results = {
        'classification': classification,
        'coords_mm': coords_mm,
        'centroids_mm': centroids_mm,
        'phi': phi,
        'phi_elem': phi_elem,
        'metrics': {
            'wt_mm': wt_mm,
            'lfc': lfc,
            'wall_stress_kpa': wall_stress,
            'stress_normalized': stress_norm,
            'combined_score': combined_score,
            'theta': stats['theta'],
            'z': stats['z']
        },
        'score_components': score_components,
        'validation': validation,
        'thresholds': {
            'wt_absolute_mm': Config.WT_SCAR_THRESHOLD_MM,
            'wt_relative_mm': stats['wt_mean'] - Config.WT_INFARCT_ZSCORE * stats['wt_std'],
            'wt_mean_mm': stats['wt_mean'],
            'wt_std_mm': stats['wt_std'],
            'lfc_threshold': Config.LFC_SCAR_THRESHOLD,
        },
        'stats': {
            'n_elements': int(n_elems),
            'n_healthy': int(n_healthy),
            'n_border': int(n_border),
            'n_infarct': int(n_infarct),
            'pct_healthy': round(100*n_healthy/n_elems, 2),
            'pct_border': round(100*n_border/n_elems, 2),
            'pct_infarct': round(infarct_pct, 2),
            'scale_factor': round(float(scale_factor), 4),
            'wt_mean_mm': round(float(stats['wt_mean']), 2),
            'wt_std_mm': round(float(stats['wt_std']), 2),
            'lfc_mean': round(float(np.mean(lfc)), 3),
            'angular_span_deg': round(float(validation['angular_span_deg']), 1),
            'mean_resultant_length': round(float(validation['mean_resultant_length']), 3),
        }
    }
    
    return results

# OUTPUT FUNCTIONS

def save_vtk_comprehensive(filepath, coords_mm, elements, classification, metrics, phi):
    """Save VTK with all computed fields"""
    n_nodes, n_elems = len(coords_mm), len(elements)
    
    with open(filepath, 'w') as f:
        f.write("# vtk DataFile Version 3.0\n")
        f.write("Comprehensive Infarct Detection\n")
        f.write("ASCII\n")
        f.write("DATASET UNSTRUCTURED_GRID\n")
        
        f.write(f"POINTS {n_nodes} float\n")
        for c in coords_mm:
            f.write(f"{c[0]:.6f} {c[1]:.6f} {c[2]:.6f}\n")
        
        f.write(f"\nCELLS {n_elems} {n_elems * 5}\n")
        for e in elements:
            f.write(f"4 {e[0]} {e[1]} {e[2]} {e[3]}\n")
        
        f.write(f"\nCELL_TYPES {n_elems}\n")
        f.write("10\n" * n_elems)
        
        # Point data
        f.write(f"\nPOINT_DATA {n_nodes}\n")
        f.write("SCALARS TransmuralPhi float 1\nLOOKUP_TABLE default\n")
        for p in phi:
            f.write(f"{p:.6f}\n")
        
        # Cell data
        f.write(f"\nCELL_DATA {n_elems}\n")
        
        f.write("SCALARS TissueType int 1\nLOOKUP_TABLE default\n")
        for c in classification:
            f.write(f"{c}\n")
        
        for name, values in metrics.items():
            if len(values) == n_elems and name not in ['theta', 'z']:
                f.write(f"\nSCALARS {name} float 1\nLOOKUP_TABLE default\n")
                for v in values:
                    f.write(f"{float(v):.6f}\n")


def save_region_vtk(filepath, coords_mm, elements, classification, region_code):
    """Save VTK for a single region"""
    mask = classification == region_code
    region_elems = np.where(mask)[0]
    
    if len(region_elems) == 0:
        return 0
    
    region_elements = elements[region_elems]
    unique_nodes = np.unique(region_elements.flatten())
    node_map = {old: new for new, old in enumerate(unique_nodes)}
    
    remapped = np.array([[node_map[n] for n in elem] for elem in region_elements])
    
    with open(filepath, 'w') as f:
        f.write("# vtk DataFile Version 3.0\nRegion\nASCII\n")
        f.write("DATASET UNSTRUCTURED_GRID\n")
        
        f.write(f"POINTS {len(unique_nodes)} float\n")
        for n in unique_nodes:
            c = coords_mm[n]
            f.write(f"{c[0]:.6f} {c[1]:.6f} {c[2]:.6f}\n")
        
        f.write(f"\nCELLS {len(region_elems)} {len(region_elems) * 5}\n")
        for e in remapped:
            f.write(f"4 {e[0]} {e[1]} {e[2]} {e[3]}\n")
        
        f.write(f"\nCELL_TYPES {len(region_elems)}\n")
        f.write("10\n" * len(region_elems))
    
    return len(region_elems)


def save_tagged_elem(filepath, elements, classification):
    """Save OpenCARP format"""
    with open(filepath, 'w') as f:
        f.write(f"{len(elements)}\n")
        for i, e in enumerate(elements):
            f.write(f"Tt {e[0]} {e[1]} {e[2]} {e[3]} {classification[i]}\n")

# PROCESS PATIENT

def process_patient(patient_id):
    """Process single patient"""
    print(f"PROCESSING: {patient_id}")
    
    try:
        pts_file, elem_file, lon_file = find_mesh_files(patient_id, BASE_DIR)
        
        coords = load_pts(pts_file)
        elements = load_elem(elem_file)
        fibers = load_lon(lon_file)
        
        results = classify_tissue_comprehensive(coords, elements, fibers, patient_id)
        
        # Save outputs
        patient_output = os.path.join(OUTPUT_DIR, patient_id)
        os.makedirs(patient_output, exist_ok=True)
        
        print("\n  [8] Saving outputs")
        
        save_vtk_comprehensive(
            os.path.join(patient_output, f"{patient_id}_classified.vtk"),
            results['coords_mm'], elements, results['classification'],
            results['metrics'], results['phi']
        )
        
        save_region_vtk(
            os.path.join(patient_output, f"{patient_id}_INFARCT.vtk"),
            results['coords_mm'], elements, results['classification'], 3
        )
        
        save_region_vtk(
            os.path.join(patient_output, f"{patient_id}_BORDER.vtk"),
            results['coords_mm'], elements, results['classification'], 2
        )
        
        save_tagged_elem(
            os.path.join(patient_output, f"{patient_id}_tagged.elem"),
            elements, results['classification']
        )
        
        # JSON summary
        def convert(o):
            if isinstance(o, (np.bool_, np.integer)):
                return int(o)
            if isinstance(o, np.floating):
                return float(o)
            if isinstance(o, dict):
                return {k: convert(v) for k, v in o.items()}
            if isinstance(o, (list, tuple, np.ndarray)):
                return [convert(i) for i in o]
            return o
        
        summary = {
            'patient_id': patient_id,
            'timestamp': datetime.now().isoformat(),
            'method': 'Comprehensive Multi-Metric Infarct Detection',
            'methodologies': [
                '1. Laplace-Dirichlet transmural coordinate',
                '2. Robust wall thickness from gradient',
                '3. Wall stress (modified Laplace law)',
                '4. Local fiber coherence',
                '5. Comprehensive scoring with weights'
            ],
            'stats': convert(results['stats']),
            'thresholds': convert(results['thresholds']),
            'validation': convert(results['validation']),
            'config': {
                'wt_dense_scar_mm': Config.WT_DENSE_SCAR_MM,
                'wt_scar_threshold_mm': Config.WT_SCAR_THRESHOLD_MM,
                'wt_border_threshold_mm': Config.WT_BORDER_THRESHOLD_MM,
                'lfc_scar_threshold': Config.LFC_SCAR_THRESHOLD,
                'max_angular_spread_deg': Config.MAX_ANGULAR_SPREAD_DEG,
                'weights': {
                    'wt': Config.WEIGHT_WT,
                    'lfc': Config.WEIGHT_LFC,
                    'stress': Config.WEIGHT_STRESS,
                    'subendo': Config.WEIGHT_SUBENDO,
                    'thinning': Config.WEIGHT_THINNING
                }
            }
        }
        
        with open(os.path.join(patient_output, f"{patient_id}_summary.json"), 'w') as f:
            json.dump(summary, f, indent=2)
        
        print(f"\n  Outputs saved to: {patient_output}")
        
        return results['stats']
        
    except Exception as e:
        print(f"\n  ERROR: {str(e)}")
        import traceback
        traceback.print_exc()
        return {'patient_id': patient_id, 'status': 'FAILED', 'error': str(e)}

# MAIN

def main():
    print("MULTI-METRIC INFARCT DETECTION")
    
    print("\nMETHODOLOGIES:")
    print("  1. Laplace-Dirichlet transmural coordinate (φ)")
    print("  2. Robust wall thickness from h = 1/|∇φ|")
    print("  3. Wall stress via modified Laplace law")
    print("  4. Local fiber coherence (LFC)")
    print("  5. Comprehensive scoring with anatomical constraints")
    
    print(f"\nLITERATURE-VALIDATED THRESHOLDS:")
    print(f"  - Dense scar: WT < {Config.WT_DENSE_SCAR_MM}mm")
    print(f"  - Scar: WT < {Config.WT_SCAR_THRESHOLD_MM}mm (Penicka: ≤5mm)")
    print(f"  - Border: WT < {Config.WT_BORDER_THRESHOLD_MM}mm")
    print(f"  - LFC scar: < {Config.LFC_SCAR_THRESHOLD}")
    print(f"  - Expected infarct: {Config.MIN_INFARCT_PCT}-{Config.MAX_INFARCT_PCT}%")
    
    print(f"\nSCORE WEIGHTS:")
    print(f"  - Wall thickness: {Config.WEIGHT_WT}")
    print(f"  - Fiber coherence: {Config.WEIGHT_LFC}")
    print(f"  - Wall stress: {Config.WEIGHT_STRESS}")
    print(f"  - Subendocardial: {Config.WEIGHT_SUBENDO}")
    print(f"  - Thinning ratio: {Config.WEIGHT_THINNING}")
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    all_results = []
    for patient_id in PATIENT_IDS:
        stats = process_patient(patient_id)
        all_results.append(stats)
    
    # Summary
    print("SUMMARY - DETECTION RESULTS")
    
    successful = [r for r in all_results if 'status' not in r]
    
    print(f"\n{'Patient':<15} {'Healthy%':>10} {'Border%':>10} {'Infarct%':>10} {'WT_mean':>10} {'Ang.Span':>10}")
    
    for i, stats in enumerate(all_results):
        if 'status' not in stats:
            pid = PATIENT_IDS[i]
            print(f"{pid:<15} "
                  f"{stats.get('pct_healthy', 0):>10.1f} "
                  f"{stats.get('pct_border', 0):>10.1f} "
                  f"{stats.get('pct_infarct', 0):>10.1f} "
                  f"{stats.get('wt_mean_mm', 0):>9.1f}mm "
                  f"{stats.get('angular_span_deg', 0):>9.1f}°")
    
    if successful:
        avg_h = np.mean([r['pct_healthy'] for r in successful])
        avg_b = np.mean([r['pct_border'] for r in successful])
        avg_i = np.mean([r['pct_infarct'] for r in successful])
        std_i = np.std([r['pct_infarct'] for r in successful])
        print(f"{'AVERAGE':<15} {avg_h:>10.1f} {avg_b:>10.1f} {avg_i:>10.1f}")
        print(f"{'STD DEV':<15} {'':<10} {'':<10} {std_i:>10.1f}")
        
        inf_vals = [r['pct_infarct'] for r in successful]
        print(f"\n  Infarct range: {min(inf_vals):.1f}% - {max(inf_vals):.1f}%")
    
    # Combined summary
    combined = {
        'timestamp': datetime.now().isoformat(),
        'method': 'Comprehensive Multi-Metric Infarct Detection',
        'methodologies': [
            'Laplace-Dirichlet transmural coordinate',
            'Robust wall thickness from gradient',
            'Wall stress (modified Laplace law)',
            'Local fiber coherence',
            'Comprehensive scoring'
        ],
        'literature': {
            'wall_thickness': 'Penicka et al. - WT≤5mm: 92% sens, 96% spec',
            'infarcted_wt': 'J Magn Reson Imaging 2018 - 2.86±1.11mm',
            'healthy_wt': 'J Magn Reson Imaging 2018 - 8.73±1.01mm',
            'expected_scar': 'Puntmann et al. JACC 2016 - 8-15%'
        },
        'results': all_results
    }
    
    with open(os.path.join(OUTPUT_DIR, "all_patients_summary.json"), 'w') as f:
        json.dump(combined, f, indent=2, default=str)
    
    print(f"\nResults saved to: {OUTPUT_DIR}")
    
    return all_results

if __name__ == "__main__":
    results = main()

MULTI-METRIC INFARCT DETECTION

METHODOLOGIES:
  1. Laplace-Dirichlet transmural coordinate (φ)
  2. Robust wall thickness from h = 1/|∇φ|
  3. Wall stress via modified Laplace law
  4. Local fiber coherence (LFC)
  5. Comprehensive scoring with anatomical constraints

LITERATURE-VALIDATED THRESHOLDS:
  - Dense scar: WT < 4.0mm
  - Scar: WT < 5.5mm (Penicka: ≤5mm)
  - Border: WT < 7.0mm
  - LFC scar: < 0.75
  - Expected infarct: 3.0-20.0%

SCORE WEIGHTS:
  - Wall thickness: 0.45
  - Fiber coherence: 0.2
  - Wall stress: 0.15
  - Subendocardial: 0.1
  - Thinning ratio: 0.1
PROCESSING: SCD0000101

  Mesh: 70,518 nodes, 383,439 elements

  [1] LAPLACE-DIRICHLET TRANSMURAL COORDINATE
      Extracting surfaces...
      Endo: 3071 nodes, Epi: 3071 nodes
      Building Laplacian matrix...
      Dirichlet BCs: 3071 endo, 3071 epi nodes
      Solving Laplace equation (FEM)...
      Transmural φ: [0.0000, 1.0000]

  [2] ROBUST WALL THICKNESS FROM ∇φ
      Scale factor: 3.5502
      WT: mean=12.2

In [ ]:
#!/usr/bin/env python3
"""
COMPLETE LAPLACE-DIRICHLET CARDIAC ANALYSIS FRAMEWORK

COMPONENT 1: LAPLACE-DIRICHLET TRANSMURAL COORDINATE
    - Solve ∇²φ = 0 with Dirichlet BCs
    - φ ∈ [0,1] defines transmural position
    
COMPONENT 2: HELICAL FIBER RECONSTRUCTION VIA φ
    - α(φ) = α_endo + (α_epi - α_endo) × φ
    - Fiber: f = cos(α)·e_c + sin(α)·e_l
    - Modified in infarct regions
    
COMPONENT 3: WALL STRESS FROM MODIFIED LAPLACE LAW WITH CURVATURE TENSOR
    - Classical: σ = Pr/(2h)
    - Modified: σ = Pr/(2h) × f(κ₁, κ₂)
    - κ₁, κ₂ = principal curvatures from shape operator
    
COMPONENT 4: LAPLACIAN-GEODESIC INJECTION SITE OPTIMIZATION
    - Geodesic distance via Heat Method (Crane et al. 2013)
    - Optimization: min(geodesic_to_border) + max(stress_reduction)
    - Constraints: avoid core scar, prefer mid-wall

"""

import numpy as np
from scipy.sparse import lil_matrix, csr_matrix, diags
from scipy.sparse.linalg import spsolve, eigsh
from scipy.spatial import cKDTree
from collections import defaultdict
import os
import json
from datetime import datetime
from multiprocessing import Pool, cpu_count
from functools import partial
import warnings
warnings.filterwarnings('ignore')

# Try to import joblib for parallel loops (more notebook-friendly)
try:
    from joblib import Parallel, delayed
    HAS_JOBLIB = True
except ImportError:
    HAS_JOBLIB = False
    print("Note: joblib not available, using sequential processing for loops")

# PARALLEL PROCESSING CONFIGURATION
N_CPUS = 44  # Number of CPUs to use
PARALLEL_PATIENTS = True  # Process patients in parallel
PARALLEL_ELEMENTS = True  # Parallelize element loops within each patient


PATIENT_IDS = [
    "SCD0000101", "SCD0000201", "SCD0000301", "SCD0000401",
    "SCD0000601", "SCD0000701", "SCD0000801", "SCD0001001",
    "SCD0001101", "SCD0001201"
]

BASE_DIR = "/home/shadeform/SCD_MODELS"
OUTPUT_DIR = "/home/shadeform/SCD_MODELS/laplace_complete"

# Fiber parameters (Streeter et al. 1969)
FIBER_ANGLE_ENDO_DEG = 60.0    # Subendocardial: right-handed helix
FIBER_ANGLE_EPI_DEG = -60.0    # Subepicardial: left-handed helix

# Pressure for wall stress
LV_PRESSURE_KPA = 16.0         # Peak systolic

# Injection optimization
N_INJECTION_SITES = 5
MIN_SITE_SEPARATION_MM = 15.0

# Tissue tags
TAG_HEALTHY = 1
TAG_BORDER = 2
TAG_INFARCT = 3


# MESH LOADING

def load_mesh(patient_id, base_dir):
    """Load tetrahedral mesh"""
    pts_file = f"{base_dir}/simulation_ready/{patient_id}/{patient_id}_tet.pts"
    elem_file = f"{base_dir}/simulation_ready/{patient_id}/{patient_id}_tet.elem"
    
    # Load coordinates
    with open(pts_file, 'r') as f:
        n_nodes = int(f.readline().strip())
        coords = np.zeros((n_nodes, 3), dtype=np.float64)
        for i in range(n_nodes):
            coords[i] = [float(x) for x in f.readline().split()[:3]]
    
    # Load elements
    with open(elem_file, 'r') as f:
        n_elems = int(f.readline().strip())
        elements = np.zeros((n_elems, 4), dtype=np.int32)
        for i in range(n_elems):
            parts = f.readline().split()
            elements[i] = [int(x) for x in parts[1:5]]
    
    return coords, elements


def load_classification(patient_id, base_dir):
    """Load tissue classification from coronary territory-based detection"""
    # Try new coronary territory results first, then fall back to comprehensive
    class_dir = f"{base_dir}/infarct_coronary_territory/{patient_id}"
    elem_file = f"{class_dir}/{patient_id}_tagged.elem"
    
    if not os.path.exists(elem_file):
        # Fallback to old location
        class_dir = f"{base_dir}/infarct_results_comprehensive/{patient_id}"
        elem_file = f"{class_dir}/{patient_id}_tagged.elem"
    
    if not os.path.exists(elem_file):
        return None
    
    with open(elem_file, 'r') as f:
        n_elems = int(f.readline().strip())
        tags = np.zeros(n_elems, dtype=np.int32)
        for i in range(n_elems):
            parts = f.readline().split()
            tags[i] = int(parts[5])
    
    return tags


# COMPONENT 1: LAPLACE-DIRICHLET TRANSMURAL COORDINATE

def extract_surfaces(coords, elements):
    """Extract endocardial and epicardial surfaces"""
    # Find boundary faces
    face_count = defaultdict(list)
    for elem_idx, elem in enumerate(elements):
        faces = [
            tuple(sorted([elem[0], elem[1], elem[2]])),
            tuple(sorted([elem[0], elem[1], elem[3]])),
            tuple(sorted([elem[0], elem[2], elem[3]])),
            tuple(sorted([elem[1], elem[2], elem[3]])),
        ]
        for face in faces:
            face_count[face].append(elem_idx)
    
    boundary_faces = [(f, elems[0]) for f, elems in face_count.items() if len(elems) == 1]
    
    # Get boundary nodes
    boundary_nodes = set()
    for face, _ in boundary_faces:
        boundary_nodes.update(face)
    boundary_nodes = np.array(list(boundary_nodes))
    
    # Classify by radial position
    z_vals = coords[:, 2]
    z_min, z_max = z_vals.min(), z_vals.max()
    z_range = z_max - z_min
    
    endo_nodes = set()
    epi_nodes = set()
    endo_faces = []
    epi_faces = []
    
    for i in range(20):
        z_lo = z_min + i * z_range / 20
        z_hi = z_min + (i + 1) * z_range / 20
        
        slice_mask = (coords[boundary_nodes, 2] >= z_lo) & (coords[boundary_nodes, 2] < z_hi)
        slice_nodes = boundary_nodes[slice_mask]
        
        if len(slice_nodes) < 20:
            continue
        
        slice_coords = coords[slice_nodes, :2]
        center = np.median(slice_coords, axis=0)
        radii = np.linalg.norm(slice_coords - center, axis=1)
        
        r_30 = np.percentile(radii, 30)
        r_70 = np.percentile(radii, 70)
        
        for j, node in enumerate(slice_nodes):
            if radii[j] < r_30:
                endo_nodes.add(node)
            elif radii[j] > r_70:
                epi_nodes.add(node)
    
    # Classify faces
    for face, elem_idx in boundary_faces:
        face_nodes = list(face)
        if all(n in endo_nodes for n in face_nodes):
            endo_faces.append(face_nodes)
        elif all(n in epi_nodes for n in face_nodes):
            epi_faces.append(face_nodes)
    
    return {
        'endo_nodes': np.array(list(endo_nodes)),
        'epi_nodes': np.array(list(epi_nodes)),
        'endo_faces': np.array(endo_faces) if endo_faces else np.array([]).reshape(0, 3),
        'epi_faces': np.array(epi_faces) if epi_faces else np.array([]).reshape(0, 3),
    }


def build_fem_laplacian(coords, elements):
    """Build FEM Laplacian stiffness matrix"""
    n_nodes = len(coords)
    K = lil_matrix((n_nodes, n_nodes))
    
    for elem in elements:
        X = coords[elem]
        J = np.array([X[1] - X[0], X[2] - X[0], X[3] - X[0]]).T
        
        detJ = np.linalg.det(J)
        if abs(detJ) < 1e-15:
            continue
        
        vol = abs(detJ) / 6.0
        Jinv = np.linalg.inv(J)
        
        # Shape function gradients
        dN = np.zeros((4, 3))
        dN[0] = -Jinv.sum(axis=1)
        dN[1] = Jinv[:, 0]
        dN[2] = Jinv[:, 1]
        dN[3] = Jinv[:, 2]
        
        # Element stiffness
        Ke = vol * (dN @ dN.T)
        
        for i in range(4):
            for j in range(4):
                K[elem[i], elem[j]] += Ke[i, j]
    
    return K.tocsr()


def solve_laplace_dirichlet(coords, elements, surfaces):
    """
    COMPONENT 1: Solve Laplace equation for transmural coordinate.
    
    Mathematical Formulation:
    ∇²φ = 0           in Ω (myocardium)
    φ = 0             on Γ_endo
    φ = 1             on Γ_epi
    
    Returns: φ(x) ∈ [0, 1] - smooth transmural scalar field
    """
    n_nodes = len(coords)
    
    print("      Building FEM Laplacian")
    K = build_fem_laplacian(coords, elements)
    
    endo_set = set(surfaces['endo_nodes'])
    epi_set = set(surfaces['epi_nodes'])
    
    print(f"      Dirichlet BCs: {len(endo_set)} endo, {len(epi_set)} epi nodes")
    
    # Apply Dirichlet BCs via penalty method
    K_mod = K.tolil()
    rhs = np.zeros(n_nodes)
    penalty = 1e12
    
    for node in endo_set:
        K_mod[node, :] = 0
        K_mod[node, node] = penalty
        rhs[node] = 0.0 * penalty
    
    for node in epi_set:
        K_mod[node, :] = 0
        K_mod[node, node] = penalty
        rhs[node] = 1.0 * penalty
    
    print("      Solving Laplace equation")
    phi = spsolve(K_mod.tocsr(), rhs)
    phi = np.clip(phi, 0, 1)
    
    return phi


def compute_phi_gradient(coords, elements, phi):
    """Compute gradient of transmural coordinate at each element"""
    n_elems = len(elements)
    grad_phi = np.zeros((n_elems, 3))
    
    for i, elem in enumerate(elements):
        X = coords[elem]
        phi_elem = phi[elem]
        
        J = np.array([X[1] - X[0], X[2] - X[0], X[3] - X[0]]).T
        detJ = np.linalg.det(J)
        
        if abs(detJ) < 1e-15:
            continue
        
        Jinv = np.linalg.inv(J)
        dphi_dxi = np.array([phi_elem[1] - phi_elem[0], 
                            phi_elem[2] - phi_elem[0], 
                            phi_elem[3] - phi_elem[0]])
        grad_phi[i] = Jinv @ dphi_dxi
    
    return grad_phi


# COMPONENT 2: HELICAL FIBER RECONSTRUCTION VIA TRANSMURAL COORDINATE

def compute_local_coordinate_system(coords, elements, phi, grad_phi):
    """
    Compute local orthonormal coordinate system at each element.
    
    e_t: Transmural direction = ∇φ / |∇φ|
    e_l: Longitudinal direction (apex → base), orthogonalized
    e_c: Circumferential direction = e_l × e_t
    """
    n_elems = len(elements)
    
    # Transmural direction
    grad_norm = np.linalg.norm(grad_phi, axis=1, keepdims=True)
    grad_norm = np.maximum(grad_norm, 1e-10)
    e_t = grad_phi / grad_norm
    
    # Long axis direction (roughly +z, pointing base-ward)
    centroids = np.mean(coords[elements], axis=1)
    
    # Find apex and base
    z_vals = centroids[:, 2]
    apex_z = z_vals.min()
    base_z = z_vals.max()
    
    # Initial longitudinal direction (towards base)
    e_l_init = np.zeros((n_elems, 3))
    e_l_init[:, 2] = 1.0  # +z direction
    
    # Orthogonalize e_l to e_t
    e_l = np.zeros((n_elems, 3))
    for i in range(n_elems):
        # Gram-Schmidt: e_l = e_l_init - (e_l_init · e_t) e_t
        proj = np.dot(e_l_init[i], e_t[i])
        e_l[i] = e_l_init[i] - proj * e_t[i]
        norm = np.linalg.norm(e_l[i])
        if norm > 1e-10:
            e_l[i] /= norm
        else:
            # Fallback: use perpendicular direction
            e_l[i] = np.cross(e_t[i], [1, 0, 0])
            norm = np.linalg.norm(e_l[i])
            if norm > 1e-10:
                e_l[i] /= norm
    
    # Circumferential direction: e_c = e_l × e_t
    e_c = np.cross(e_l, e_t)
    e_c /= np.linalg.norm(e_c, axis=1, keepdims=True) + 1e-10
    
    return e_t, e_l, e_c


def reconstruct_helical_fibers(coords, elements, phi, grad_phi, tags=None,
                               alpha_endo=FIBER_ANGLE_ENDO_DEG,
                               alpha_epi=FIBER_ANGLE_EPI_DEG):
    """
    COMPONENT 2: Reconstruct myofibers via helical angle rotation.
    
    Mathematical Formulation:
    α(φ) = α_endo + (α_epi - α_endo) × φ
         = 60° - 120° × φ
    
    Fiber direction:
    f = cos(α) · e_c + sin(α) · e_l
    
    Sheet direction:
    s = f × e_t (perpendicular to fiber, in wall plane)
    
    In infarct regions:
    - Core: α = 0° (circumferential, no helical rotation)
    - Border: α reduced to ±30° (partial preservation)
    
    Reference: Streeter et al. "Fiber orientation in the canine left ventricle 
               during diastole and systole." Circ Res 1969.
    """
    n_elems = len(elements)
    
    print("      Computing local coordinate system")
    e_t, e_l, e_c = compute_local_coordinate_system(coords, elements, phi, grad_phi)
    
    # Transmural position per element
    phi_elem = np.array([np.mean(phi[elem]) for elem in elements])
    
    # Fiber angle: linear interpolation through wall
    alpha_endo_rad = np.radians(alpha_endo)
    alpha_epi_rad = np.radians(alpha_epi)
    
    alpha = alpha_endo_rad + (alpha_epi_rad - alpha_endo_rad) * phi_elem
    
    # Modify in infarct regions
    if tags is not None:
        for i in range(n_elems):
            if tags[i] == TAG_INFARCT:
                # Core scar: no helical rotation (fibrotic, circumferential only)
                alpha[i] = 0.0
            elif tags[i] == TAG_BORDER:
                # Border zone: reduced rotation (50% of normal)
                alpha[i] = alpha[i] * 0.5
    
    # Compute fiber and sheet directions
    print("      Computing fiber orientations via helical rotation")
    fibers = np.zeros((n_elems, 3))
    sheets = np.zeros((n_elems, 3))
    
    for i in range(n_elems):
        # Fiber: f = cos(α) e_c + sin(α) e_l
        fibers[i] = np.cos(alpha[i]) * e_c[i] + np.sin(alpha[i]) * e_l[i]
        fibers[i] /= np.linalg.norm(fibers[i]) + 1e-10
        
        # Sheet: s = f × e_t
        sheets[i] = np.cross(fibers[i], e_t[i])
        sheets[i] /= np.linalg.norm(sheets[i]) + 1e-10
    
    fiber_angles_deg = np.degrees(alpha)
    
    print(f"      Fiber angle range: {fiber_angles_deg.min():.1f}° to {fiber_angles_deg.max():.1f}°")
    
    return fibers, sheets, fiber_angles_deg, (e_t, e_l, e_c)


# COMPONENT 3: WALL STRESS FROM MODIFIED LAPLACE LAW WITH CURVATURE TENSOR

def compute_surface_curvature_tensor(coords, faces, vertex_normals):
    """
    Compute principal curvatures from the shape operator (Weingarten map).
    
    The shape operator S relates the change in normal to surface position:
    S = -dN/dX
    
    Principal curvatures κ₁, κ₂ are the eigenvalues of S.
    Mean curvature: H = (κ₁ + κ₂) / 2
    Gaussian curvature: K = κ₁ × κ₂
    """
    n_nodes = len(coords)
    
    # Build vertex-to-face connectivity
    node_faces = defaultdict(list)
    for fi, face in enumerate(faces):
        for node in face:
            node_faces[node].append(fi)
    
    kappa_1 = np.zeros(n_nodes)  # Max principal curvature
    kappa_2 = np.zeros(n_nodes)  # Min principal curvature
    mean_curvature = np.zeros(n_nodes)
    gaussian_curvature = np.zeros(n_nodes)
    
    for node in node_faces.keys():
        if len(node_faces[node]) < 3:
            continue
        
        p = coords[node]
        n = vertex_normals[node]
        
        if np.linalg.norm(n) < 0.1:
            continue
        
        # Find neighbors
        neighbors = set()
        for fi in node_faces[node]:
            neighbors.update(faces[fi])
        neighbors.discard(node)
        
        if len(neighbors) < 3:
            continue
        
        neighbor_list = list(neighbors)
        neighbor_coords = coords[neighbor_list]
        neighbor_normals = vertex_normals[neighbor_list]
        
        # Project to tangent plane
        diffs = neighbor_coords - p
        
        # Build tangent basis
        t1 = diffs[0] - np.dot(diffs[0], n) * n
        if np.linalg.norm(t1) < 1e-10:
            continue
        t1 /= np.linalg.norm(t1)
        t2 = np.cross(n, t1)
        
        # Fit shape operator using least squares
        # dn = S @ dx in tangent coordinates
        A = []
        b = []
        for i, neighbor in enumerate(neighbor_list):
            dx = diffs[i]
            dn = neighbor_normals[i] - n
            
            # Project to tangent plane
            dx_t = np.array([np.dot(dx, t1), np.dot(dx, t2)])
            dn_t = np.array([np.dot(dn, t1), np.dot(dn, t2)])
            
            if np.linalg.norm(dx_t) > 1e-10:
                # Shape operator equation: dn = -S @ dx
                A.append([dx_t[0], dx_t[1], 0, 0])
                A.append([0, 0, dx_t[0], dx_t[1]])
                b.append(-dn_t[0])
                b.append(-dn_t[1])
        
        if len(A) < 4:
            continue
        
        A = np.array(A)
        b = np.array(b)
        
        # Solve for shape operator components [S11, S12, S21, S22]
        try:
            S_flat, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
            S = np.array([[S_flat[0], S_flat[1]], 
                         [S_flat[2], S_flat[3]]])
            
            # Symmetrize (shape operator should be symmetric)
            S = (S + S.T) / 2
            
            # Eigenvalues = principal curvatures
            eigenvalues = np.linalg.eigvalsh(S)
            kappa_1[node] = np.max(eigenvalues)
            kappa_2[node] = np.min(eigenvalues)
            mean_curvature[node] = (kappa_1[node] + kappa_2[node]) / 2
            gaussian_curvature[node] = kappa_1[node] * kappa_2[node]
        except:
            pass
    
    return {
        'kappa_1': kappa_1,
        'kappa_2': kappa_2,
        'mean_curvature': mean_curvature,
        'gaussian_curvature': gaussian_curvature
    }


def compute_vertex_normals(coords, faces):
    """Compute area-weighted vertex normals from surface faces"""
    n_nodes = len(coords)
    vertex_normals = np.zeros((n_nodes, 3))
    
    for face in faces:
        v0, v1, v2 = coords[face]
        e1 = v1 - v0
        e2 = v2 - v0
        n = np.cross(e1, e2)
        area = np.linalg.norm(n) / 2
        if area > 1e-10:
            n = n / (2 * area)  # Unit normal
            for node in face:
                vertex_normals[node] += n * area
    
    # Normalize
    norms = np.linalg.norm(vertex_normals, axis=1, keepdims=True)
    norms = np.maximum(norms, 1e-10)
    vertex_normals /= norms
    
    return vertex_normals


def compute_wall_stress_with_curvature(coords, elements, phi, grad_phi, surfaces,
                                        pressure=LV_PRESSURE_KPA, tags=None):
    """
    COMPONENT 3: Wall stress from modified Law of Laplace with curvature tensor.
    
    Mathematical Formulation:
    
    Classical Laplace Law (thin-walled sphere):
    σ = P × r / (2 × h)
    
    Modified with local curvature tensor:
    σ(x) = P × r_local(x) / (2 × h(x)) × f(κ₁, κ₂)
    
    Where:
    - h(x) = 1/|∇φ| = local wall thickness
    - r_local = 1/H = local radius of curvature (H = mean curvature)
    - κ₁, κ₂ = principal curvatures from shape operator
    - f(κ₁, κ₂) = curvature anisotropy factor = 1 + |κ₁/κ₂ - 1| × 0.5
    
    Reference: Zhong et al. "Finite element analysis of the stress distribution 
               in left ventricle aneurysm." Int J Cardiol 2008.
    """
    n_elems = len(elements)
    centroids = np.mean(coords[elements], axis=1)
    
    # Wall thickness from Laplace gradient: h = 1/|∇φ|
    grad_norm = np.linalg.norm(grad_phi, axis=1)
    grad_norm = np.maximum(grad_norm, 1e-8)
    wall_thickness = 1.0 / grad_norm
    
    # Clip to physiological bounds
    wall_thickness = np.clip(wall_thickness, 0.5, 25.0)
    
    print("      Computing surface curvature tensor")
    
    # Compute curvature on endocardial surface
    if len(surfaces['endo_faces']) > 0:
        vertex_normals = compute_vertex_normals(coords, surfaces['endo_faces'])
        curvature = compute_surface_curvature_tensor(coords, surfaces['endo_faces'], vertex_normals)
    else:
        # Fallback: estimate from radial position
        curvature = {'kappa_1': np.zeros(len(coords)), 
                     'kappa_2': np.zeros(len(coords)),
                     'mean_curvature': np.zeros(len(coords))}
    
    # Interpolate curvature to elements
    kappa_1_elem = np.zeros(n_elems)
    kappa_2_elem = np.zeros(n_elems)
    mean_curv_elem = np.zeros(n_elems)
    
    for i, elem in enumerate(elements):
        kappa_1_elem[i] = np.mean(curvature['kappa_1'][elem])
        kappa_2_elem[i] = np.mean(curvature['kappa_2'][elem])
        mean_curv_elem[i] = np.mean(curvature['mean_curvature'][elem])
    
    # Local radius of curvature: r = 1/|H|
    mean_curv_elem = np.maximum(np.abs(mean_curv_elem), 1e-6)
    local_radius = 1.0 / mean_curv_elem
    local_radius = np.clip(local_radius, 10, 200)  # mm bounds
    
    # Curvature anisotropy factor: f(κ₁, κ₂) = 1 + |κ₁/κ₂ - 1| × 0.5
    kappa_2_safe = np.where(np.abs(kappa_2_elem) > 1e-8, kappa_2_elem, 1e-8)
    anisotropy = np.abs(kappa_1_elem / kappa_2_safe - 1.0)
    curvature_factor = 1.0 + anisotropy * 0.5
    curvature_factor = np.clip(curvature_factor, 1.0, 2.5)
    
    # For elements far from surface, use geometric estimate
    z_vals = centroids[:, 2]
    for i in range(n_elems):
        if local_radius[i] > 150 or local_radius[i] < 15:
            # Estimate from radial position at this z-level
            z = z_vals[i]
            z_mask = np.abs(z_vals - z) < 5.0
            if np.sum(z_mask) > 10:
                level_centroids = centroids[z_mask, :2]
                center = np.mean(level_centroids, axis=0)
                radii = np.linalg.norm(level_centroids - center, axis=1)
                local_radius[i] = np.mean(radii)
    
    # Modified Law of Laplace: σ = P × r / (2h) × f(κ)
    wall_stress = (pressure * local_radius) / (2 * wall_thickness) * curvature_factor
    
    # Modify for tissue type
    if tags is not None:
        for i in range(n_elems):
            if tags[i] == TAG_INFARCT:
                # Scar is stiffer, but load is transferred to border
                wall_stress[i] *= 0.7
            elif tags[i] == TAG_BORDER:
                # Stress concentration at border!
                wall_stress[i] *= 1.5
    
    print(f"      Wall thickness: {wall_thickness.min():.2f} - {wall_thickness.max():.2f} mm")
    print(f"      Local radius: {local_radius.min():.2f} - {local_radius.max():.2f} mm")
    print(f"      Wall stress: {wall_stress.min():.2f} - {wall_stress.max():.2f} kPa")
    
    return {
        'wall_stress': wall_stress,
        'wall_thickness': wall_thickness,
        'local_radius': local_radius,
        'curvature_factor': curvature_factor,
        'kappa_1': kappa_1_elem,
        'kappa_2': kappa_2_elem,
        'mean_curvature': mean_curv_elem
    }


# COMPONENT 4: LAPLACIAN-GEODESIC INJECTION SITE OPTIMIZATION

def build_mass_matrix(coords, elements):
    """Build lumped mass matrix for heat equation"""
    n_nodes = len(coords)
    M = np.zeros(n_nodes)
    
    for elem in elements:
        X = coords[elem]
        J = np.array([X[1] - X[0], X[2] - X[0], X[3] - X[0]]).T
        vol = abs(np.linalg.det(J)) / 6.0
        
        for node in elem:
            M[node] += vol / 4.0
    
    return diags(M)


def compute_geodesic_heat_method(coords, elements, source_nodes, L, t_factor=1.0):
    """
    Compute geodesic distance via the Heat Method (Crane et al. 2013).
    
    Algorithm:
    1. Solve heat equation: (M - tL)u = δ_source
    2. Compute normalized gradient: X = -∇u / |∇u|
    3. Solve Poisson equation: Lφ = ∇·X
    
    The solution φ gives approximate geodesic distances.
    
    Reference: Crane, Weischedel, Wardetzky. "Geodesics in Heat: A New Approach 
               to Computing Distance Based on Heat Flow." ACM TOG 2013.
    """
    n_nodes = len(coords)
    n_elems = len(elements)
    
    # Build mass matrix
    M = build_mass_matrix(coords, elements)
    
    # Estimate time step from mean edge length
    edge_lengths = []
    for elem in elements[:min(1000, n_elems)]:
        for i in range(4):
            for j in range(i+1, 4):
                edge_lengths.append(np.linalg.norm(coords[elem[i]] - coords[elem[j]]))
    h = np.mean(edge_lengths)
    t = t_factor * h * h
    
    # Step 1: Solve heat equation (M - tL)u = b
    A = M - t * L
    
    b = np.zeros(n_nodes)
    for node in source_nodes:
        b[node] = 1.0
    
    u = spsolve(A.tocsr(), b)
    
    # Step 2: Compute normalized gradient field
    X = np.zeros((n_elems, 3))
    
    for i, elem in enumerate(elements):
        verts = coords[elem]
        u_elem = u[elem]
        
        J = np.array([verts[1] - verts[0], verts[2] - verts[0], verts[3] - verts[0]]).T
        detJ = np.linalg.det(J)
        
        if abs(detJ) < 1e-15:
            continue
        
        Jinv = np.linalg.inv(J)
        du_dxi = np.array([u_elem[1] - u_elem[0], u_elem[2] - u_elem[0], u_elem[3] - u_elem[0]])
        grad_u = Jinv @ du_dxi
        
        norm = np.linalg.norm(grad_u)
        if norm > 1e-10:
            X[i] = -grad_u / norm  # Normalize and flip
    
    # Step 3: Compute divergence and solve Poisson
    div_X = np.zeros(n_nodes)
    
    for i, elem in enumerate(elements):
        verts = coords[elem]
        J = np.array([verts[1] - verts[0], verts[2] - verts[0], verts[3] - verts[0]]).T
        detJ = np.linalg.det(J)
        
        if abs(detJ) < 1e-15:
            continue
        
        vol = abs(detJ) / 6.0
        Jinv = np.linalg.inv(J)
        
        # Shape function gradients
        dN = np.zeros((4, 3))
        dN[0] = -Jinv.sum(axis=1)
        dN[1] = Jinv[:, 0]
        dN[2] = Jinv[:, 1]
        dN[3] = Jinv[:, 2]
        
        for j in range(4):
            div_X[elem[j]] += vol * np.dot(dN[j], X[i])
    
    # Fix one node for Poisson equation
    L_mod = L.tolil()
    ref_node = source_nodes[0]
    L_mod[ref_node, :] = 0
    L_mod[ref_node, ref_node] = 1
    div_X[ref_node] = 0
    
    phi = spsolve(L_mod.tocsr(), div_X)
    phi = phi - phi[source_nodes].min()
    
    return phi


def optimize_injection_sites(coords, elements, tags, wall_stress, phi_trans,
                             geodesic_to_border, centroids, n_sites=5):
    """
    COMPONENT 4: Laplacian-geodesic optimization for injection sites.
    
    Optimization Objective:
    Maximize: score(x) = w₁ × proximity_to_border + w₂ × stress_reduction + 
                         w₃ × mid_wall_preference + w₄ × border_bonus
    
    Subject to:
    - Avoid core scar (no perfusion → injection won't diffuse)
    - Maintain spatial separation between sites (min_separation)
    - Prefer border zone (therapeutic target)
    - Prefer mid-wall (φ ≈ 0.5) for better distribution
    
    The geodesic distance ensures optimal path lengths on the manifold,
    while the stress component targets high-risk regions for remodeling.
    """
    n_elems = len(elements)
    
    # Transmural position per element
    phi_elem = np.array([np.mean(phi_trans[elem]) for elem in elements])
    
    # Geodesic distance to elements (average of node distances)
    geodesic_elem = np.array([np.mean(geodesic_to_border[elem]) for elem in elements])
    
    # Normalize stress
    stress_norm = wall_stress['wall_stress'] / np.median(wall_stress['wall_stress'])
    
    # Initialize scores
    scores = np.zeros(n_elems)
    
    for i in range(n_elems):
        # EXCLUDE: Core scar (no perfusion)
        if tags[i] == TAG_INFARCT:
            scores[i] = -np.inf
            continue
        
        # Component 1: Proximity to border zone (minimize geodesic distance)
        # Higher score when CLOSER to border
        dist_score = 1.0 / (geodesic_elem[i] + 1.0)
        
        # Component 2: Wall stress reduction potential (target high stress)
        stress_score = stress_norm[i]
        
        # Component 3: Mid-wall preference (φ ≈ 0.5)
        mid_wall_score = 1.0 - 2.0 * abs(phi_elem[i] - 0.5)
        
        # Component 4: Border zone bonus
        border_bonus = 2.0 if tags[i] == TAG_BORDER else 1.0
        
        # Combined score with weights
        scores[i] = (0.3 * dist_score + 
                     0.3 * stress_score + 
                     0.2 * mid_wall_score + 
                     0.2 * border_bonus)
    
    # Select top sites with spatial diversity
    selected_sites = []
    sorted_indices = np.argsort(scores)[::-1]
    
    for idx in sorted_indices:
        if len(selected_sites) >= n_sites:
            break
        
        if scores[idx] == -np.inf:
            continue
        
        # Check spatial separation
        centroid = centroids[idx]
        too_close = False
        
        for site_idx in selected_sites:
            dist = np.linalg.norm(centroid - centroids[site_idx])
            if dist < MIN_SITE_SEPARATION_MM:
                too_close = True
                break
        
        if not too_close:
            selected_sites.append(idx)
    
    # Compile results
    injection_sites = []
    for idx in selected_sites:
        injection_sites.append({
            'element_id': int(idx),
            'coordinates': centroids[idx].tolist(),
            'score': float(scores[idx]),
            'transmural_position': float(phi_elem[idx]),
            'wall_stress_kPa': float(wall_stress['wall_stress'][idx]),
            'geodesic_distance': float(geodesic_elem[idx]),
            'tissue_type': 'border' if tags[idx] == TAG_BORDER else 'healthy'
        })
    
    return injection_sites


# OUTPUT FUNCTIONS

def write_vtk_complete(filepath, coords, elements, scalars, vectors):
    """Write VTK with all computed fields"""
    n_nodes, n_elems = len(coords), len(elements)
    
    with open(filepath, 'w') as f:
        f.write("# vtk DataFile Version 3.0\n")
        f.write("Complete Laplace-Dirichlet Analysis\n")
        f.write("ASCII\n")
        f.write("DATASET UNSTRUCTURED_GRID\n")
        
        f.write(f"POINTS {n_nodes} float\n")
        for c in coords:
            f.write(f"{c[0]:.6f} {c[1]:.6f} {c[2]:.6f}\n")
        
        f.write(f"\nCELLS {n_elems} {n_elems * 5}\n")
        for e in elements:
            f.write(f"4 {e[0]} {e[1]} {e[2]} {e[3]}\n")
        
        f.write(f"\nCELL_TYPES {n_elems}\n")
        f.write("10\n" * n_elems)
        
        # Point data
        point_scalars = {k: v for k, v in scalars.items() if len(v) == n_nodes}
        if point_scalars:
            f.write(f"\nPOINT_DATA {n_nodes}\n")
            for name, data in point_scalars.items():
                f.write(f"SCALARS {name} float 1\nLOOKUP_TABLE default\n")
                for val in data:
                    f.write(f"{float(val):.6f}\n")
        
        # Cell data
        cell_scalars = {k: v for k, v in scalars.items() if len(v) == n_elems}
        if cell_scalars or vectors:
            f.write(f"\nCELL_DATA {n_elems}\n")
            
            for name, data in cell_scalars.items():
                f.write(f"SCALARS {name} float 1\nLOOKUP_TABLE default\n")
                for val in data:
                    f.write(f"{float(val):.6f}\n")
            
            for name, data in vectors.items():
                if len(data) == n_elems:
                    f.write(f"\nVECTORS {name} float\n")
                    for v in data:
                        f.write(f"{v[0]:.6f} {v[1]:.6f} {v[2]:.6f}\n")


def write_lon_file(filepath, fibers, sheets):
    """Write CARP-format fiber file"""
    with open(filepath, 'w') as f:
        f.write("2\n")  # 2 directions
        for fiber, sheet in zip(fibers, sheets):
            f.write(f"{fiber[0]:.6f} {fiber[1]:.6f} {fiber[2]:.6f} "
                   f"{sheet[0]:.6f} {sheet[1]:.6f} {sheet[2]:.6f}\n")


def write_injection_coords(filepath, sites):
    """Write injection site coordinates"""
    with open(filepath, 'w') as f:
        f.write("# Optimal injection sites from Laplacian-geodesic optimization\n")
        f.write("# X Y Z score stress_kPa geodesic_dist tissue_type\n")
        for site in sites:
            c = site['coordinates']
            f.write(f"{c[0]:.4f} {c[1]:.4f} {c[2]:.4f} "
                   f"{site['score']:.4f} {site['wall_stress_kPa']:.2f} "
                   f"{site['geodesic_distance']:.2f} {site['tissue_type']}\n")


# MAIN PIPELINE

def process_patient_complete(patient_id):
    """
    Complete Laplace-Dirichlet analysis implementing ALL components from the abstract.
    """
    print(f"PROCESSING: {patient_id}")
    
    results = {'patient_id': patient_id}
    
    # Load mesh
    print("\n  Loading mesh")
    coords, elements = load_mesh(patient_id, BASE_DIR)
    n_nodes, n_elems = len(coords), len(elements)
    print(f"      {n_nodes:,} nodes, {n_elems:,} elements")
    
    centroids = np.mean(coords[elements], axis=1)
    
    # Load tissue classification
    tags = load_classification(patient_id, BASE_DIR)
    if tags is None:
        print("      No classification found - using all healthy")
        tags = np.ones(n_elems, dtype=np.int32)
    else:
        print(f"      Classification loaded: {np.sum(tags==TAG_INFARCT)} infarct, "
              f"{np.sum(tags==TAG_BORDER)} border")
    
    # Extract surfaces
    print("\n  [1] LAPLACE-DIRICHLET TRANSMURAL COORDINATE")
    print("      Extracting surfaces")
    surfaces = extract_surfaces(coords, elements)
    print(f"      Endo: {len(surfaces['endo_nodes'])} nodes, "
          f"Epi: {len(surfaces['epi_nodes'])} nodes")
    
    # Solve Laplace-Dirichlet
    phi = solve_laplace_dirichlet(coords, elements, surfaces)
    print(f"      φ range: [{phi.min():.4f}, {phi.max():.4f}]")
    
    # Compute gradient
    grad_phi = compute_phi_gradient(coords, elements, phi)
    
    # COMPONENT 2: Helical Fiber Reconstruction
    print("\n  [2] HELICAL FIBER RECONSTRUCTION")
    fibers, sheets, fiber_angles, coord_system = reconstruct_helical_fibers(
        coords, elements, phi, grad_phi, tags,
        FIBER_ANGLE_ENDO_DEG, FIBER_ANGLE_EPI_DEG
    )
    
    # COMPONENT 3: Wall Stress with Curvature Tensor
    print("\n  [3] WALL STRESS (MODIFIED LAPLACE + CURVATURE TENSOR)")
    stress_data = compute_wall_stress_with_curvature(
        coords, elements, phi, grad_phi, surfaces, LV_PRESSURE_KPA, tags
    )
    
    # COMPONENT 4: Laplacian-Geodesic Injection Optimization
    print("\n  [4] LAPLACIAN-GEODESIC INJECTION OPTIMIZATION")
    
    # Compute geodesic distance to border zone
    border_elements = np.where(tags == TAG_BORDER)[0]
    if len(border_elements) > 0:
        border_nodes = list(set(elements[border_elements[:100]].flatten()))
        
        print("      Computing geodesic distances (Heat Method)")
        L = build_fem_laplacian(coords, elements)
        geodesic_to_border = compute_geodesic_heat_method(coords, elements, border_nodes, L)
        print(f"      Geodesic range: [{geodesic_to_border.min():.2f}, "
              f"{geodesic_to_border.max():.2f}]")
    else:
        print("      No border zone - using endo distance")
        geodesic_to_border = np.zeros(n_nodes)
        endo_tree = cKDTree(coords[surfaces['endo_nodes']])
        for i, c in enumerate(coords):
            d, _ = endo_tree.query(c)
            geodesic_to_border[i] = d
    
    # Optimize injection sites
    print("      Optimizing injection sites")
    injection_sites = optimize_injection_sites(
        coords, elements, tags, stress_data, phi,
        geodesic_to_border, centroids, N_INJECTION_SITES
    )
    
    print(f"      Selected {len(injection_sites)} optimal sites:")
    for site in injection_sites:
        print(f"        Element {site['element_id']}: "
              f"score={site['score']:.3f}, stress={site['wall_stress_kPa']:.1f}kPa, "
              f"type={site['tissue_type']}")
    
    # Save outputs
    print("\n  [5] SAVING OUTPUTS")
    
    out_dir = os.path.join(OUTPUT_DIR, patient_id)
    os.makedirs(out_dir, exist_ok=True)
    
    # Transmural position per element
    phi_elem = np.array([np.mean(phi[elem]) for elem in elements])
    
    # VTK with all fields
    write_vtk_complete(
        os.path.join(out_dir, f"{patient_id}_complete_analysis.vtk"),
        coords, elements,
        scalars={
            'TransmuralPhi': phi,  # Node
            'TransmuralPhi_elem': phi_elem,  # Cell
            'TissueType': tags.astype(float),
            'FiberAngle_deg': fiber_angles,
            'WallThickness_mm': stress_data['wall_thickness'],
            'WallStress_kPa': stress_data['wall_stress'],
            'LocalRadius_mm': stress_data['local_radius'],
            'CurvatureFactor': stress_data['curvature_factor'],
            'Kappa1': stress_data['kappa_1'],
            'Kappa2': stress_data['kappa_2'],
        },
        vectors={
            'FiberDirection': fibers,
            'SheetDirection': sheets,
            'TransmuralDirection': coord_system[0],
        }
    )
    
    # Fiber file (CARP format)
    write_lon_file(os.path.join(out_dir, f"{patient_id}_reconstructed.lon"), fibers, sheets)
    
    # Injection sites
    with open(os.path.join(out_dir, f"{patient_id}_injection_sites.json"), 'w') as f:
        json.dump(injection_sites, f, indent=2)
    
    write_injection_coords(
        os.path.join(out_dir, f"{patient_id}_injection_coords.txt"),
        injection_sites
    )
    
    # Summary
    summary = {
        'patient_id': patient_id,
        'timestamp': datetime.now().isoformat(),
        'methodology': {
            'component_1': 'Laplace-Dirichlet transmural coordinate (∇²φ=0)',
            'component_2': 'Helical fiber reconstruction (α(φ) = 60° - 120°×φ)',
            'component_3': 'Wall stress with curvature tensor (σ = Pr/(2h)×f(κ₁,κ₂))',
            'component_4': 'Laplacian-geodesic injection optimization (Heat Method)'
        },
        'mesh': {
            'n_nodes': int(n_nodes),
            'n_elements': int(n_elems),
        },
        'transmural': {
            'phi_range': [float(phi.min()), float(phi.max())],
        },
        'fibers': {
            'angle_range_deg': [float(fiber_angles.min()), float(fiber_angles.max())],
            'endo_angle': FIBER_ANGLE_ENDO_DEG,
            'epi_angle': FIBER_ANGLE_EPI_DEG,
        },
        'wall_stress': {
            'range_kPa': [float(stress_data['wall_stress'].min()), 
                         float(stress_data['wall_stress'].max())],
            'mean_kPa': float(np.mean(stress_data['wall_stress'])),
            'pressure_kPa': LV_PRESSURE_KPA,
        },
        'wall_thickness': {
            'range_mm': [float(stress_data['wall_thickness'].min()),
                        float(stress_data['wall_thickness'].max())],
            'mean_mm': float(np.mean(stress_data['wall_thickness'])),
        },
        'injection_sites': injection_sites,
    }
    
    with open(os.path.join(out_dir, f"{patient_id}_summary.json"), 'w') as f:
        json.dump(summary, f, indent=2, default=str)
    
    print(f"\n  Outputs saved to: {out_dir}")
    
    return summary


def process_patient_wrapper(patient_id):
    """Wrapper for parallel processing"""
    try:
        result = process_patient_complete(patient_id)
        return result
    except Exception as e:
        import traceback
        traceback.print_exc()
        return {'patient_id': patient_id, 'status': 'FAILED', 'error': str(e)}


def main():
    """Main entry point with parallel processing"""
    print("COMPLETE LAPLACE-DIRICHLET CARDIAC ANALYSIS FRAMEWORK")
    
    print(f"\n  Using {N_CPUS} CPUs for parallel processing")
    
    print("\nIMPLEMENTED COMPONENTS (from abstract):")
    print("  [1] Laplace-Dirichlet transmural coordinate: ∇²φ = 0")
    print("  [2] Helical fiber reconstruction: α(φ) = α_endo + (α_epi - α_endo)×φ")
    print("  [3] Wall stress with curvature tensor: σ = Pr/(2h) × f(κ₁,κ₂)")
    print("  [4] Laplacian-geodesic injection optimization: Heat Method + stress")
    
    print("\n  Loading tissue tags from: infarct_coronary_territory/")
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # Set environment variables for multi-threaded linear algebra
    import os as os_env
    threads_per_patient = max(1, N_CPUS // len(PATIENT_IDS))
    os_env.environ['OMP_NUM_THREADS'] = str(threads_per_patient)
    os_env.environ['MKL_NUM_THREADS'] = str(threads_per_patient)
    os_env.environ['OPENBLAS_NUM_THREADS'] = str(threads_per_patient)
    os_env.environ['NUMEXPR_NUM_THREADS'] = str(threads_per_patient)
    
    print(f"  Processing {len(PATIENT_IDS)} patients in parallel")
    print(f"  Threads per patient: {threads_per_patient}")
    
    # Process patients in parallel
    n_workers = min(N_CPUS, len(PATIENT_IDS))
    
    with Pool(processes=n_workers) as pool:
        all_results = pool.map(process_patient_wrapper, PATIENT_IDS)
    
    # Summary table
    print("SUMMARY")
    
    print(f"\n{'Patient':<15} {'φ range':<15} {'Fiber°':<15} {'Stress kPa':<15} {'Sites':<10}")
    
    for r in all_results:
        if 'status' not in r:
            phi_r = r.get('transmural', {}).get('phi_range', [0, 1])
            fib_r = r.get('fibers', {}).get('angle_range_deg', [-60, 60])
            stress_r = r.get('wall_stress', {}).get('range_kPa', [0, 0])
            sites = len(r.get('injection_sites', []))
            print(f"{r['patient_id']:<15} "
                  f"[{phi_r[0]:.2f}, {phi_r[1]:.2f}]     "
                  f"[{fib_r[0]:.0f}°, {fib_r[1]:.0f}°]    "
                  f"[{stress_r[0]:.1f}, {stress_r[1]:.1f}]    "
                  f"{sites}")
        else:
            print(f"{r['patient_id']:<15} FAILED: {r.get('error', 'Unknown')[:40]}")
    
    # Save combined
    with open(os.path.join(OUTPUT_DIR, "complete_analysis_summary.json"), 'w') as f:
        json.dump(all_results, f, indent=2, default=str)
    
    print(f"\nResults saved to: {OUTPUT_DIR}")
    
    return all_results


if __name__ == "__main__":
    results = main()


def main_notebook():
    """
    Alternative main function optimized for Jupyter notebooks.
    Uses joblib instead of multiprocessing Pool (more compatible with notebooks).
    """
    print("COMPLETE LAPLACE-DIRICHLET CARDIAC ANALYSIS FRAMEWORK")
    
    print(f"\n  Using {N_CPUS} CPUs for parallel processing (notebook mode)")
    
    print("\nIMPLEMENTED COMPONENTS (from abstract):")
    print("  [1] Laplace-Dirichlet transmural coordinate: ∇²φ = 0")
    print("  [2] Helical fiber reconstruction: α(φ) = α_endo + (α_epi - α_endo)×φ")
    print("  [3] Wall stress with curvature tensor: σ = Pr/(2h) × f(κ₁,κ₂)")
    print("  [4] Laplacian-geodesic injection optimization: Heat Method + stress")
    
    print("\n  Loading tissue tags from: infarct_coronary_territory/")
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # Set environment variables for multi-threaded linear algebra
    os.environ['OMP_NUM_THREADS'] = str(N_CPUS)
    os.environ['MKL_NUM_THREADS'] = str(N_CPUS)
    os.environ['OPENBLAS_NUM_THREADS'] = str(N_CPUS)
    os.environ['NUMEXPR_NUM_THREADS'] = str(N_CPUS)
    
    print(f"  Processing {len(PATIENT_IDS)} patients")
    
    if HAS_JOBLIB:
        # Use joblib for parallel processing (notebook-friendly)
        all_results = Parallel(n_jobs=min(N_CPUS, len(PATIENT_IDS)), verbose=10)(
            delayed(process_patient_wrapper)(pid) for pid in PATIENT_IDS
        )
    else:
        # Sequential fallback
        all_results = []
        for pid in PATIENT_IDS:
            result = process_patient_wrapper(pid)
            all_results.append(result)
    
    # Summary table
    print("SUMMARY")
    
    print(f"\n{'Patient':<15} {'φ range':<15} {'Fiber°':<15} {'Stress kPa':<15} {'Sites':<10}")
    
    for r in all_results:
        if 'status' not in r:
            phi_r = r.get('transmural', {}).get('phi_range', [0, 1])
            fib_r = r.get('fibers', {}).get('angle_range_deg', [-60, 60])
            stress_r = r.get('wall_stress', {}).get('range_kPa', [0, 0])
            sites = len(r.get('injection_sites', []))
            print(f"{r['patient_id']:<15} "
                  f"[{phi_r[0]:.2f}, {phi_r[1]:.2f}]     "
                  f"[{fib_r[0]:.0f}°, {fib_r[1]:.0f}°]    "
                  f"[{stress_r[0]:.1f}, {stress_r[1]:.1f}]    "
                  f"{sites}")
        else:
            print(f"{r['patient_id']:<15} FAILED: {r.get('error', 'Unknown')[:40]}")
    
    # Save combined
    with open(os.path.join(OUTPUT_DIR, "complete_analysis_summary.json"), 'w') as f:
        json.dump(all_results, f, indent=2, default=str)
    
    print(f"\nResults saved to: {OUTPUT_DIR}")
    
    return all_results


# For notebook usage:
# from laplace_dirichlet_complete import main_notebook
# results = main_notebook()

In [ ]:
#!/usr/bin/env python3
"""

FIXES APPLIED:
1. Wall thickness computed (not from Laplace gradient)
   - Eliminates vertical stripe artifacts
   - Direct endo-to-epi distance along transmural direction


"""

import numpy as np
from scipy.sparse import lil_matrix, csr_matrix, diags
from scipy.sparse.linalg import spsolve
from scipy.spatial import cKDTree
from collections import defaultdict
import os
import json
from datetime import datetime
from multiprocessing import Pool, cpu_count
import warnings
warnings.filterwarnings('ignore')


N_CPUS = 44  # Your notebook's CPU count

PATIENT_IDS = [
    "SCD0000101", "SCD0000201", "SCD0000301", "SCD0000401",
    "SCD0000601", "SCD0000701", "SCD0000801", "SCD0001001",
    "SCD0001101", "SCD0001201"
]

BASE_DIR = "/home/shadeform/SCD_MODELS"
OUTPUT_DIR = "/home/shadeform/SCD_MODELS/laplace_fixed"

# Fiber parameters (Streeter et al. 1969)
FIBER_ANGLE_ENDO_DEG = 60.0
FIBER_ANGLE_EPI_DEG = -60.0

# Pressure for wall stress
LV_PRESSURE_KPA = 16.0

# Injection optimization
N_INJECTION_SITES = 5
MIN_SITE_SEPARATION_MM = 15.0

# Tissue tags
TAG_HEALTHY = 1
TAG_BORDER = 2
TAG_INFARCT = 3


# MESH LOADING (Vectorized)

def load_mesh_fast(patient_id, base_dir):
    """Load tetrahedral mesh with optimized I/O"""
    pts_file = f"{base_dir}/simulation_ready/{patient_id}/{patient_id}_tet.pts"
    elem_file = f"{base_dir}/simulation_ready/{patient_id}/{patient_id}_tet.elem"
    
    # Load coordinates - use numpy for speed
    with open(pts_file, 'r') as f:
        n_nodes = int(f.readline().strip())
        lines = f.readlines()
    
    coords = np.array([[float(x) for x in line.split()[:3]] for line in lines[:n_nodes]])
    
    # Load elements
    with open(elem_file, 'r') as f:
        n_elems = int(f.readline().strip())
        lines = f.readlines()
    
    elements = np.array([[int(x) for x in line.split()[1:5]] for line in lines[:n_elems]], dtype=np.int32)
    
    return coords, elements


def load_classification(patient_id, base_dir):
    """Load tissue classification"""
    class_dir = f"{base_dir}/infarct_results_comprehensive/{patient_id}"
    elem_file = f"{class_dir}/{patient_id}_tagged.elem"
    
    if not os.path.exists(elem_file):
        return None
    
    with open(elem_file, 'r') as f:
        n_elems = int(f.readline().strip())
        lines = f.readlines()
    
    tags = np.array([int(line.split()[5]) for line in lines[:n_elems]], dtype=np.int32)
    return tags


# SURFACE EXTRACTION (Vectorized)

def extract_surfaces_fast(coords, elements):
    """Extract endo/epi surfaces efficiently"""
    n_elems = len(elements)
    
    # Build face dictionary using vectorized operations
    face_count = defaultdict(list)
    
    # Generate all faces
    face_indices = np.array([[0,1,2], [0,1,3], [0,2,3], [1,2,3]])
    
    for elem_idx, elem in enumerate(elements):
        for fi in range(4):
            face = tuple(sorted(elem[face_indices[fi]]))
            face_count[face].append(elem_idx)
    
    # Get boundary faces
    boundary_faces = [(f, elems[0]) for f, elems in face_count.items() if len(elems) == 1]
    
    # Get boundary nodes
    boundary_nodes = set()
    for face, _ in boundary_faces:
        boundary_nodes.update(face)
    boundary_nodes = np.array(list(boundary_nodes))
    
    # Classify by radial position using vectorized operations
    z_vals = coords[:, 2]
    z_min, z_max = z_vals.min(), z_vals.max()
    z_range = z_max - z_min
    
    endo_nodes = set()
    epi_nodes = set()
    
    n_slices = 25
    for i in range(n_slices):
        z_lo = z_min + i * z_range / n_slices
        z_hi = z_min + (i + 1) * z_range / n_slices
        
        # Vectorized slice selection
        slice_mask = (coords[boundary_nodes, 2] >= z_lo) & (coords[boundary_nodes, 2] < z_hi)
        slice_nodes = boundary_nodes[slice_mask]
        
        if len(slice_nodes) < 20:
            continue
        
        # Vectorized center and radii computation
        slice_coords = coords[slice_nodes, :2]
        center = np.median(slice_coords, axis=0)
        radii = np.linalg.norm(slice_coords - center, axis=1)
        
        r_30 = np.percentile(radii, 30)
        r_70 = np.percentile(radii, 70)
        
        # Vectorized classification
        endo_mask = radii < r_30
        epi_mask = radii > r_70
        
        endo_nodes.update(slice_nodes[endo_mask])
        epi_nodes.update(slice_nodes[epi_mask])
    
    # Classify faces
    endo_faces = []
    epi_faces = []
    
    for face, elem_idx in boundary_faces:
        face_nodes = list(face)
        if all(n in endo_nodes for n in face_nodes):
            endo_faces.append(face_nodes)
        elif all(n in epi_nodes for n in face_nodes):
            epi_faces.append(face_nodes)
    
    return {
        'endo_nodes': np.array(list(endo_nodes)),
        'epi_nodes': np.array(list(epi_nodes)),
        'endo_faces': np.array(endo_faces) if endo_faces else np.array([]).reshape(0, 3),
        'epi_faces': np.array(epi_faces) if epi_faces else np.array([]).reshape(0, 3),
    }


# LAPLACE-DIRICHLET TRANSMURAL COORDINATE

def build_fem_laplacian_fast(coords, elements):
    """Build FEM Laplacian with vectorized element loop"""
    n_nodes = len(coords)
    n_elems = len(elements)
    
    # Pre-allocate for COO format (faster assembly)
    max_entries = n_elems * 16
    rows = np.zeros(max_entries, dtype=np.int32)
    cols = np.zeros(max_entries, dtype=np.int32)
    vals = np.zeros(max_entries, dtype=np.float64)
    
    entry_idx = 0
    
    for elem in elements:
        X = coords[elem]
        J = np.array([X[1] - X[0], X[2] - X[0], X[3] - X[0]]).T
        
        detJ = np.linalg.det(J)
        if abs(detJ) < 1e-15:
            continue
        
        vol = abs(detJ) / 6.0
        Jinv = np.linalg.inv(J)
        
        # Shape function gradients
        dN = np.zeros((4, 3))
        dN[0] = -Jinv.sum(axis=1)
        dN[1] = Jinv[:, 0]
        dN[2] = Jinv[:, 1]
        dN[3] = Jinv[:, 2]
        
        # Element stiffness
        Ke = vol * (dN @ dN.T)
        
        # Store in COO format
        for i in range(4):
            for j in range(4):
                rows[entry_idx] = elem[i]
                cols[entry_idx] = elem[j]
                vals[entry_idx] = Ke[i, j]
                entry_idx += 1
    
    # Trim and convert to CSR
    from scipy.sparse import coo_matrix
    K = coo_matrix((vals[:entry_idx], (rows[:entry_idx], cols[:entry_idx])), 
                   shape=(n_nodes, n_nodes))
    return K.tocsr()


def solve_laplace_dirichlet(coords, elements, surfaces):
    """Solve Laplace equation for transmural coordinate"""
    n_nodes = len(coords)
    
    print("      Building FEM Laplacian")
    K = build_fem_laplacian_fast(coords, elements)
    
    endo_set = set(surfaces['endo_nodes'])
    epi_set = set(surfaces['epi_nodes'])
    
    print(f"      Dirichlet BCs: {len(endo_set)} endo, {len(epi_set)} epi nodes")
    
    # Apply Dirichlet BCs via penalty method
    K_mod = K.tolil()
    rhs = np.zeros(n_nodes)
    penalty = 1e12
    
    for node in endo_set:
        K_mod[node, :] = 0
        K_mod[node, node] = penalty
        rhs[node] = 0.0 * penalty
    
    for node in epi_set:
        K_mod[node, :] = 0
        K_mod[node, node] = penalty
        rhs[node] = 1.0 * penalty
    
    print("      Solving Laplace equation")
    phi = spsolve(K_mod.tocsr(), rhs)
    phi = np.clip(phi, 0, 1)
    
    return phi


# FIXED: GEOMETRIC WALL THICKNESS (NOT FROM LAPLACE GRADIENT)

def compute_wall_thickness_geometric(coords, elements, surfaces, phi):
    """
    FIXED: Compute wall thickness GEOMETRICALLY.
    
    This eliminates the vertical stripe artifacts from h = 1/|∇φ|.
    
    Method:
    1. For each element centroid, find transmural direction from ∇φ
    2. Cast ray toward endo and epi surfaces
    3. Wall thickness = endo_distance + epi_distance
    
    Fallback: Use KDTree distance to surfaces if ray fails.
    """
    n_elems = len(elements)
    centroids = np.mean(coords[elements], axis=1)
    
    # Build KD-trees for fast surface queries
    endo_coords = coords[surfaces['endo_nodes']]
    epi_coords = coords[surfaces['epi_nodes']]
    
    endo_tree = cKDTree(endo_coords)
    epi_tree = cKDTree(epi_coords)
    
    # Compute gradient direction (for ray casting direction)
    grad_phi = compute_phi_gradient_fast(coords, elements, phi)
    grad_norm = np.linalg.norm(grad_phi, axis=1, keepdims=True)
    grad_norm = np.maximum(grad_norm, 1e-10)
    transmural_dir = grad_phi / grad_norm
    
    # Compute wall thickness
    wall_thickness = np.zeros(n_elems)
    endo_dist = np.zeros(n_elems)
    epi_dist = np.zeros(n_elems)
    
    for i in range(n_elems):
        c = centroids[i]
        
        # Distance to endocardium
        d_endo, _ = endo_tree.query(c)
        endo_dist[i] = d_endo
        
        # Distance to epicardium  
        d_epi, _ = epi_tree.query(c)
        epi_dist[i] = d_epi
        
        # Wall thickness = sum of distances weighted by transmural position
        # At mid-wall (φ=0.5): WT = 2 * min(d_endo, d_epi)
        # This gives consistent thickness estimate
        phi_elem = np.mean(phi[elements[i]])
        
        # Weighted combination based on transmural position
        wall_thickness[i] = d_endo / max(phi_elem, 0.01) if phi_elem < 0.5 else d_epi / max(1 - phi_elem, 0.01)
        
        # Clamp to reasonable range
        wall_thickness[i] = np.clip(wall_thickness[i], 2.0, 25.0)
    
    # Smooth circumferentially to remove any remaining artifacts
    wall_thickness = smooth_circumferentially(centroids, wall_thickness)
    
    return wall_thickness, endo_dist, epi_dist, transmural_dir


def smooth_circumferentially(centroids, values, n_neighbors=15):
    """
    Smooth values circumferentially at each z-level.
    This removes vertical stripe artifacts.
    """
    z_vals = centroids[:, 2]
    z_min, z_max = z_vals.min(), z_vals.max()
    z_range = z_max - z_min
    
    smoothed = values.copy()
    
    # Process each z-slice
    n_slices = 30
    for i in range(n_slices):
        z_lo = z_min + i * z_range / n_slices
        z_hi = z_min + (i + 1) * z_range / n_slices
        
        slice_mask = (z_vals >= z_lo) & (z_vals < z_hi)
        slice_idx = np.where(slice_mask)[0]
        
        if len(slice_idx) < n_neighbors * 2:
            continue
        
        # Get XY positions and sort by angle
        slice_centroids = centroids[slice_idx, :2]
        center = np.mean(slice_centroids, axis=0)
        
        angles = np.arctan2(slice_centroids[:, 1] - center[1], 
                           slice_centroids[:, 0] - center[0])
        
        # Sort by angle
        sort_idx = np.argsort(angles)
        sorted_vals = values[slice_idx[sort_idx]]
        
        # Moving average (circular)
        kernel_size = min(n_neighbors, len(sorted_vals) // 4)
        if kernel_size < 3:
            continue
        
        # Pad for circular smoothing
        padded = np.concatenate([sorted_vals[-kernel_size:], sorted_vals, sorted_vals[:kernel_size]])
        
        # Convolve
        kernel = np.ones(kernel_size * 2 + 1) / (kernel_size * 2 + 1)
        smoothed_slice = np.convolve(padded, kernel, mode='valid')
        
        # Handle size mismatch
        if len(smoothed_slice) >= len(sorted_vals):
            smoothed_slice = smoothed_slice[:len(sorted_vals)]
        
        # Unsort
        unsort_idx = np.argsort(sort_idx)
        smoothed[slice_idx] = smoothed_slice[unsort_idx[:len(slice_idx)]]
    
    return smoothed


def compute_phi_gradient_fast(coords, elements, phi):
    """Compute gradient of phi at each element (vectorized)"""
    n_elems = len(elements)
    grad_phi = np.zeros((n_elems, 3))
    
    for i, elem in enumerate(elements):
        X = coords[elem]
        phi_elem = phi[elem]
        
        J = np.array([X[1] - X[0], X[2] - X[0], X[3] - X[0]]).T
        detJ = np.linalg.det(J)
        
        if abs(detJ) < 1e-15:
            continue
        
        Jinv = np.linalg.inv(J)
        dphi_dxi = np.array([phi_elem[1] - phi_elem[0], 
                            phi_elem[2] - phi_elem[0], 
                            phi_elem[3] - phi_elem[0]])
        grad_phi[i] = Jinv @ dphi_dxi
    
    return grad_phi


# HELICAL FIBER RECONSTRUCTION

def compute_local_coords_vectorized(coords, elements, grad_phi):
    """Compute local coordinate system (vectorized)"""
    n_elems = len(elements)
    
    # Transmural direction
    grad_norm = np.linalg.norm(grad_phi, axis=1, keepdims=True)
    grad_norm = np.maximum(grad_norm, 1e-10)
    e_t = grad_phi / grad_norm
    
    # Longitudinal direction (+z, orthogonalized)
    e_l = np.zeros((n_elems, 3))
    e_l[:, 2] = 1.0
    
    # Gram-Schmidt orthogonalization (vectorized)
    proj = np.sum(e_l * e_t, axis=1, keepdims=True)
    e_l = e_l - proj * e_t
    e_l_norm = np.linalg.norm(e_l, axis=1, keepdims=True)
    e_l_norm = np.maximum(e_l_norm, 1e-10)
    e_l = e_l / e_l_norm
    
    # Circumferential direction (vectorized cross product)
    e_c = np.cross(e_l, e_t)
    e_c_norm = np.linalg.norm(e_c, axis=1, keepdims=True)
    e_c_norm = np.maximum(e_c_norm, 1e-10)
    e_c = e_c / e_c_norm
    
    return e_t, e_l, e_c


def reconstruct_helical_fibers(coords, elements, phi, grad_phi, tags=None):
    """
    Reconstruct myofibers via helical angle rotation.
    
    α(φ) = α_endo + (α_epi - α_endo) × φ = 60° - 120° × φ
    """
    n_elems = len(elements)
    
    print("      Computing local coordinate system")
    e_t, e_l, e_c = compute_local_coords_vectorized(coords, elements, grad_phi)
    
    # Transmural position per element (vectorized)
    phi_elem = np.mean(phi[elements], axis=1)
    
    # Fiber angle
    alpha_endo_rad = np.radians(FIBER_ANGLE_ENDO_DEG)
    alpha_epi_rad = np.radians(FIBER_ANGLE_EPI_DEG)
    
    alpha = alpha_endo_rad + (alpha_epi_rad - alpha_endo_rad) * phi_elem
    
    # Modify in infarct regions
    if tags is not None:
        infarct_mask = tags == TAG_INFARCT
        border_mask = tags == TAG_BORDER
        alpha[infarct_mask] = 0.0  # Circumferential only
        alpha[border_mask] = alpha[border_mask] * 0.5  # Reduced rotation
    
    # Compute fiber directions (vectorized)
    cos_alpha = np.cos(alpha)[:, np.newaxis]
    sin_alpha = np.sin(alpha)[:, np.newaxis]
    
    fibers = cos_alpha * e_c + sin_alpha * e_l
    fibers = fibers / (np.linalg.norm(fibers, axis=1, keepdims=True) + 1e-10)
    
    # Sheet directions (vectorized)
    sheets = np.cross(fibers, e_t)
    sheets = sheets / (np.linalg.norm(sheets, axis=1, keepdims=True) + 1e-10)
    
    fiber_angles_deg = np.degrees(alpha)
    
    print(f"      Fiber angle range: {fiber_angles_deg.min():.1f}° to {fiber_angles_deg.max():.1f}°")
    
    return fibers, sheets, fiber_angles_deg, (e_t, e_l, e_c)


# WALL STRESS WITH CURVATURE TENSOR

def compute_vertex_normals_fast(coords, faces):
    """Compute area-weighted vertex normals (vectorized)"""
    n_nodes = len(coords)
    vertex_normals = np.zeros((n_nodes, 3))
    
    if len(faces) == 0:
        return vertex_normals
    
    # Vectorized face normal computation
    v0 = coords[faces[:, 0]]
    v1 = coords[faces[:, 1]]
    v2 = coords[faces[:, 2]]
    
    e1 = v1 - v0
    e2 = v2 - v0
    normals = np.cross(e1, e2)
    areas = np.linalg.norm(normals, axis=1) / 2
    
    # Normalize
    valid = areas > 1e-10
    normals[valid] = normals[valid] / (2 * areas[valid, np.newaxis])
    
    # Accumulate to vertices
    for i, face in enumerate(faces):
        if valid[i]:
            for node in face:
                vertex_normals[node] += normals[i] * areas[i]
    
    # Normalize
    norms = np.linalg.norm(vertex_normals, axis=1, keepdims=True)
    norms = np.maximum(norms, 1e-10)
    vertex_normals = vertex_normals / norms
    
    return vertex_normals


def compute_wall_stress_fixed(coords, elements, wall_thickness, surfaces, 
                              tags=None, pressure=LV_PRESSURE_KPA):
    """
    Wall stress from modified Law of Laplace.
    
    σ = P × r_local / (2 × h) × curvature_factor
    
    Uses GEOMETRIC wall thickness (not from Laplace gradient).
    """
    n_elems = len(elements)
    centroids = np.mean(coords[elements], axis=1)
    
    # Estimate local radius at each z-level
    z_vals = centroids[:, 2]
    local_radius = np.zeros(n_elems)
    
    n_slices = 20
    z_min, z_max = z_vals.min(), z_vals.max()
    z_range = z_max - z_min
    
    for i in range(n_slices):
        z_lo = z_min + i * z_range / n_slices
        z_hi = z_min + (i + 1) * z_range / n_slices
        
        mask = (z_vals >= z_lo) & (z_vals < z_hi)
        if np.sum(mask) < 10:
            continue
        
        level_centroids = centroids[mask, :2]
        center = np.median(level_centroids, axis=0)
        radii = np.linalg.norm(level_centroids - center, axis=1)
        mean_radius = np.mean(radii)
        
        local_radius[mask] = mean_radius
    
    # Fill zeros with mean
    local_radius[local_radius == 0] = np.mean(local_radius[local_radius > 0])
    
    # Curvature factor (simplified - based on wall thickness)
    # Thinner walls have higher curvature effect
    curvature_factor = 1.0 + 0.3 * (10.0 - np.clip(wall_thickness, 3, 12)) / 7.0
    curvature_factor = np.clip(curvature_factor, 1.0, 1.5)
    
    # Wall stress: σ = P × r / (2h) × curvature_factor
    wall_stress = (pressure * local_radius) / (2 * wall_thickness) * curvature_factor
    
    # Modify for tissue type
    if tags is not None:
        wall_stress[tags == TAG_INFARCT] *= 0.7  # Scar is stiffer
        wall_stress[tags == TAG_BORDER] *= 1.5   # Stress concentration
    
    print(f"      Wall thickness: {wall_thickness.min():.2f} - {wall_thickness.max():.2f} mm")
    print(f"      Local radius: {local_radius.min():.2f} - {local_radius.max():.2f} mm")
    print(f"      Wall stress: {wall_stress.min():.2f} - {wall_stress.max():.2f} kPa")
    
    return {
        'wall_stress': wall_stress,
        'wall_thickness': wall_thickness,
        'local_radius': local_radius,
        'curvature_factor': curvature_factor,
    }


# GEODESIC DISTANCE (HEAT METHOD)

def build_mass_matrix(coords, elements):
    """Build lumped mass matrix"""
    n_nodes = len(coords)
    M = np.zeros(n_nodes)
    
    for elem in elements:
        X = coords[elem]
        J = np.array([X[1] - X[0], X[2] - X[0], X[3] - X[0]]).T
        vol = abs(np.linalg.det(J)) / 6.0
        
        for node in elem:
            M[node] += vol / 4.0
    
    return diags(M)


def compute_geodesic_heat_method(coords, elements, source_nodes, L, t_factor=1.0):
    """Compute geodesic distance via Heat Method (Crane et al. 2013)"""
    n_nodes = len(coords)
    n_elems = len(elements)
    
    M = build_mass_matrix(coords, elements)
    
    # Time step from mean edge length
    edge_lengths = []
    sample_elems = elements[:min(1000, n_elems)]
    for elem in sample_elems:
        for i in range(4):
            for j in range(i+1, 4):
                edge_lengths.append(np.linalg.norm(coords[elem[i]] - coords[elem[j]]))
    h = np.mean(edge_lengths)
    t = t_factor * h * h
    
    # Step 1: Solve heat equation
    A = M - t * L
    b = np.zeros(n_nodes)
    for node in source_nodes:
        b[node] = 1.0
    
    u = spsolve(A.tocsr(), b)
    
    # Step 2: Normalized gradient
    X = np.zeros((n_elems, 3))
    
    for i, elem in enumerate(elements):
        verts = coords[elem]
        u_elem = u[elem]
        
        J = np.array([verts[1] - verts[0], verts[2] - verts[0], verts[3] - verts[0]]).T
        detJ = np.linalg.det(J)
        
        if abs(detJ) < 1e-15:
            continue
        
        Jinv = np.linalg.inv(J)
        du_dxi = np.array([u_elem[1] - u_elem[0], u_elem[2] - u_elem[0], u_elem[3] - u_elem[0]])
        grad_u = Jinv @ du_dxi
        
        norm = np.linalg.norm(grad_u)
        if norm > 1e-10:
            X[i] = -grad_u / norm
    
    # Step 3: Divergence and Poisson
    div_X = np.zeros(n_nodes)
    
    for i, elem in enumerate(elements):
        verts = coords[elem]
        J = np.array([verts[1] - verts[0], verts[2] - verts[0], verts[3] - verts[0]]).T
        detJ = np.linalg.det(J)
        
        if abs(detJ) < 1e-15:
            continue
        
        vol = abs(detJ) / 6.0
        Jinv = np.linalg.inv(J)
        
        dN = np.zeros((4, 3))
        dN[0] = -Jinv.sum(axis=1)
        dN[1] = Jinv[:, 0]
        dN[2] = Jinv[:, 1]
        dN[3] = Jinv[:, 2]
        
        for j in range(4):
            div_X[elem[j]] += vol * np.dot(dN[j], X[i])
    
    L_mod = L.tolil()
    ref_node = source_nodes[0]
    L_mod[ref_node, :] = 0
    L_mod[ref_node, ref_node] = 1
    div_X[ref_node] = 0
    
    phi = spsolve(L_mod.tocsr(), div_X)
    phi = phi - phi[source_nodes].min()
    
    return phi


# INJECTION SITE OPTIMIZATION

def optimize_injection_sites(coords, elements, tags, stress_data, phi_trans,
                             geodesic_to_border, centroids, n_sites=5):
    """Optimize injection sites using geodesic + stress"""
    n_elems = len(elements)
    
    phi_elem = np.mean(phi_trans[elements], axis=1)
    geodesic_elem = np.mean(geodesic_to_border[elements], axis=1)
    stress_norm = stress_data['wall_stress'] / np.median(stress_data['wall_stress'])
    
    # Scoring
    scores = np.zeros(n_elems)
    
    for i in range(n_elems):
        if tags[i] == TAG_INFARCT:
            scores[i] = -np.inf
            continue
        
        dist_score = 1.0 / (geodesic_elem[i] + 1.0)
        stress_score = stress_norm[i]
        mid_wall_score = 1.0 - 2.0 * abs(phi_elem[i] - 0.5)
        border_bonus = 2.0 if tags[i] == TAG_BORDER else 1.0
        
        scores[i] = 0.3 * dist_score + 0.3 * stress_score + 0.2 * mid_wall_score + 0.2 * border_bonus
    
    # Select with spatial diversity
    selected_sites = []
    sorted_indices = np.argsort(scores)[::-1]
    
    for idx in sorted_indices:
        if len(selected_sites) >= n_sites:
            break
        if scores[idx] == -np.inf:
            continue
        
        centroid = centroids[idx]
        too_close = False
        for site_idx in selected_sites:
            if np.linalg.norm(centroid - centroids[site_idx]) < MIN_SITE_SEPARATION_MM:
                too_close = True
                break
        
        if not too_close:
            selected_sites.append(idx)
    
    injection_sites = []
    for idx in selected_sites:
        injection_sites.append({
            'element_id': int(idx),
            'coordinates': centroids[idx].tolist(),
            'score': float(scores[idx]),
            'transmural_position': float(phi_elem[idx]),
            'wall_stress_kPa': float(stress_data['wall_stress'][idx]),
            'geodesic_distance': float(geodesic_elem[idx]),
            'tissue_type': 'border' if tags[idx] == TAG_BORDER else 'healthy'
        })
    
    return injection_sites


# OUTPUT FUNCTIONS

def write_vtk_complete(filepath, coords, elements, scalars, vectors):
    """Write VTK with all fields"""
    n_nodes, n_elems = len(coords), len(elements)
    
    with open(filepath, 'w') as f:
        f.write("# vtk DataFile Version 3.0\n")
        f.write("Laplace-Dirichlet Analysis (Fixed WT)\n")
        f.write("ASCII\nDATASET UNSTRUCTURED_GRID\n")
        
        f.write(f"POINTS {n_nodes} float\n")
        for c in coords:
            f.write(f"{c[0]:.6f} {c[1]:.6f} {c[2]:.6f}\n")
        
        f.write(f"\nCELLS {n_elems} {n_elems * 5}\n")
        for e in elements:
            f.write(f"4 {e[0]} {e[1]} {e[2]} {e[3]}\n")
        
        f.write(f"\nCELL_TYPES {n_elems}\n" + "10\n" * n_elems)
        
        # Point data
        point_scalars = {k: v for k, v in scalars.items() if len(v) == n_nodes}
        if point_scalars:
            f.write(f"\nPOINT_DATA {n_nodes}\n")
            for name, data in point_scalars.items():
                f.write(f"SCALARS {name} float 1\nLOOKUP_TABLE default\n")
                for val in data:
                    f.write(f"{float(val):.6f}\n")
        
        # Cell data
        cell_scalars = {k: v for k, v in scalars.items() if len(v) == n_elems}
        if cell_scalars or vectors:
            f.write(f"\nCELL_DATA {n_elems}\n")
            
            for name, data in cell_scalars.items():
                f.write(f"SCALARS {name} float 1\nLOOKUP_TABLE default\n")
                for val in data:
                    f.write(f"{float(val):.6f}\n")
            
            for name, data in vectors.items():
                if len(data) == n_elems:
                    f.write(f"\nVECTORS {name} float\n")
                    for v in data:
                        f.write(f"{v[0]:.6f} {v[1]:.6f} {v[2]:.6f}\n")


def write_lon_file(filepath, fibers, sheets):
    """Write CARP-format fiber file"""
    with open(filepath, 'w') as f:
        f.write("2\n")
        for fiber, sheet in zip(fibers, sheets):
            f.write(f"{fiber[0]:.6f} {fiber[1]:.6f} {fiber[2]:.6f} "
                   f"{sheet[0]:.6f} {sheet[1]:.6f} {sheet[2]:.6f}\n")


# MAIN PIPELINE

def process_patient(patient_id):
    """Process single patient"""
    print(f"PROCESSING: {patient_id}")
    
    results = {'patient_id': patient_id}
    
    # Load mesh
    print("\n  Loading mesh")
    coords, elements = load_mesh_fast(patient_id, BASE_DIR)
    n_nodes, n_elems = len(coords), len(elements)
    print(f"      {n_nodes:,} nodes, {n_elems:,} elements")
    
    centroids = np.mean(coords[elements], axis=1)
    
    # Load classification
    tags = load_classification(patient_id, BASE_DIR)
    if tags is None:
        print("      No classification found - using all healthy")
        tags = np.ones(n_elems, dtype=np.int32)
    else:
        print(f"      Classification: {np.sum(tags==TAG_INFARCT)} infarct, {np.sum(tags==TAG_BORDER)} border")
    
    # Extract surfaces
    print("\n  [1] LAPLACE-DIRICHLET TRANSMURAL COORDINATE")
    print("      Extracting surfaces")
    surfaces = extract_surfaces_fast(coords, elements)
    print(f"      Endo: {len(surfaces['endo_nodes'])} nodes, Epi: {len(surfaces['epi_nodes'])} nodes")
    
    # Solve Laplace
    phi = solve_laplace_dirichlet(coords, elements, surfaces)
    print(f"      φ range: [{phi.min():.4f}, {phi.max():.4f}]")
    
    # Compute gradient
    grad_phi = compute_phi_gradient_fast(coords, elements, phi)
    
    # FIXED: Geometric wall thickness
    print("\n  [2] GEOMETRIC WALL THICKNESS (FIXED)")
    wall_thickness, endo_dist, epi_dist, transmural_dir = compute_wall_thickness_geometric(
        coords, elements, surfaces, phi
    )
    print(f"      WT range: [{wall_thickness.min():.2f}, {wall_thickness.max():.2f}] mm")
    print(f"      WT mean: {wall_thickness.mean():.2f} mm")
    
    # Fiber reconstruction
    print("\n  [3] HELICAL FIBER RECONSTRUCTION")
    fibers, sheets, fiber_angles, coord_system = reconstruct_helical_fibers(
        coords, elements, phi, grad_phi, tags
    )
    
    # Wall stress (using geometric WT)
    print("\n  [4] WALL STRESS (MODIFIED LAPLACE)")
    stress_data = compute_wall_stress_fixed(
        coords, elements, wall_thickness, surfaces, tags, LV_PRESSURE_KPA
    )
    
    # Geodesic optimization
    print("\n  [5] GEODESIC INJECTION OPTIMIZATION")
    border_elements = np.where(tags == TAG_BORDER)[0]
    
    if len(border_elements) > 0:
        border_nodes = list(set(elements[border_elements[:100]].flatten()))
        print("      Computing geodesic distances")
        L = build_fem_laplacian_fast(coords, elements)
        geodesic_to_border = compute_geodesic_heat_method(coords, elements, border_nodes, L)
        print(f"      Geodesic range: [{geodesic_to_border.min():.2f}, {geodesic_to_border.max():.2f}]")
    else:
        print("      No border zone - using endo distance")
        geodesic_to_border = endo_dist[np.arange(n_elems)]  # Use per-element endo distance
        # Interpolate to nodes
        geodesic_to_border = np.zeros(n_nodes)
        for i, elem in enumerate(elements):
            for node in elem:
                geodesic_to_border[node] = endo_dist[i]
    
    print("      Optimizing injection sites")
    injection_sites = optimize_injection_sites(
        coords, elements, tags, stress_data, phi,
        geodesic_to_border, centroids, N_INJECTION_SITES
    )
    
    print(f"      Selected {len(injection_sites)} sites")
    for site in injection_sites:
        print(f"        Element {site['element_id']}: score={site['score']:.3f}, "
              f"stress={site['wall_stress_kPa']:.1f}kPa")
    
    # Save outputs
    print("\n  [6] SAVING OUTPUTS")
    out_dir = os.path.join(OUTPUT_DIR, patient_id)
    os.makedirs(out_dir, exist_ok=True)
    
    phi_elem = np.mean(phi[elements], axis=1)
    
    write_vtk_complete(
        os.path.join(out_dir, f"{patient_id}_laplace_fixed.vtk"),
        coords, elements,
        scalars={
            'TransmuralPhi': phi,
            'TransmuralPhi_elem': phi_elem,
            'TissueType': tags.astype(float),
            'FiberAngle_deg': fiber_angles,
            'WallThickness_mm': wall_thickness,
            'WallStress_kPa': stress_data['wall_stress'],
            'LocalRadius_mm': stress_data['local_radius'],
            'EndoDist_mm': endo_dist,
            'EpiDist_mm': epi_dist,
        },
        vectors={
            'FiberDirection': fibers,
            'SheetDirection': sheets,
            'TransmuralDirection': transmural_dir,
        }
    )
    
    write_lon_file(os.path.join(out_dir, f"{patient_id}_reconstructed.lon"), fibers, sheets)
    
    with open(os.path.join(out_dir, f"{patient_id}_injection_sites.json"), 'w') as f:
        json.dump(injection_sites, f, indent=2)
    
    summary = {
        'patient_id': patient_id,
        'timestamp': datetime.now().isoformat(),
        'mesh': {'n_nodes': n_nodes, 'n_elements': n_elems},
        'transmural': {'phi_range': [float(phi.min()), float(phi.max())]},
        'wall_thickness': {
            'method': 'GEOMETRIC (fixed)',
            'range_mm': [float(wall_thickness.min()), float(wall_thickness.max())],
            'mean_mm': float(wall_thickness.mean()),
        },
        'fibers': {'angle_range_deg': [float(fiber_angles.min()), float(fiber_angles.max())]},
        'wall_stress': {
            'range_kPa': [float(stress_data['wall_stress'].min()), float(stress_data['wall_stress'].max())],
            'mean_kPa': float(stress_data['wall_stress'].mean()),
        },
        'injection_sites': injection_sites,
    }
    
    with open(os.path.join(out_dir, f"{patient_id}_summary.json"), 'w') as f:
        json.dump(summary, f, indent=2, default=str)
    
    print(f"\n  Outputs saved to: {out_dir}")
    
    return summary


def process_patient_wrapper(patient_id):
    """Wrapper for parallel processing"""
    try:
        return process_patient(patient_id)
    except Exception as e:
        import traceback
        traceback.print_exc()
        return {'patient_id': patient_id, 'status': 'FAILED', 'error': str(e)}


def main():
    """Main entry with parallel processing"""
    print("LAPLACE-DIRICHLET CARDIAC ANALYSIS")
    
    print(f"\n  Using {N_CPUS} CPUs for parallel processing")
    print(f"\n  FIXES APPLIED:")
    print(f"  ✓ Wall thickness: GEOMETRIC (not h=1/|∇φ|)")
    print(f"  ✓ Circumferential smoothing to remove stripe artifacts")
    print(f"  ✓ Proper parallel processing")
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # Set thread counts
    import os as os_env
    threads_per_patient = max(1, N_CPUS // len(PATIENT_IDS))
    os_env.environ['OMP_NUM_THREADS'] = str(threads_per_patient)
    os_env.environ['MKL_NUM_THREADS'] = str(threads_per_patient)
    os_env.environ['OPENBLAS_NUM_THREADS'] = str(threads_per_patient)
    
    print(f"\n  Processing {len(PATIENT_IDS)} patients")
    
    # Parallel processing
    n_workers = min(N_CPUS, len(PATIENT_IDS))
    
    with Pool(processes=n_workers) as pool:
        all_results = pool.map(process_patient_wrapper, PATIENT_IDS)
    
    # Summary
    print("SUMMARY")
    
    print(f"\n{'Patient':<15} {'WT (mm)':<20} {'Stress (kPa)':<20} {'Sites':<10}")
    
    for r in all_results:
        if 'status' not in r:
            wt = r.get('wall_thickness', {})
            stress = r.get('wall_stress', {})
            sites = len(r.get('injection_sites', []))
            print(f"{r['patient_id']:<15} "
                  f"{wt.get('range_mm', [0,0])[0]:.1f}-{wt.get('range_mm', [0,0])[1]:.1f}          "
                  f"{stress.get('range_kPa', [0,0])[0]:.1f}-{stress.get('range_kPa', [0,0])[1]:.1f}           "
                  f"{sites}")
        else:
            print(f"{r['patient_id']:<15} FAILED: {r.get('error', 'Unknown')[:40]}")
    
    with open(os.path.join(OUTPUT_DIR, "summary.json"), 'w') as f:
        json.dump(all_results, f, indent=2, default=str)
    
    print(f"\nResults saved to: {OUTPUT_DIR}")
    
    return all_results


if __name__ == "__main__":
    results = main()

In [9]:
#!/usr/bin/env python3
"""
LAPLACE-DIRICHLET CARDIAC ANALYSIS FRAMEWORK


Complete computational pipeline for cardiac mechanics analysis:

1. TRANSMURAL COORDINATE
   - Solve Laplace equation ∇²φ = 0 with Dirichlet boundary conditions
   - φ ∈ [0,1] defines position through myocardial wall

2. MYOFIBER RECONSTRUCTION  
   - Helical angle rotation: α(φ) = α_endo + (α_epi - α_endo) × φ
   - Fiber direction: f = cos(α)·e_c + sin(α)·e_l

3. WALL STRESS COMPUTATION
   - Modified Law of Laplace with local curvature tensor
   - Shape operator computation for principal curvatures κ₁, κ₂
   - σ = Pr/(2h) × f(κ₁, κ₂)

4. INJECTION SITE OPTIMIZATION
   - Geodesic distance computation via Heat Method
   - Multi-objective optimization: minimize geodesic, maximize stress reduction

References:
- Bishop et al. Am J Physiol Heart Circ Physiol 2010 (Laplace-Dirichlet)
- Streeter et al. Circ Res 1969 (Helical fiber architecture)
- Crane et al. ACM TOG 2013 (Heat Method for geodesics)
- Zhong et al. Int J Cardiol 2008 (Wall stress analysis)

"""

import numpy as np
from scipy.sparse import lil_matrix, csr_matrix, diags
from scipy.sparse.linalg import spsolve
from scipy.spatial import cKDTree
from collections import defaultdict
import os
import json
from datetime import datetime
from multiprocessing import Pool
import warnings
warnings.filterwarnings('ignore')


N_CPUS = 44

PATIENT_IDS = [
    "SCD0000101", "SCD0000201", "SCD0000301", "SCD0000401",
    "SCD0000601", "SCD0000701", "SCD0000801", "SCD0001001",
    "SCD0001101", "SCD0001201"
]

BASE_DIR = "/home/shadeform/SCD_MODELS"
OUTPUT_DIR = "/home/shadeform/SCD_MODELS/laplace_complete_v2"

# Fiber angle parameters (Streeter et al. 1969)
FIBER_ANGLE_ENDO_DEG = 60.0
FIBER_ANGLE_EPI_DEG = -60.0

# Biomechanical parameters
LV_PRESSURE_KPA = 16.0
INJECTION_STIFFENING_FACTOR = 2.0

# Optimization parameters
N_INJECTION_SITES = 5
MIN_SITE_SEPARATION_MM = 15.0

# Tissue classification tags
TAG_HEALTHY = 1
TAG_BORDER = 2
TAG_INFARCT = 3


# MESH I/O

def load_mesh(patient_id, base_dir):
    """Load tetrahedral mesh from CARP format files."""
    pts = f"{base_dir}/simulation_ready/{patient_id}/{patient_id}_tet.pts"
    elem = f"{base_dir}/simulation_ready/{patient_id}/{patient_id}_tet.elem"
    
    with open(pts) as f:
        n = int(f.readline())
        coords = np.array([[float(x) for x in f.readline().split()[:3]] for _ in range(n)])
    
    with open(elem) as f:
        n = int(f.readline())
        elements = np.array([[int(x) for x in f.readline().split()[1:5]] for _ in range(n)], dtype=np.int32)
    
    return coords, elements


def load_tags(patient_id, base_dir):
    """Load tissue classification tags."""
    path = f"{base_dir}/infarct_results_comprehensive/{patient_id}/{patient_id}_tagged.elem"
    if not os.path.exists(path):
        return None
    with open(path) as f:
        n = int(f.readline())
        return np.array([int(f.readline().split()[5]) for _ in range(n)], dtype=np.int32)


# SURFACE EXTRACTION

def extract_surfaces(coords, elements):
    """Extract endocardial and epicardial surface meshes."""
    face_count = defaultdict(list)
    for ei, elem in enumerate(elements):
        for fi in [[0,1,2], [0,1,3], [0,2,3], [1,2,3]]:
            face_count[tuple(sorted(elem[fi]))].append(ei)
    
    boundary_faces = [(f, e[0]) for f, e in face_count.items() if len(e) == 1]
    boundary_nodes = np.array(list(set(n for f, _ in boundary_faces for n in f)))
    
    z = coords[:, 2]
    z_min, z_max = z.min(), z.max()
    endo, epi = set(), set()
    endo_f, epi_f = [], []
    
    for i in range(25):
        lo, hi = z_min + i*(z_max-z_min)/25, z_min + (i+1)*(z_max-z_min)/25
        mask = (coords[boundary_nodes, 2] >= lo) & (coords[boundary_nodes, 2] < hi)
        nodes = boundary_nodes[mask]
        if len(nodes) < 20:
            continue
        
        xy = coords[nodes, :2]
        center = np.median(xy, axis=0)
        r = np.linalg.norm(xy - center, axis=1)
        r30, r70 = np.percentile(r, [30, 70])
        
        endo.update(nodes[r < r30])
        epi.update(nodes[r > r70])
    
    for face, _ in boundary_faces:
        if all(n in endo for n in face):
            endo_f.append(list(face))
        elif all(n in epi for n in face):
            epi_f.append(list(face))
    
    return {
        'endo_nodes': np.array(list(endo)),
        'epi_nodes': np.array(list(epi)),
        'endo_faces': np.array(endo_f) if endo_f else np.zeros((0, 3), dtype=int),
        'epi_faces': np.array(epi_f) if epi_f else np.zeros((0, 3), dtype=int),
    }


# LAPLACE-DIRICHLET TRANSMURAL COORDINATE

def build_laplacian(coords, elements):
    """
    Build FEM Laplacian stiffness matrix.
    
    For linear tetrahedral elements:
    K_ij = ∫_Ω ∇N_i · ∇N_j dV
    """
    n = len(coords)
    K = lil_matrix((n, n))
    
    for elem in elements:
        X = coords[elem]
        J = np.array([X[1] - X[0], X[2] - X[0], X[3] - X[0]]).T
        det = np.linalg.det(J)
        if abs(det) < 1e-15:
            continue
        
        vol = abs(det) / 6.0
        Jinv = np.linalg.inv(J)
        
        # Shape function gradients
        dN = np.zeros((4, 3))
        dN[0] = -Jinv.sum(axis=1)
        dN[1:4] = Jinv.T
        
        # Element stiffness matrix
        Ke = vol * (dN @ dN.T)
        for i in range(4):
            for j in range(4):
                K[elem[i], elem[j]] += Ke[i, j]
    
    return K.tocsr()


def solve_laplace_dirichlet(coords, elements, surfaces):
    """
    Solve Laplace equation for transmural coordinate.
    
    ∇²φ = 0 in Ω
    φ = 0 on Γ_endo (endocardium)
    φ = 1 on Γ_epi (epicardium)
    
    Returns transmural coordinate φ(x) ∈ [0,1].
    """
    n = len(coords)
    
    print("      Building FEM Laplacian")
    K = build_laplacian(coords, elements)
    
    endo, epi = set(surfaces['endo_nodes']), set(surfaces['epi_nodes'])
    print(f"      Dirichlet BCs: {len(endo)} endo, {len(epi)} epi")
    
    # Apply Dirichlet BCs via penalty method
    K_mod = K.tolil()
    rhs = np.zeros(n)
    penalty = 1e12
    
    for node in endo:
        K_mod[node, :] = 0
        K_mod[node, node] = penalty
        rhs[node] = 0.0 * penalty
    
    for node in epi:
        K_mod[node, :] = 0
        K_mod[node, node] = penalty
        rhs[node] = 1.0 * penalty
    
    print("      Solving Laplace equation")
    phi = spsolve(K_mod.tocsr(), rhs)
    return np.clip(phi, 0, 1)


def compute_gradient(coords, elements, phi):
    """Compute gradient ∇φ at element centroids."""
    grad = np.zeros((len(elements), 3))
    for i, elem in enumerate(elements):
        X = coords[elem]
        J = np.array([X[1]-X[0], X[2]-X[0], X[3]-X[0]]).T
        det = np.linalg.det(J)
        if abs(det) < 1e-15:
            continue
        Jinv = np.linalg.inv(J)
        dp = np.array([phi[elem[j]] - phi[elem[0]] for j in [1,2,3]])
        grad[i] = Jinv @ dp
    return grad


# HELICAL FIBER RECONSTRUCTION

def reconstruct_helical_fibers(coords, elements, phi, grad_phi, tags=None):
    """
    Reconstruct myofiber orientations using helical angle rotation.
    
    The fiber angle varies linearly through the wall:
    α(φ) = α_endo + (α_epi - α_endo) × φ
    
    Fiber direction in local coordinates:
    f = cos(α)·e_c + sin(α)·e_l
    
    Where:
    - e_t: transmural direction (∇φ/|∇φ|)
    - e_l: longitudinal direction (apex to base)
    - e_c: circumferential direction (e_l × e_t)
    
    In infarct regions, fibers are modified:
    - Core scar: α = 0° (circumferential, no rotation)
    - Border zone: α reduced by 50%
    """
    n = len(elements)
    
    # Transmural direction e_t = ∇φ/|∇φ|
    norm = np.linalg.norm(grad_phi, axis=1, keepdims=True)
    norm = np.maximum(norm, 1e-10)
    e_t = grad_phi / norm
    
    # Longitudinal direction (orthogonalized to e_t)
    e_l = np.zeros((n, 3))
    e_l[:, 2] = 1.0
    proj = np.sum(e_l * e_t, axis=1, keepdims=True)
    e_l = e_l - proj * e_t
    e_l = e_l / (np.linalg.norm(e_l, axis=1, keepdims=True) + 1e-10)
    
    # Circumferential direction e_c = e_l × e_t
    e_c = np.cross(e_l, e_t)
    e_c = e_c / (np.linalg.norm(e_c, axis=1, keepdims=True) + 1e-10)
    
    # Fiber angle: α(φ) = 60° - 120°×φ
    phi_elem = np.mean(phi[elements], axis=1)
    alpha = np.radians(FIBER_ANGLE_ENDO_DEG + (FIBER_ANGLE_EPI_DEG - FIBER_ANGLE_ENDO_DEG) * phi_elem)
    
    # Modify in infarct regions
    if tags is not None:
        alpha[tags == TAG_INFARCT] = 0.0
        alpha[tags == TAG_BORDER] *= 0.5
    
    # Fiber direction: f = cos(α)·e_c + sin(α)·e_l
    cos_a, sin_a = np.cos(alpha)[:, np.newaxis], np.sin(alpha)[:, np.newaxis]
    fibers = cos_a * e_c + sin_a * e_l
    fibers = fibers / (np.linalg.norm(fibers, axis=1, keepdims=True) + 1e-10)
    
    # Sheet direction: s = f × e_t
    sheets = np.cross(fibers, e_t)
    sheets = sheets / (np.linalg.norm(sheets, axis=1, keepdims=True) + 1e-10)
    
    return fibers, sheets, np.degrees(alpha), (e_t, e_l, e_c)


# CURVATURE TENSOR (SHAPE OPERATOR)

def compute_vertex_normals(coords, faces):
    """Compute area-weighted vertex normals from surface mesh."""
    n = len(coords)
    normals = np.zeros((n, 3))
    
    for face in faces:
        v0, v1, v2 = coords[face]
        fn = np.cross(v1 - v0, v2 - v0)
        area = np.linalg.norm(fn) / 2
        if area > 1e-10:
            fn = fn / (2 * area)
            for node in face:
                normals[node] += fn * area
    
    norms = np.linalg.norm(normals, axis=1, keepdims=True)
    return normals / np.maximum(norms, 1e-10)


def compute_shape_operator(coords, faces, vertex_normals):
    """
    Compute the shape operator (Weingarten map) at surface vertices.
    
    The shape operator S relates normal variation to surface position:
    S = -∂n/∂x
    
    Principal curvatures κ₁, κ₂ are the eigenvalues of S.
    Mean curvature: H = (κ₁ + κ₂)/2
    Gaussian curvature: K = κ₁ × κ₂
    """
    n = len(coords)
    
    node_faces = defaultdict(list)
    for fi, face in enumerate(faces):
        for node in face:
            node_faces[node].append(fi)
    
    kappa_1 = np.zeros(n)
    kappa_2 = np.zeros(n)
    mean_curv = np.zeros(n)
    gauss_curv = np.zeros(n)
    
    for node in node_faces.keys():
        if len(node_faces[node]) < 3:
            continue
        
        p = coords[node]
        normal = vertex_normals[node]
        
        if np.linalg.norm(normal) < 0.1:
            continue
        
        # Find one-ring neighbors
        neighbors = set()
        for fi in node_faces[node]:
            neighbors.update(faces[fi])
        neighbors.discard(node)
        
        if len(neighbors) < 3:
            continue
        
        neighbor_list = list(neighbors)
        diffs = coords[neighbor_list] - p
        neighbor_normals = vertex_normals[neighbor_list]
        
        # Build local tangent basis
        t1 = diffs[0] - np.dot(diffs[0], normal) * normal
        if np.linalg.norm(t1) < 1e-10:
            continue
        t1 = t1 / np.linalg.norm(t1)
        t2 = np.cross(normal, t1)
        
        # Fit shape operator via least squares: dn = -S @ dx
        A, b = [], []
        for i in range(len(neighbor_list)):
            dx = diffs[i]
            dn = neighbor_normals[i] - normal
            
            dx_t = np.array([np.dot(dx, t1), np.dot(dx, t2)])
            dn_t = np.array([np.dot(dn, t1), np.dot(dn, t2)])
            
            if np.linalg.norm(dx_t) > 1e-10:
                A.append([dx_t[0], dx_t[1], 0, 0])
                A.append([0, 0, dx_t[0], dx_t[1]])
                b.append(-dn_t[0])
                b.append(-dn_t[1])
        
        if len(A) < 4:
            continue
        
        try:
            S_flat, _, _, _ = np.linalg.lstsq(np.array(A), np.array(b), rcond=None)
            S = np.array([[S_flat[0], S_flat[1]], [S_flat[2], S_flat[3]]])
            S = (S + S.T) / 2  # Symmetrize
            
            # Principal curvatures = eigenvalues
            eig = np.linalg.eigvalsh(S)
            kappa_1[node] = np.max(eig)
            kappa_2[node] = np.min(eig)
            mean_curv[node] = (kappa_1[node] + kappa_2[node]) / 2
            gauss_curv[node] = kappa_1[node] * kappa_2[node]
        except:
            pass
    
    return {'kappa_1': kappa_1, 'kappa_2': kappa_2, 
            'mean_curvature': mean_curv, 'gaussian_curvature': gauss_curv}


# WALL STRESS COMPUTATION

def compute_wall_stress(coords, elements, surfaces, tags=None):
    """
    Compute wall stress using modified Law of Laplace with curvature tensor.
    
    Classical Laplace law for thin-walled pressure vessel:
    σ = P × r / (2 × h)
    
    Modified with local curvature:
    σ = P × r_local / (2 × h) × f(κ₁, κ₂)
    
    Where:
    - r_local = 1/|H| (local radius from mean curvature)
    - h = wall thickness (geometric)
    - f(κ₁, κ₂) = 1 + |κ₁/κ₂ - 1| × 0.5 (curvature anisotropy factor)
    """
    n_elems = len(elements)
    centroids = np.mean(coords[elements], axis=1)
    
    # Geometric wall thickness
    print("      Computing geometric wall thickness")
    endo_tree = cKDTree(coords[surfaces['endo_nodes']])
    epi_tree = cKDTree(coords[surfaces['epi_nodes']])
    
    endo_dist = np.array([endo_tree.query(c)[0] for c in centroids])
    epi_dist = np.array([epi_tree.query(c)[0] for c in centroids])
    wall_thickness = np.clip(endo_dist + epi_dist, 2.0, 25.0)
    
    # Shape operator for curvature tensor
    print("      Computing shape operator (curvature tensor)")
    if len(surfaces['endo_faces']) > 0:
        vertex_normals = compute_vertex_normals(coords, surfaces['endo_faces'])
        curvature = compute_shape_operator(coords, surfaces['endo_faces'], vertex_normals)
    else:
        curvature = {'kappa_1': np.zeros(len(coords)), 'kappa_2': np.zeros(len(coords)),
                     'mean_curvature': np.zeros(len(coords))}
    
    # Interpolate curvature to elements
    kappa_1 = np.array([np.mean(curvature['kappa_1'][e]) for e in elements])
    kappa_2 = np.array([np.mean(curvature['kappa_2'][e]) for e in elements])
    mean_curv = np.array([np.mean(curvature['mean_curvature'][e]) for e in elements])
    
    # Local radius = 1/|H|
    local_radius = 1.0 / np.maximum(np.abs(mean_curv), 1e-6)
    local_radius = np.clip(local_radius, 10, 200)
    
    # Fallback for elements with poor curvature estimates
    z = centroids[:, 2]
    for i in range(n_elems):
        if local_radius[i] > 150:
            mask = np.abs(z - z[i]) < 5.0
            if np.sum(mask) > 10:
                xy = centroids[mask, :2]
                center = np.mean(xy, axis=0)
                local_radius[i] = np.mean(np.linalg.norm(xy - center, axis=1))
    
    # Curvature anisotropy factor: f(κ₁, κ₂) = 1 + |κ₁/κ₂ - 1| × 0.5
    k2_safe = np.where(np.abs(kappa_2) > 1e-8, kappa_2, 1e-8)
    anisotropy = np.abs(kappa_1 / k2_safe - 1.0)
    curv_factor = np.clip(1.0 + anisotropy * 0.5, 1.0, 2.5)
    
    # Modified Laplace: σ = P × r / (2h) × f(κ)
    stress = (LV_PRESSURE_KPA * local_radius) / (2 * wall_thickness) * curv_factor
    
    # Tissue-specific modifications
    if tags is not None:
        stress[tags == TAG_INFARCT] *= 0.7  # Scar is stiffer
        stress[tags == TAG_BORDER] *= 1.5   # Stress concentration at border
    
    print(f"      κ₁: [{kappa_1.min():.4f}, {kappa_1.max():.4f}]")
    print(f"      κ₂: [{kappa_2.min():.4f}, {kappa_2.max():.4f}]")
    print(f"      Wall stress: {stress.min():.2f} - {stress.max():.2f} kPa")
    
    return {
        'wall_stress': stress, 'wall_thickness': wall_thickness,
        'local_radius': local_radius, 'curvature_factor': curv_factor,
        'kappa_1': kappa_1, 'kappa_2': kappa_2, 'mean_curvature': mean_curv,
        'endo_dist': endo_dist, 'epi_dist': epi_dist,
    }


# GEODESIC DISTANCE (HEAT METHOD)

def compute_geodesic_distance(coords, elements, source_nodes, L):
    """
    Compute geodesic distances using the Heat Method (Crane et al. 2013).
    
    Algorithm:
    1. Solve heat equation: (M - tL)u = δ_source
    2. Normalize gradient: X = -∇u/|∇u|
    3. Solve Poisson equation: Lφ = ∇·X
    
    The solution φ gives approximate geodesic distances.
    """
    n_nodes, n_elems = len(coords), len(elements)
    
    # Lumped mass matrix
    M = np.zeros(n_nodes)
    for elem in elements:
        X = coords[elem]
        J = np.array([X[1]-X[0], X[2]-X[0], X[3]-X[0]]).T
        vol = abs(np.linalg.det(J)) / 6.0
        for node in elem:
            M[node] += vol / 4.0
    M = diags(M)
    
    # Time step from mean edge length
    h = np.mean([np.linalg.norm(coords[e[i]] - coords[e[j]]) 
                 for e in elements[:1000] for i in range(4) for j in range(i+1, 4)])
    t = h * h
    
    # Step 1: Solve heat equation
    b = np.zeros(n_nodes)
    for node in source_nodes:
        b[node] = 1.0
    u = spsolve((M - t * L).tocsr(), b)
    
    # Step 2: Compute normalized gradient field
    X = np.zeros((n_elems, 3))
    for i, elem in enumerate(elements):
        verts = coords[elem]
        J = np.array([verts[1]-verts[0], verts[2]-verts[0], verts[3]-verts[0]]).T
        det = np.linalg.det(J)
        if abs(det) < 1e-15:
            continue
        Jinv = np.linalg.inv(J)
        du = np.array([u[elem[j]] - u[elem[0]] for j in [1,2,3]])
        grad = Jinv @ du
        norm = np.linalg.norm(grad)
        if norm > 1e-10:
            X[i] = -grad / norm
    
    # Step 3: Compute divergence and solve Poisson
    div = np.zeros(n_nodes)
    for i, elem in enumerate(elements):
        verts = coords[elem]
        J = np.array([verts[1]-verts[0], verts[2]-verts[0], verts[3]-verts[0]]).T
        det = np.linalg.det(J)
        if abs(det) < 1e-15:
            continue
        vol = abs(det) / 6.0
        Jinv = np.linalg.inv(J)
        dN = np.zeros((4, 3))
        dN[0] = -Jinv.sum(axis=1)
        dN[1:4] = Jinv.T
        for j in range(4):
            div[elem[j]] += vol * np.dot(dN[j], X[i])
    
    L_mod = L.tolil()
    L_mod[source_nodes[0], :] = 0
    L_mod[source_nodes[0], source_nodes[0]] = 1
    div[source_nodes[0]] = 0
    
    phi = spsolve(L_mod.tocsr(), div)
    return phi - phi[source_nodes].min()


# STRESS REDUCTION PREDICTION

def compute_stress_reduction(stress, injection_elements, stiffening=2.0):
    """
    Predict wall stress reduction from therapeutic injection.
    
    Model: Injection stiffens local tissue, reducing local strain
    and redistributing stress.
    
    Δσ = σ_before × (1 - 1/stiffening_factor)
    """
    stress_after = stress.copy()
    for idx in injection_elements:
        stress_after[idx] *= (1.0 / stiffening)
    
    reduction = stress - stress_after
    total = np.sum(reduction[reduction > 0])
    return reduction, total


# INJECTION SITE OPTIMIZATION

def optimize_injection_sites(coords, elements, tags, stress_data, phi, geodesic, centroids, n_sites=5):
    """
    Optimize injection sites using multi-objective optimization.
    
    Objectives:
    1. MINIMIZE geodesic distance to border zone
    2. MAXIMIZE predicted wall stress reduction
    
    Constraints:
    - Avoid core scar (no perfusion)
    - Minimum spatial separation between sites
    - Prefer mid-wall transmural position
    """
    n = len(elements)
    
    phi_elem = np.mean(phi[elements], axis=1)
    geo_elem = np.mean(geodesic[elements], axis=1) if len(geodesic) == len(coords) else geodesic
    
    # Compute stress reduction potential
    stress_red = np.zeros(n)
    for i in range(n):
        if tags[i] != TAG_INFARCT:
            _, red = compute_stress_reduction(stress_data['wall_stress'], [i], INJECTION_STIFFENING_FACTOR)
            stress_red[i] = red
    
    # Normalize metrics
    geo_norm = geo_elem / (np.max(geo_elem) + 1e-10)
    stress_norm = stress_red / (np.max(stress_red[stress_red > 0]) + 1e-10)
    
    # Multi-objective: minimize geodesic, maximize stress reduction
    objective = 0.5 * geo_norm - 0.5 * stress_norm
    
    # Additional terms
    midwall_penalty = 2.0 * np.abs(phi_elem - 0.5)
    border_bonus = np.where(tags == TAG_BORDER, -0.5, 0)
    
    objective = objective + 0.2 * midwall_penalty + 0.2 * border_bonus
    objective[tags == TAG_INFARCT] = np.inf
    
    # Greedy selection with spatial constraints
    selected = []
    for idx in np.argsort(objective):
        if len(selected) >= n_sites or not np.isfinite(objective[idx]):
            break
        if not any(np.linalg.norm(centroids[idx] - centroids[s]) < MIN_SITE_SEPARATION_MM for s in selected):
            selected.append(idx)
    
    _, total_red = compute_stress_reduction(stress_data['wall_stress'], selected, INJECTION_STIFFENING_FACTOR)
    
    sites = [{
        'element_id': int(idx),
        'coordinates': centroids[idx].tolist(),
        'geodesic_distance': float(geo_elem[idx]),
        'stress_reduction_kPa': float(stress_red[idx]),
        'transmural_position': float(phi_elem[idx]),
        'wall_stress_kPa': float(stress_data['wall_stress'][idx]),
        'tissue_type': 'border' if tags[idx] == TAG_BORDER else 'healthy'
    } for idx in selected]
    
    return sites, total_red


# OUTPUT FUNCTIONS

def write_vtk(path, coords, elements, scalars, vectors):
    """Write VTK unstructured grid file."""
    n_nodes, n_elems = len(coords), len(elements)
    with open(path, 'w') as f:
        f.write("# vtk DataFile Version 3.0\n")
        f.write("Laplace-Dirichlet Cardiac Analysis\n")
        f.write("ASCII\nDATASET UNSTRUCTURED_GRID\n")
        
        f.write(f"POINTS {n_nodes} float\n")
        for c in coords:
            f.write(f"{c[0]:.6f} {c[1]:.6f} {c[2]:.6f}\n")
        
        f.write(f"\nCELLS {n_elems} {n_elems*5}\n")
        for e in elements:
            f.write(f"4 {e[0]} {e[1]} {e[2]} {e[3]}\n")
        
        f.write(f"\nCELL_TYPES {n_elems}\n" + "10\n" * n_elems)
        
        pt = {k: v for k, v in scalars.items() if len(v) == n_nodes}
        if pt:
            f.write(f"\nPOINT_DATA {n_nodes}\n")
            for name, data in pt.items():
                f.write(f"SCALARS {name} float 1\nLOOKUP_TABLE default\n")
                for v in data:
                    f.write(f"{float(v):.6f}\n")
        
        cl = {k: v for k, v in scalars.items() if len(v) == n_elems}
        if cl or vectors:
            f.write(f"\nCELL_DATA {n_elems}\n")
            for name, data in cl.items():
                f.write(f"SCALARS {name} float 1\nLOOKUP_TABLE default\n")
                for v in data:
                    f.write(f"{float(v):.6f}\n")
            for name, data in vectors.items():
                if len(data) == n_elems:
                    f.write(f"\nVECTORS {name} float\n")
                    for v in data:
                        f.write(f"{v[0]:.6f} {v[1]:.6f} {v[2]:.6f}\n")


def write_lon(path, fibers, sheets):
    """Write CARP format fiber orientation file."""
    with open(path, 'w') as f:
        f.write("2\n")
        for fib, sht in zip(fibers, sheets):
            f.write(f"{fib[0]:.6f} {fib[1]:.6f} {fib[2]:.6f} {sht[0]:.6f} {sht[1]:.6f} {sht[2]:.6f}\n")


# PATIENT PROCESSING

def process_patient(patient_id):
    """Run complete analysis pipeline for one patient."""
    print(f"\nPROCESSING: {patient_id}")
    
    # Load mesh
    coords, elements = load_mesh(patient_id, BASE_DIR)
    n_nodes, n_elems = len(coords), len(elements)
    centroids = np.mean(coords[elements], axis=1)
    print(f"  Mesh: {n_nodes:,} nodes, {n_elems:,} elements")
    
    # Load tissue tags
    tags = load_tags(patient_id, BASE_DIR)
    if tags is None:
        tags = np.ones(n_elems, dtype=np.int32)
    else:
        print(f"  Tags: {np.sum(tags==TAG_INFARCT)} infarct, {np.sum(tags==TAG_BORDER)} border")
    
    # Extract surfaces
    print("\n  [1] TRANSMURAL COORDINATE")
    surfaces = extract_surfaces(coords, elements)
    print(f"      Endo: {len(surfaces['endo_nodes'])} nodes, Epi: {len(surfaces['epi_nodes'])} nodes")
    
    # Solve Laplace-Dirichlet
    phi = solve_laplace_dirichlet(coords, elements, surfaces)
    print(f"      φ: [{phi.min():.4f}, {phi.max():.4f}]")
    
    grad_phi = compute_gradient(coords, elements, phi)
    
    # Fiber reconstruction
    print("\n  [2] FIBER RECONSTRUCTION")
    fibers, sheets, angles, coord_sys = reconstruct_helical_fibers(coords, elements, phi, grad_phi, tags)
    print(f"      Fiber angle: [{angles.min():.1f}°, {angles.max():.1f}°]")
    
    # Wall stress with curvature tensor
    print("\n  [3] WALL STRESS COMPUTATION")
    stress_data = compute_wall_stress(coords, elements, surfaces, tags)
    
    # Geodesic optimization
    print("\n  [4] INJECTION SITE OPTIMIZATION")
    border_elem = np.where(tags == TAG_BORDER)[0]
    if len(border_elem) > 0:
        border_nodes = list(set(elements[border_elem[:100]].flatten()))
        print("      Computing geodesic distances")
        L = build_laplacian(coords, elements)
        geodesic = compute_geodesic_distance(coords, elements, border_nodes, L)
    else:
        geodesic = stress_data['endo_dist']
    
    sites, total_red = optimize_injection_sites(coords, elements, tags, stress_data, phi, geodesic, centroids, N_INJECTION_SITES)
    
    print(f"      Selected {len(sites)} sites, total Δσ = {total_red:.2f} kPa")
    for s in sites:
        print(f"        Element {s['element_id']}: geodesic={s['geodesic_distance']:.2f}, Δσ={s['stress_reduction_kPa']:.2f}")
    
    # Save outputs
    print("\n  [5] SAVING OUTPUTS")
    out_dir = os.path.join(OUTPUT_DIR, patient_id)
    os.makedirs(out_dir, exist_ok=True)
    
    phi_elem = np.mean(phi[elements], axis=1)
    geo_elem = np.mean(geodesic[elements], axis=1) if len(geodesic) == n_nodes else np.zeros(n_elems)
    
    write_vtk(os.path.join(out_dir, f"{patient_id}_analysis.vtk"), coords, elements, {
        'TransmuralPhi': phi, 'TransmuralPhi_elem': phi_elem, 'TissueType': tags.astype(float),
        'FiberAngle_deg': angles, 'WallThickness_mm': stress_data['wall_thickness'],
        'WallStress_kPa': stress_data['wall_stress'], 'Kappa1': stress_data['kappa_1'],
        'Kappa2': stress_data['kappa_2'], 'MeanCurvature': stress_data['mean_curvature'],
        'CurvatureFactor': stress_data['curvature_factor'], 'GeodesicDistance': geo_elem,
    }, {'FiberDirection': fibers, 'SheetDirection': sheets, 'TransmuralDirection': coord_sys[0]})
    
    write_lon(os.path.join(out_dir, f"{patient_id}.lon"), fibers, sheets)
    
    summary = {
        'patient_id': patient_id,
        'timestamp': datetime.now().isoformat(),
        'mesh': {'n_nodes': n_nodes, 'n_elements': n_elems},
        'transmural': {'phi_range': [float(phi.min()), float(phi.max())]},
        'fibers': {'angle_range_deg': [float(angles.min()), float(angles.max())]},
        'curvature': {
            'kappa_1_range': [float(stress_data['kappa_1'].min()), float(stress_data['kappa_1'].max())],
            'kappa_2_range': [float(stress_data['kappa_2'].min()), float(stress_data['kappa_2'].max())],
        },
        'wall_stress': {
            'range_kPa': [float(stress_data['wall_stress'].min()), float(stress_data['wall_stress'].max())],
            'mean_kPa': float(stress_data['wall_stress'].mean()),
        },
        'optimization': {
            'total_stress_reduction_kPa': float(total_red),
            'n_sites': len(sites),
        },
        'injection_sites': sites,
    }
    
    with open(os.path.join(out_dir, f"{patient_id}_summary.json"), 'w') as f:
        json.dump(summary, f, indent=2)
    
    print(f"  Saved to: {out_dir}")
    
    return summary


def process_patient_wrapper(patient_id):
    """Wrapper for parallel processing with error handling."""
    try:
        return process_patient(patient_id)
    except Exception as e:
        import traceback
        traceback.print_exc()
        return {'patient_id': patient_id, 'status': 'FAILED', 'error': str(e)}


# MAIN

def main():
    """Main entry point."""
    print("LAPLACE-DIRICHLET CARDIAC ANALYSIS FRAMEWORK")
    
    print("\n  Components:")
    print("  [1] Transmural coordinate (Laplace-Dirichlet)")
    print("  [2] Fiber reconstruction (helical angle rotation)")
    print("  [3] Wall stress (modified Laplace with curvature tensor)")
    print("  [4] Injection optimization (geodesic + stress reduction)")
    
    print(f"\n  Using {N_CPUS} CPUs")
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # Process patients in parallel
    n_workers = min(N_CPUS, len(PATIENT_IDS))
    with Pool(n_workers) as pool:
        results = pool.map(process_patient_wrapper, PATIENT_IDS)
    
    # Summary
    print("SUMMARY")
    
    print(f"\n{'Patient':<15} {'κ₁ range':<25} {'Stress Red.':<15} {'Sites'}")
    
    for r in results:
        if 'error' not in r:
            k1 = r['curvature']['kappa_1_range']
            red = r['optimization']['total_stress_reduction_kPa']
            sites = r['optimization']['n_sites']
            print(f"{r['patient_id']:<15} [{k1[0]:.4f}, {k1[1]:.4f}]       {red:.2f} kPa         {sites}")
        else:
            print(f"{r['patient_id']:<15} FAILED: {r['error'][:40]}")
    
    # Save combined results
    with open(os.path.join(OUTPUT_DIR, "summary.json"), 'w') as f:
        json.dump(results, f, indent=2, default=str)
    
    print(f"\nResults saved to: {OUTPUT_DIR}")
    
    return results


if __name__ == "__main__":
    main()

LAPLACE-DIRICHLET CARDIAC ANALYSIS FRAMEWORK

  Components:
  [1] Transmural coordinate (Laplace-Dirichlet)
  [2] Fiber reconstruction (helical angle rotation)
  [3] Wall stress (modified Laplace with curvature tensor)
  [4] Injection optimization (geodesic + stress reduction)

  Using 44 CPUs

PROCESSING: SCD0000301
PROCESSING: SCD0000101
PROCESSING: SCD0000201
PROCESSING: SCD0000401
PROCESSING: SCD0000701
PROCESSING: SCD0000801
PROCESSING: SCD0001001
PROCESSING: SCD0000601
PROCESSING: SCD0001201
PROCESSING: SCD0001101









  Mesh: 49,235 nodes, 262,852 elements
  Mesh: 56,415 nodes, 291,380 elements
  Mesh: 57,018 nodes, 305,488 elements
  Mesh: 58,955 nodes, 315,135 elements
  Tags: 18396 infarct, 40022 border

  [1] TRANSMURAL COORDINATE
  Tags: 21693 infarct, 56974 border

  [1] TRANSMURAL COORDINATE
  Tags: 22163 infarct, 58325 border

  [1] TRANSMURAL COORDINATE  Tags: 24899 infarct, 59665 border


  [1] TRANSMURAL COORDINATE
  Mesh: 70,518 nodes, 383,439 elements
  Tags: 28